# 🔥 Flare industrial: LES 3D, vento de atlas eólico e radiação térmica (Colab)

Simulação de um **flare elevado** queimando **gás de refinaria (mistura de H₂, C1–C5, olefinas, inertes e H₂S)** ou um combustível puro, com o vento típico do local tirado de um **atlas eólico**, gráficos atualizados **em tempo real** enquanto o CFD resolve e um **vídeo MP4** no final, com a chama em zoom. O caso padrão usa a localização e o clima de vento da **REPLAN (Paulínia, SP)**, com composição e vazão **ilustrativas** de literatura aberta (não são dados operacionais da Petrobras).

| Etapa | O que faz |
|---|---|
| 1 | **Clima de vento do local**: Global Wind Atlas (clima generalizado, Weibull por setor), NASA POWER (MERRA-2) ou Open-Meteo (ERA5) → perfil de camada limite (lei log) e rosa dos ventos na altura do tip |
| 2 | Combustível (mistura ou puro) e referências: **API 521** (fonte pontual), **Chamberlain/Shell (1987)** com fração radiante corrigida pela composição, correlações de comprimento de chama |
| 3 | Termoquímica: equilíbrio (Cantera) na fração de mistura Z e tabela **β-PDF** de submalha |
| 4 | **LES 3D de baixo Mach em PyTorch (GPU)**, 60 s simulados, malha fina ajustada à chama, gravando um quadro a cada 0,1 s |
| 5 | Validação (LES × Chamberlain/API) e segurança no vento de projeto |
| 6 | **Varredura das direções do vento**: pegada de radiação girando pela rosa dos ventos → probabilidade de excedência dos níveis do API 521 e envoltória |
| 7 | Vídeo final em HD (1080p a 4K): abertura, contexto para gestão, chama em HD, painel com validação, varredura do vento, resumos e fechamento; texto sugerido para o post |

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Sem GPU o notebook usa o preset `cpu` (Δ = 0,7 m), que roda, mas encurta um pouco a chama.

A teoria, a álgebra e as referências estão no documento de revisão que acompanha o notebook (`docs/revisao_flare_cfd.html`).

In [ ]:
#@title Ambiente: GPU, Cantera e pacote `flarekit`
import subprocess, sys, os
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
print("GPU:", gpu or "nenhuma (vai rodar em CPU, com malha mais grossa)")
try:
    import cantera
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cantera"], check=False)
os.makedirs("flarekit", exist_ok=True)

In [ ]:
%%writefile flarekit/__init__.py
"""flarekit — simulação de flares industriais: modelos semi-empíricos (API 521, Chamberlain),
termoquímica em fração de mistura, LES 3D de baixo Mach em PyTorch e critérios de segurança."""
__version__ = "0.1.0"

In [ ]:
%%writefile flarekit/props.py
"""Termoquímica do flare: estequiometria, relação de estado em Z e tabela com β-PDF.

A química é tratada no limite de fração de mistura (seção 3.2 e 4.4 do documento):
o estado local depende só de Z (equilíbrio via Cantera, ou Burke–Schumann se o
Cantera não estiver instalado). Na LES, as médias filtradas vêm da integração
na β-PDF com variância de submalha, o que inclui a interação turbulência–química
e a interação turbulência–radiação na emissão.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from scipy.special import betainc

R_U = 8.314462618        # J/(mol K)
SIGMA = 5.670374419e-8   # W/(m2 K4)
P_ATM = 101325.0         # Pa
G = 9.80665              # m/s2

MW = {"O2": 31.998e-3, "N2": 28.014e-3, "CO2": 44.009e-3, "H2O": 18.015e-3, "SO2": 64.058e-3}
Y_O2_AIR = MW["O2"] / (MW["O2"] + 3.76 * MW["N2"])
M_AIR = 0.028965


def pressure_at_altitude(z: float) -> float:
    """Pressão da atmosfera padrão (ISA) na altitude z [m] acima do nível do mar [Pa]."""
    return P_ATM * (1.0 - 2.25577e-5 * z) ** 5.25588


@dataclass(frozen=True)
class Fuel:
    """Combustível puro ou mistura. nC, nH, nO, nN, nS são átomos por mol de combustível
    (fracionários numa mistura); X guarda a composição molar ((espécie, fração), ...)."""
    name: str
    nC: float
    nH: float
    M: float          # kg/mol
    LHV: float        # J/kg (PCI)
    gamma: float      # cp/cv a ~300 K
    soot_yield: float  # kg fuligem / kg combustível (informativo)
    cantera_name: str  # espécie do gri30 (combustível puro); "" para misturas
    nO: float = 0.0
    nN: float = 0.0
    nS: float = 0.0
    X: tuple = ()

    @property
    def is_mixture(self) -> bool:
        return len(self.X) > 0

    def composition(self) -> dict:
        """Frações molares {espécie: x} (nomes de SPECIES)."""
        return dict(self.X) if self.X else {CANTERA_TO_KEY.get(self.cantera_name, self.cantera_name): 1.0}

    def describe(self) -> str:
        comp = " · ".join(f"{k} {100 * v:.1f}%" for k, v in sorted(self.composition().items(),
                                                                   key=lambda kv: -kv[1]))
        return (f"{self.name}: {comp} | M = {1e3 * self.M:.2f} g/mol · PCI = {self.LHV / 1e6:.2f} MJ/kg · "
                f"γ = {self.gamma:.3f}")


@dataclass(frozen=True)
class SpeciesData:
    atoms: tuple       # (C, H, O, N, S)
    M: float           # kg/mol
    LHV_mol: float     # J/mol (PCI a 25 °C, H2O vapor, S → SO2)
    cp: float          # J/(mol K) a 298 K
    nasa: str          # nome no nasa_gas.yaml do Cantera
    soot_yield: float  # kg/kg (Tewarson, SFPE Handbook; aproximado, informativo)


# Propriedades a 25 °C dos polinômios NASA (Glenn) do Cantera (nasa_gas.yaml); PCI conferido em
# tests/test_flarekit.py contra o próprio Cantera.
SPECIES = {
    "H2":     SpeciesData((0, 2, 0, 0, 0), 2.016e-3, 241.8e3, 28.84, "H2", 0.0),
    "CH4":    SpeciesData((1, 4, 0, 0, 0), 16.043e-3, 802.6e3, 35.69, "CH4", 0.001),
    "C2H6":   SpeciesData((2, 6, 0, 0, 0), 30.070e-3, 1428.6e3, 52.50, "C2H6", 0.013),
    "C2H4":   SpeciesData((2, 4, 0, 0, 0), 28.054e-3, 1323.2e3, 42.89, "C2H4", 0.043),
    "C3H8":   SpeciesData((3, 8, 0, 0, 0), 44.097e-3, 2043.1e3, 73.59, "C3H8", 0.024),
    "C3H6":   SpeciesData((3, 6, 0, 0, 0), 42.081e-3, 1925.7e3, 64.43, "C3H6,propylene", 0.095),
    "nC4H10": SpeciesData((4, 10, 0, 0, 0), 58.124e-3, 2657.4e3, 98.66, "C4H10,n-butane", 0.029),
    "iC4H10": SpeciesData((4, 10, 0, 0, 0), 58.124e-3, 2648.2e3, 96.64, "C4H10,isobutane", 0.029),
    "C4H8":   SpeciesData((4, 8, 0, 0, 0), 56.108e-3, 2540.8e3, 85.56, "C4H8,1-butene", 0.095),
    "nC5H12": SpeciesData((5, 12, 0, 0, 0), 72.151e-3, 3271.7e3, 119.95, "C5H12,n-pentane", 0.030),
    "CO":     SpeciesData((1, 0, 1, 0, 0), 28.010e-3, 283.0e3, 29.14, "CO", 0.0),
    "H2S":    SpeciesData((0, 2, 0, 0, 1), 34.076e-3, 518.2e3, 34.19, "H2S", 0.0),
    "CO2":    SpeciesData((1, 0, 2, 0, 0), 44.009e-3, 0.0, 37.14, "CO2", 0.0),
    "N2":     SpeciesData((0, 0, 0, 2, 0), 28.014e-3, 0.0, 29.12, "N2", 0.0),
    "O2":     SpeciesData((0, 0, 2, 0, 0), 31.998e-3, 0.0, 29.38, "O2", 0.0),
    "H2O":    SpeciesData((0, 2, 1, 0, 0), 18.015e-3, 0.0, 33.59, "H2O", 0.0),
}
CANTERA_TO_KEY = {"C3H8": "C3H8", "CH4": "CH4", "C2H6": "C2H6", "H2": "H2"}


def mixture(name: str, comp: dict) -> Fuel:
    """Combustível a partir da composição molar {espécie: fração ou %} (normalizada para 1)."""
    bad = [k for k in comp if k not in SPECIES]
    if bad:
        raise ValueError(f"espécies desconhecidas: {bad}; use {sorted(SPECIES)}")
    if any(v < 0 for v in comp.values()):
        raise ValueError("frações negativas na composição")
    tot = float(sum(comp.values()))
    if tot <= 0:
        raise ValueError("composição vazia")
    x = {k: float(v) / tot for k, v in comp.items() if v > 0}
    at = np.zeros(5)
    M = cp = lhv = soot = 0.0
    for k, xi in x.items():
        d = SPECIES[k]
        at += xi * np.asarray(d.atoms, float)
        M += xi * d.M
        cp += xi * d.cp
        lhv += xi * d.LHV_mol
    for k, xi in x.items():
        soot += xi * SPECIES[k].M / M * SPECIES[k].soot_yield
    if lhv <= 0:
        raise ValueError("a mistura não tem componentes combustíveis")
    if at[0] + at[1] / 4 + at[4] - at[2] / 2 <= 0:
        raise ValueError("a mistura já traz O2 suficiente para queimar sozinha: não é um gás de tocha")
    gamma = cp / (cp - R_U)
    return Fuel(name, float(at[0]), float(at[1]), M, lhv / M, gamma, soot, "",
                nO=float(at[2]), nN=float(at[3]), nS=float(at[4]), X=tuple(sorted(x.items())))


def parse_composition(text: str) -> dict:
    """'H2: 20, CH4: 40, C2H6: 10' → {'H2': 20.0, 'CH4': 40.0, 'C2H6': 10.0}. Aceita vírgula ou ponto
    decimal ('10,5'), '=' no lugar de ':', '%' opcional e separação por vírgula, ponto e vírgula ou espaço."""
    import re
    pat = re.compile(r"([A-Za-z][A-Za-z0-9]*)\s*[:=]\s*([0-9]+(?:[.,][0-9]+)?)\s*%?")
    out = {}
    for k, v in pat.findall(text):
        out[k] = out.get(k, 0.0) + float(v.replace(",", "."))
    rest = pat.sub("", text).replace(";", "").replace(",", "").strip()
    if not out or rest:
        raise ValueError(f"composição inválida perto de '{rest[:30]}'; use 'espécie: valor', ex. 'H2: 20, CH4: 40,5'")
    return out


PROPANE = Fuel("propano", 3, 8, 44.097e-3, 46.35e6, 1.13, 0.024, "C3H8")
METHANE = Fuel("metano", 1, 4, 16.043e-3, 50.03e6, 1.31, 0.001, "CH4")
ETHANE = Fuel("etano", 2, 6, 30.069e-3, 47.51e6, 1.19, 0.013, "C2H6")
FUELS = {f.name: f for f in (PROPANE, METHANE, ETHANE)}


# Composições de gás de tocha/refinaria (% molar). Não há composição medida pública da REPLAN;
# estas são referências abertas, para estudo ilustrativo (confira com dados da unidade).
FLARE_GAS_PRESETS = {
    # pontos médios das faixas da FISPQ "Gás Residual de Refinaria" (Acelen, ex-RLAM/Petrobras;
    # H2 12,5–52,5 · CH4 12,5–42,5 · C2 10–40 · C3 1,5–3,5 · C4 1,5–3,5 · C5 0–1 · N2+CO2 2–8 ·
    # CO 0,5–2,5 · H2S 0–1,2), com as frações agrupadas (C2, C3, inertes) divididas por estimativa
    "gás de refinaria típico (FISPQ ex-RLAM)": {
        "H2": 33.0, "CH4": 29.0, "C2H6": 15.0, "C2H4": 10.5, "C3H8": 1.5, "C3H6": 1.0,
        "nC4H10": 2.5, "nC5H12": 0.5, "N2": 3.5, "CO2": 1.5, "CO": 1.5, "H2S": 0.5},
    # média de gás de tocha "de uma planta típica" em Emam (2015), Petroleum & Coal — mais
    # pesada, rica em GLP (i-C5 somado ao n-C5)
    "gás de tocha médio de refinaria (Emam 2015)": {
        "CH4": 43.6, "C2H6": 3.66, "C3H8": 20.3, "nC4H10": 2.78, "iC4H10": 14.3, "nC5H12": 0.796,
        "C2H4": 1.05, "C3H6": 2.73, "C4H8": 0.696, "H2": 5.54, "CO": 0.186, "CO2": 0.713,
        "H2S": 0.256, "O2": 0.357, "N2": 1.30, "H2O": 1.14},
    # caso hipotético de despressurização de unidades de hidrogênio/hidrotratamento
    "rico em H2 (despressurização, hipotético)": {
        "H2": 75.0, "CH4": 15.0, "C2H6": 5.0, "C3H8": 3.0, "N2": 2.0},
}


def flare_gas(name: str) -> Fuel:
    """Combustível a partir de FLARE_GAS_PRESETS."""
    return mixture(name, FLARE_GAS_PRESETS[name])


def lhv_volumetric(fuel: Fuel, T: float = 273.15, p: float = P_ATM) -> float:
    """PCI por volume de gás nas condições (T, p) [J/m³]; padrão: Nm³ (0 °C, 1 atm)."""
    return fuel.LHV * fuel.M * p / (R_U * T)


def o2_demand(fuel: Fuel) -> float:
    """mols de O2 por mol de combustível (C → CO2, H → H2O, S → SO2, descontando o O do combustível)."""
    return fuel.nC + fuel.nH / 4.0 + fuel.nS - fuel.nO / 2.0


def products_per_mol(fuel: Fuel) -> dict:
    """Mols de produtos da combustão estequiométrica com ar, por mol de combustível."""
    a = o2_demand(fuel)
    return {"CO2": fuel.nC, "H2O": fuel.nH / 2.0, "SO2": fuel.nS, "N2": 3.76 * a + fuel.nN / 2.0}


def stoichiometry(fuel: Fuel) -> dict:
    """a = mols de O2 por mol de combustível; s = razão mássica ar/combustível; Z_st."""
    a = o2_demand(fuel)
    s = a * (MW["O2"] + 3.76 * MW["N2"]) / fuel.M
    return {"a": a, "s": s, "Z_st": 1.0 / (1.0 + s)}


def burke_schumann(fuel: Fuel, Z: np.ndarray) -> dict:
    """Frações mássicas de combustão completa (química infinitamente rápida)."""
    Z = np.asarray(Z, dtype=float)
    pr = products_per_mol(fuel)
    r_O2 = o2_demand(fuel) * MW["O2"] / fuel.M
    r_CO2 = pr["CO2"] * MW["CO2"] / fuel.M
    r_H2O = pr["H2O"] * MW["H2O"] / fuel.M
    r_SO2 = pr["SO2"] * MW["SO2"] / fuel.M
    r_N2f = fuel.nN / 2.0 * MW["N2"] / fuel.M                 # N2 que já vem no combustível
    burned = np.minimum(Z, (1.0 - Z) * Y_O2_AIR / r_O2)   # kg combustível queimado / kg mistura
    Y = {
        "F": Z - burned,
        "O2": (1.0 - Z) * Y_O2_AIR - burned * r_O2,
        "N2": (1.0 - Z) * (1.0 - Y_O2_AIR) + burned * r_N2f,
        "CO2": burned * r_CO2,
        "H2O": burned * r_H2O,
        "SO2": burned * r_SO2,
    }
    Y["O2"] = np.maximum(Y["O2"], 0.0)
    Y["burned"] = burned
    return Y


def radcal_ap(species: str, T: np.ndarray) -> np.ndarray:
    """Coeficiente médio de Planck [1/(atm m)] — ajustes RADCAL da TNF Workshop."""
    t = 1000.0 / np.asarray(T, dtype=float)
    if species == "H2O":
        c = (-0.23093, -1.12390, 9.41530, -2.99880, 0.51382, -1.86840e-5)
    elif species == "CO2":
        c = (18.741, -121.310, 273.500, -194.050, 56.310, -5.8169)
    else:
        raise ValueError(species)
    return sum(ci * t**i for i, ci in enumerate(c))


def soot_kappa_coefficient(n: float = 1.57, k: float = 0.56) -> float:
    """κ_P,fuligem = C·f_v·T, com C = 3.83·C0/C2 (dedução da seção 4.6)."""
    C0 = 36 * np.pi * n * k / ((n**2 - k**2 + 2) ** 2 + 4 * n**2 * k**2)
    return 3.83 * C0 / 1.4388e-2


@dataclass
class StateRelation:
    """Estado em função de Z (sem flutuação): T, ρ, emissão, frações."""
    Z: np.ndarray
    T: np.ndarray
    rho: np.ndarray
    M: np.ndarray
    x_H2O: np.ndarray
    x_CO2: np.ndarray
    Y_F: np.ndarray
    Y_soot: np.ndarray
    kappa: np.ndarray
    emission: np.ndarray   # emissão luminosa da fuligem 4σκ_s T⁴ (f_v,max nominal = 1 ppm) [W/m3]
    T_ad_st: float
    source: str


def _products_cp(fuel: Fuel) -> float:
    """c_p médio (298–2000 K) dos produtos da combustão estequiométrica com ar [J/(kg K)]."""
    cp = {"CO2": 1230.0, "H2O": 2430.0, "N2": 1220.0, "SO2": 820.0}
    Mi = {"CO2": MW["CO2"], "H2O": MW["H2O"], "N2": MW["N2"], "SO2": MW["SO2"]}
    pr = products_per_mol(fuel)
    m = {k: pr[k] * Mi[k] for k in pr}
    return sum(m[k] * cp[k] for k in m) / sum(m.values())


# produtos e radicais do equilíbrio quando o combustível é uma mistura (base NASA do Cantera)
_EQ_SPECIES = ["N2", "O2", "H2O", "CO2", "CO", "H2", "OH", "H", "O", "NO", "N", "CH4", "C2H2,acetylene"]
_EQ_SULFUR = ["SO2", "SO", "S", "S2", "H2S", "COS"]


def _cantera_gas(fuel: Fuel):
    """Fase gasosa do Cantera para o equilíbrio: gri30 para combustível puro; para misturas, um
    conjunto reduzido de espécies da base NASA (cobre C4/C5, olefinas e H2S, que o gri30 não tem)."""
    import cantera as ct
    if not fuel.is_mixture:
        return ct.Solution("gri30.yaml"), {fuel.cantera_name: 1.0}, "gri30"
    comp = fuel.composition()
    names = list(_EQ_SPECIES) + (list(_EQ_SULFUR) if fuel.nS > 0 else [])
    names += [SPECIES[k].nasa for k in comp if SPECIES[k].nasa not in names]
    lib = {sp.name: sp for sp in ct.Species.list_from_file("nasa_gas.yaml")}
    gas = ct.Solution(thermo="ideal-gas", species=[lib[n] for n in names])
    return gas, {SPECIES[k].nasa: x for k, x in comp.items()}, "base NASA, mistura"


def state_relation(fuel: Fuel, T_fuel: float, T_inf: float, chi_loss: float,
                   n: int = 2001, use_cantera: bool = True, p: float = P_ATM,
                   phi_rich: float = 2.5) -> StateRelation:
    """Relação de estado φ(Z).

    T_eq(Z) vem de equilíbrio HP (Cantera, gri30) ou de Burke–Schumann com c_p efetivo.
    A perda radiativa usa a abordagem de fração radiante constante (como no FDS):
    T(Z) = T_mix(Z) + (1 − χ)·(T_eq(Z) − T_mix(Z)).
    """
    st = stoichiometry(fuel)
    # malha em Z refinada perto de Z_st (onde tudo acontece)
    Zs = st["Z_st"]
    zb = min(3 * Zs, 0.5 * (1 + Zs))          # gases pobres (Z_st > 1/3) não podem passar de Z = 1
    z1 = np.linspace(0.0, zb, n // 2, endpoint=False)
    z2 = np.linspace(zb, 1.0, n - n // 2)
    Z = np.concatenate([z1, z2])
    Y = burke_schumann(fuel, Z)
    T_mix = None
    source = "Burke-Schumann"
    T_eq = None
    M_mix = None
    xH2O = xCO2 = None
    if use_cantera:
        try:
            gas, X_fuel, mech = _cantera_gas(fuel)
            gas.TPX = T_fuel, p, X_fuel
            hF, YF = gas.enthalpy_mass, gas.Y.copy()
            gas.TPX = T_inf, p, {"O2": 1.0, "N2": 3.76}
            hA, YA = gas.enthalpy_mass, gas.Y.copy()
            T_eq = np.empty_like(Z); T_mix = np.empty_like(Z); M_mix = np.empty_like(Z)
            xH2O = np.empty_like(Z); xCO2 = np.empty_like(Z)
            iH2O, iCO2 = gas.species_index("H2O"), gas.species_index("CO2")
            # Limite rico de flamabilidade (como o "rich flammability limit" dos códigos
            # comerciais): acima de Z_rich o equilíbrio prevê craqueamento irreal; a mistura
            # é tratada como o estado queimado em Z_rich misturado com combustível frio.
            Z_rich = phi_rich * Zs / (1 - Zs + phi_rich * Zs)
            gas.HPY = Z_rich * hF + (1 - Z_rich) * hA, p, Z_rich * YF + (1 - Z_rich) * YA
            gas.equilibrate("HP")
            h_r, Y_r = gas.enthalpy_mass, gas.Y.copy()
            for i, z in enumerate(Z):
                gas.HPY = z * hF + (1 - z) * hA, p, z * YF + (1 - z) * YA
                T_mix[i] = gas.T
                if z <= Z_rich:
                    if z > 1e-6:
                        gas.equilibrate("HP")
                else:
                    w = (z - Z_rich) / (1 - Z_rich)
                    gas.HPY = (1 - w) * h_r + w * hF, p, (1 - w) * Y_r + w * YF
                T_eq[i] = gas.T; M_mix[i] = gas.mean_molecular_weight / 1000.0
                xH2O[i] = gas.X[iH2O]; xCO2[i] = gas.X[iCO2]
            source = f"Cantera (equilíbrio HP, {mech})"
        except Exception as e:  # noqa: BLE001 — sem Cantera (ou sem convergência), cai para Burke–Schumann
            T_eq = None
            source = f"Burke-Schumann (Cantera indisponível: {type(e).__name__})"
    if T_eq is None:
        cp_mix = 1100.0
        T_mix = (Z * T_fuel * 1700.0 + (1 - Z) * T_inf * 1005.0) / (Z * 1700.0 + (1 - Z) * 1005.0)
        # 1420 J/(kg K) reproduz T_ad de alcanos leves (seção 3.2); para outras composições escala pela
        # razão do c_p médio dos produtos estequiométricos em relação ao do propano
        cp_eff = 1420.0 * _products_cp(fuel) / _products_cp(PROPANE)
        T_eq = T_mix + Y["burned"] * fuel.LHV / cp_eff
        del cp_mix
        moles = {k: Y[k] / (fuel.M if k == "F" else MW[k]) for k in ("F", "O2", "N2", "CO2", "H2O", "SO2")}
        ntot = sum(moles.values())
        M_mix = 1.0 / ntot
        xH2O, xCO2 = moles["H2O"] / ntot, moles["CO2"] / ntot
    T = T_mix + (1.0 - chi_loss) * (T_eq - T_mix)
    rho = p * M_mix / (R_U * T)
    # Fuligem por relação de estado (Sivathanu & Faeth 1990): só no lado rico da chama,
    # nula para φ ≤ 1, máxima em φ = 2, nula de novo em φ = 5. A emissão para os receptores
    # vem dessa zona luminosa; a potência total é escalada depois para X_rad·Q.
    phi = (Z / np.maximum(1 - Z, 1e-12)) / (Zs / (1 - Zs))
    f_shape = np.clip(np.minimum(phi - 1.0, (5.0 - phi) / 3.0), 0.0, 1.0)
    f_v = 1e-6 * f_shape
    Y_soot = f_v * 1800.0 / rho
    kappa_gas = (p / P_ATM) * (xH2O * radcal_ap("H2O", T) + xCO2 * radcal_ap("CO2", T))
    kappa = np.maximum(kappa_gas, 0.0) + soot_kappa_coefficient() * f_v * T
    emission = 4 * SIGMA * soot_kappa_coefficient() * f_v * T * T**4
    T_ad_st = float(np.interp(Zs, Z, T_eq))
    return StateRelation(Z, T, rho, M_mix, xH2O, xCO2, Y["F"], Y_soot, kappa, emission, T_ad_st, source)


@dataclass
class BetaPDFTable:
    """Tabela φ̃(Z̃, s) com s = Z''²/(Z̃(1−Z̃)) ∈ [0, s_max]. Grades uniformes (busca O(1) na GPU)."""
    Zt: np.ndarray
    s: np.ndarray
    rho: np.ndarray
    drho_dZ: np.ndarray
    T: np.ndarray
    emission: np.ndarray
    Y_F: np.ndarray
    Z_st: float
    T_ad_st: float
    rho_inf: float
    source: str


def beta_pdf_table(sr: StateRelation, Z_st: float, nZ: int = 401, ns: int = 17,
                   s_max: float = 0.8) -> BetaPDFTable:
    """Integra φ(Z) na β-PDF de Favre. Usa diferenças da função beta incompleta em
    cada intervalo, que é robusto mesmo quando a < 1 ou b < 1 (PDF singular nas bordas)."""
    Zt = np.linspace(0.0, 1.0, nZ)
    s = np.linspace(0.0, s_max, ns)
    zf = np.linspace(0.0, 1.0, 4001)
    zm = 0.5 * (zf[1:] + zf[:-1])
    inv_rho = 1.0 / np.interp(zm, sr.Z, sr.rho)
    T_m = np.interp(zm, sr.Z, sr.T)
    e_over_rho = np.interp(zm, sr.Z, sr.emission) * inv_rho
    YF_m = np.interp(zm, sr.Z, sr.Y_F)
    out = {k: np.zeros((nZ, ns)) for k in ("inv_rho", "T", "e_over_rho", "Y_F")}
    for j, sj in enumerate(s):
        for i, z in enumerate(Zt):
            var_ok = sj > 1e-6 and 1e-6 < z < 1 - 1e-6
            if not var_ok:
                w = None
            else:
                gam = 1.0 / sj - 1.0
                a, b = z * gam, (1 - z) * gam
                cdf = betainc(a, b, zf)
                w = np.diff(cdf)
                tot = w.sum()
                w = w / tot if tot > 0 else None
            if w is None:
                out["inv_rho"][i, j] = 1.0 / np.interp(z, sr.Z, sr.rho)
                out["T"][i, j] = np.interp(z, sr.Z, sr.T)
                out["e_over_rho"][i, j] = np.interp(z, sr.Z, sr.emission) * out["inv_rho"][i, j]
                out["Y_F"][i, j] = np.interp(z, sr.Z, sr.Y_F)
            else:
                out["inv_rho"][i, j] = w @ inv_rho
                out["T"][i, j] = w @ T_m
                out["e_over_rho"][i, j] = w @ e_over_rho
                out["Y_F"][i, j] = w @ YF_m
    rho = 1.0 / out["inv_rho"]
    drho = np.gradient(rho, Zt, axis=0)
    return BetaPDFTable(Zt, s, rho, drho, out["T"], rho * out["e_over_rho"], out["Y_F"],
                        Z_st, sr.T_ad_st, float(sr.rho[0]), sr.source)

In [ ]:
%%writefile flarekit/semiempirical.py
"""Modelos semi-empíricos de chama e radiação (níveis N0–N1 do documento, seção 3).

Funções vetorizadas em NumPy. Servem de referência de validação para a LES e de
ferramenta de projeto (altura do flare, distâncias de segurança).
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from .props import G, M_AIR, P_ATM, R_U, Fuel, products_per_mol, radcal_ap, stoichiometry


# --------------------------------------------------------------------------- tip
def tip_conditions(fuel: Fuel, mdot: float, T_j: float, mach: float, p: float = P_ATM) -> dict:
    """Seção 3.1: gás ideal na pressão ambiente, Mach prescrito."""
    rho_j = p * fuel.M / (R_U * T_j)
    c_j = np.sqrt(fuel.gamma * R_U * T_j / fuel.M)
    u_j = mach * c_j
    d_j = np.sqrt(4 * mdot / (np.pi * rho_j * u_j))
    return {"rho_j": rho_j, "c_j": c_j, "u_j": u_j, "d_j": d_j, "Q": mdot * fuel.LHV}


def air_density(T_inf: float, p: float = P_ATM) -> float:
    return p * M_AIR / (R_U * T_inf)


# ---------------------------------------------------------------- comprimentos
def api_flame_length(Q: float) -> float:
    """Ajuste de Beychok à curva do API 521: L[m] = 0.00323·Q[W]^0.478."""
    return 0.0059 * 0.3048 * (3.412142 * Q) ** 0.478


def delichatsios_length(u_j, d_j, rho_j, rho_inf, Z_st, T_ad, T_inf) -> dict:
    """Seção 3.3: Froude da chama e L* (Delichatsios 1993; Schefer et al. 2006)."""
    Fr = u_j * Z_st**1.5 / ((rho_j / rho_inf) ** 0.25 * np.sqrt((T_ad - T_inf) / T_inf * G * d_j))
    Lstar = 13.5 * Fr**0.4 / (1 + 0.07 * Fr**2) ** 0.2 if Fr < 5 else 23.0
    dstar = d_j * np.sqrt(rho_j / rho_inf)
    return {"Fr_f": Fr, "L_star": Lstar, "d_star": dstar, "L": Lstar * dstar / Z_st}


def heskestad_length(Q: float, D: float) -> float:
    """Altura média de chamas de empuxo (Heskestad): L = 0.235 Q[kW]^(2/5) − 1.02 D."""
    return 0.235 * (Q / 1e3) ** 0.4 - 1.02 * D


def molina_xrad(fuel: Fuel, mdot, L, Z_st, T_ad, rho_f) -> dict:
    """Seção 3.4: X_rad = 9.45e-9 (τ_G a_p T_ad⁴)^0.47, τ_G em ms."""
    pr = products_per_mol(fuel)
    n_tot = sum(pr.values())
    x_CO2, x_H2O = pr["CO2"] / n_tot, pr["H2O"] / n_tot
    a_p = x_CO2 * radcal_ap("CO2", T_ad) + x_H2O * radcal_ap("H2O", T_ad)
    W_f = 0.17 * L
    tau = np.pi / 12 * rho_f * W_f**2 * L * Z_st / mdot * 1000.0
    return {"a_p": float(a_p), "tau_G_ms": float(tau), "X_rad": float(9.45e-9 * (tau * a_p * T_ad**4) ** 0.47)}


# --------------------------------------------------------------- Chamberlain
@dataclass
class Chamberlain:
    D_s: float
    W: float
    Y: float
    L_b0: float
    L_b: float
    R: float
    Ri_Lb0: float
    alpha: float      # graus
    K: float
    b: float
    R_l: float
    W1: float
    W2: float
    A: float
    F_s: float
    SEP: float        # W/m2
    Q: float


def radiant_fraction_factor(fuel: Fuel) -> float:
    """Correção aproximada de F_s pela composição (F_s de Chamberlain foi ajustado a gás natural).

    Usa a razão r = PCI molar (≈ PCI volumétrico) do combustível / PCI molar do CH4 e a tendência
    da tabela de fração radiante do API 521 (chamas de laboratório, maiores queimadores):
    H2 ≈ 0,70 × gás natural e butano ≈ 1,25 × gás natural → f = r^0,30 (r < 1) e r^0,19 (r ≥ 1).
    É uma interpolação de engenharia, não uma correlação publicada; f fica em [0,6; 1,35]."""
    r = fuel.LHV * fuel.M / 802.6e3
    f = r ** 0.30 if r < 1.0 else r ** 0.19
    return float(np.clip(f, 0.6, 1.35))


def chamberlain(fuel: Fuel, mdot: float, u_j: float, rho_j: float, rho_inf: float,
                u_w: float, theta_jv: float = 90.0, fs_factor: float = 1.0) -> Chamberlain:
    """Seção 3.7 (Chamberlain 1987). Ângulos em graus; u_w é o vento na altura do tip.
    fs_factor multiplica a fração radiante de superfície (1 = correlação original, gás natural)."""
    from scipy.optimize import brentq
    D_s = np.sqrt(4 * mdot / (np.pi * rho_inf * u_j))
    M = fuel.M
    # W = fração mássica de combustível na mistura estequiométrica com ar (definição de Chamberlain);
    # a correlação W(M) do artigo vale para alcanos; para misturas (H2, inertes...) usa-se Z_st exato
    W = stoichiometry(fuel)["Z_st"] if fuel.is_mixture else M / (15.816 * M + 0.0395)
    Ca = 0.024 * (G * D_s / u_j**2) ** (1 / 3)
    Cb, Cc = 0.2, (2.85 / W) ** (2 / 3)
    Y = brentq(lambda y: Ca * y ** (5 / 3) + Cb * y ** (2 / 3) - Cc, 1e-6, 1e6)
    L_b0 = Y * D_s
    L_b = L_b0 * (0.51 * np.exp(-0.4 * u_w) + 0.49) * (1 - 6.07e-3 * (theta_jv - 90))
    Ri = lambda ell: (G / (D_s**2 * u_j**2)) ** (1 / 3) * ell  # noqa: E731
    R = u_w / u_j
    base = (theta_jv - 90) * (1 - np.exp(-25.6 * R))
    if R <= 0.05:
        alpha = base + 8000 * R / Ri(L_b0)
    else:
        alpha = base + (134 + 1726 * np.sqrt(R - 0.026)) / Ri(L_b0)
    K = 0.185 * np.exp(-20 * R) + 0.015
    a_r = np.radians(alpha)
    b = L_b * np.sin(K * a_r) / np.sin(a_r) if abs(alpha) > 1e-9 else K * L_b
    R_l = np.sqrt(L_b**2 - b**2 * np.sin(a_r) ** 2) - b * np.cos(a_r)
    Cp = 1000 * np.exp(-100 * R) + 0.8
    W1 = D_s * (13.5 * np.exp(-6 * R) + 1.5) * (
        1 - (1 - np.sqrt(rho_inf / rho_j) / 15) * np.exp(-70 * Ri(D_s) * Cp * R))
    W2 = L_b * (0.18 * np.exp(-1.5 * R) + 0.31) * (1 - 0.47 * np.exp(-25 * R))
    A = np.pi / 4 * (W1**2 + W2**2) + np.pi / 2 * (W1 + W2) * np.sqrt(R_l**2 + ((W2 - W1) / 2) ** 2)
    F_s = (0.21 * np.exp(-0.00323 * u_j) + 0.11) * fs_factor
    Q = mdot * fuel.LHV
    return Chamberlain(D_s, W, Y, L_b0, L_b, R, Ri(L_b0), alpha, K, b, R_l, W1, W2, A, F_s, F_s * Q / A, Q)


def frustum_axis(ch: Chamberlain, tip: np.ndarray, wind_dir: np.ndarray = np.array([1.0, 0, 0])):
    """Centro da base, vetor do eixo (unitário) e base ortonormal do frustum de um tip vertical."""
    a_r = np.radians(ch.alpha)
    wd = wind_dir / np.linalg.norm(wind_dir)
    ax = np.sin(a_r) * wd + np.cos(a_r) * np.array([0, 0, 1.0])
    base = np.asarray(tip, float) + ch.b * np.array([0, 0, 1.0])
    e2 = np.cross(ax, wd); e2 = e2 / np.linalg.norm(e2) if np.linalg.norm(e2) > 1e-9 else np.array([0, 1.0, 0])
    e1 = np.cross(e2, ax)
    return base, ax, e1, e2


def frustum_panels(ch: Chamberlain, tip, n_ax: int = 48, n_th: int = 64, n_r: int = 12):
    """Painéis (centro, normal, área) da superfície lateral e dos dois discos."""
    base, ax, e1, e2 = frustum_axis(ch, tip)
    s = (np.arange(n_ax) + 0.5) / n_ax
    th = (np.arange(n_th) + 0.5) / n_th * 2 * np.pi
    S, TH = np.meshgrid(s, th, indexing="ij")
    r = (ch.W1 + (ch.W2 - ch.W1) * S) / 2
    slope = (ch.W2 - ch.W1) / 2 / ch.R_l
    radial = np.cos(TH)[..., None] * e1 + np.sin(TH)[..., None] * e2
    pts = base + (S[..., None] * ch.R_l) * ax + r[..., None] * radial
    nrm = radial - slope * ax
    nrm /= np.linalg.norm(nrm, axis=-1, keepdims=True)
    dA = r * (2 * np.pi / n_th) * (ch.R_l / n_ax) * np.sqrt(1 + slope**2)
    P, N, A = [pts.reshape(-1, 3)], [nrm.reshape(-1, 3)], [dA.ravel()]
    for frac, rad, sgn in ((0.0, ch.W1 / 2, -1.0), (1.0, ch.W2 / 2, 1.0)):
        rr = (np.arange(n_r) + 0.5) / n_r * rad
        RR, TT = np.meshgrid(rr, th, indexing="ij")
        c = base + frac * ch.R_l * ax
        p = c + RR[..., None] * (np.cos(TT)[..., None] * e1 + np.sin(TT)[..., None] * e2)
        P.append(p.reshape(-1, 3)); N.append(np.tile(sgn * ax, (p.size // 3, 1)))
        A.append((RR * (rad / n_r) * (2 * np.pi / n_th)).ravel())
    return np.vstack(P), np.vstack(N), np.concatenate(A)


def flame_box(ch: Chamberlain, H: float, grow: float = 1.2, pad: float = 3.0) -> dict:
    """Caixa que envolve a chama de Chamberlain (com folga) para a malha fina da LES:
    x a jusante, meia-largura em y e alturas relativas ao tip. A LES tende a inclinar e alongar
    um pouco a chama em relação a Chamberlain, por isso o fator `grow`."""
    tip = np.array([0.0, 0.0, H])
    base, ax, e1, _ = frustum_axis(ch, tip)
    top = base + ch.R_l * ax
    pts = np.array([base - ch.W1 / 2 * e1, base + ch.W1 / 2 * e1, top - ch.W2 / 2 * e1, top + ch.W2 / 2 * e1])
    rel = (pts - tip) * grow
    return {"x": (min(-pad, rel[:, 0].min() - pad), rel[:, 0].max() + pad),
            "y_half": max(6.0, 1.1 * ch.W2 / 2 + 2.0),
            "z": (-2.0, rel[:, 2].max() + pad)}


def flame_center_chamberlain(ch: Chamberlain, tip) -> np.ndarray:
    base, ax, *_ = frustum_axis(ch, tip)
    return base + 0.5 * ch.R_l * ax


# --------------------------------------------------------- transmissividade
def p_sat_water(T: float) -> float:
    """Pa, Antoine (TNO Yellow Book)."""
    return float(np.exp(23.18986 - 3816.42 / (T - 46.13)))


def tau_bagster(X, T_inf: float, RH: float):
    X = np.maximum(np.asarray(X, float), 1e-3)
    return np.minimum(1.0, 2.02 * (RH * p_sat_water(T_inf) * X) ** -0.09)


def tau_wayne(X, T_inf: float, RH: float, co2_ppm: float = 335.0):
    """Wayne (1991), como implementado no HyRAM+."""
    X = np.maximum(np.asarray(X, float), 1e-2)
    psat_mmHg = np.exp(20.386 - 5132.0 / T_inf)
    XH2O = RH * X * psat_mmHg * 2.88651e2 / T_inf
    XCO2 = X * 273.0 / T_inf * co2_ppm / 335.0
    lw, lc = np.log10(XH2O), np.log10(XCO2)
    return np.clip(1.006 - 0.01171 * lw - 0.02368 * lw**2 - 0.03188 * lc + 0.001164 * lc**2, 0.0, 1.0)


# --------------------------------------------------- fluxo nos receptores
def view_vector(receivers: np.ndarray, P: np.ndarray, N: np.ndarray, dA: np.ndarray,
                chunk: int = 512):
    """Seção 3.8: V = Σ cosβ_f dA/(π r²) ê (só a parte visível, cosβ_f > 0). F = n·V; F_max = |V|.
    Retorna V (n_rec, 3) e a distância média ponderada até a chama (para τ)."""
    receivers = np.atleast_2d(receivers)
    V = np.zeros_like(receivers, dtype=float)
    Xm = np.zeros(len(receivers))
    for i0 in range(0, len(receivers), chunk):
        r = receivers[i0:i0 + chunk]
        d = P[None, :, :] - r[:, None, :]
        dist = np.linalg.norm(d, axis=-1)
        e = d / dist[..., None]
        cos_f = np.maximum(-(e * N[None]).sum(-1), 0.0)
        w = cos_f * dA[None] / (np.pi * dist**2)
        V[i0:i0 + chunk] = (w[..., None] * e).sum(1)
        Xm[i0:i0 + chunk] = (w * dist).sum(1) / np.maximum(w.sum(1), 1e-30)
    return V, Xm


def q_chamberlain(receivers, ch: Chamberlain, tip, T_inf: float, RH: float,
                  normal: np.ndarray | None = None) -> np.ndarray:
    """Fluxo incidente [W/m2] pelo modelo de chama sólida; normal=None → pior orientação."""
    P, N, dA = frustum_panels(ch, tip)
    V, Xm = view_vector(np.asarray(receivers, float), P, N, dA)
    F = np.linalg.norm(V, axis=1) if normal is None else np.maximum(V @ np.asarray(normal, float), 0.0)
    return ch.SEP * F * tau_wayne(np.maximum(Xm - (ch.W1 + ch.W2) / 4, 1.0), T_inf, RH)


def q_point_source(receivers, center, F_rad: float, Q: float, T_inf: float, RH: float) -> np.ndarray:
    """Seção 3.5 (Hajek–Ludwig/API 521): q = τ F Q / (4π D²), pior orientação."""
    D = np.linalg.norm(np.atleast_2d(receivers) - np.asarray(center, float)[None], axis=1)
    return tau_wayne(D, T_inf, RH) * F_rad * Q / (4 * np.pi * D**2)


@dataclass
class Scenario:
    """Cenário do flare e todas as previsões semi-empíricas que servem de referência."""
    fuel: Fuel
    mdot: float = 12.6
    T_j: float = 311.0
    mach: float = 0.5
    H: float = 30.0
    u_w: float = 8.9
    T_inf: float = 298.15
    RH: float = 0.5
    p_atm: float = P_ATM     # pressão local (use props.pressure_at_altitude para sítios elevados)
    xrad_comp: bool = True   # corrige F_s de Chamberlain pela composição (radiant_fraction_factor)

    def __post_init__(self):
        self.tip = tip_conditions(self.fuel, self.mdot, self.T_j, self.mach, self.p_atm)
        self.rho_inf = air_density(self.T_inf, self.p_atm)
        self.fs_factor = radiant_fraction_factor(self.fuel) if self.xrad_comp else 1.0
        self.st = stoichiometry(self.fuel)
        self.Q = self.tip["Q"]
        self.L_api = api_flame_length(self.Q)
        self.cham = chamberlain(self.fuel, self.mdot, self.tip["u_j"], self.tip["rho_j"],
                                self.rho_inf, self.u_w, fs_factor=self.fs_factor)
        self.cham0 = chamberlain(self.fuel, self.mdot, self.tip["u_j"], self.tip["rho_j"],
                                 self.rho_inf, 0.0, fs_factor=self.fs_factor)
        self.tip_xyz = np.array([0.0, 0.0, self.H])

    def references(self, T_ad: float, rho_f: float | None = None) -> dict:
        """Correlações de referência. ρ_f padrão: produtos estequiométricos a T_ad (gás ideal)."""
        if rho_f is None:
            pr = products_per_mol(self.fuel)
            Mi = {"CO2": 44.009e-3, "H2O": 18.015e-3, "SO2": 64.058e-3, "N2": 28.014e-3}
            M_p = sum(pr[k] * Mi[k] for k in pr) / sum(pr.values())
            rho_f = self.p_atm * M_p / (R_U * T_ad)
        de = delichatsios_length(self.tip["u_j"], self.tip["d_j"], self.tip["rho_j"], self.rho_inf,
                                 self.st["Z_st"], T_ad, self.T_inf)
        mo = molina_xrad(self.fuel, self.mdot, de["L"], self.st["Z_st"], T_ad, rho_f)
        return {"delichatsios": de, "molina": mo, "L_api": self.L_api,
                "L_heskestad": heskestad_length(self.Q, self.tip["d_j"])}

In [ ]:
%%writefile flarekit/les.py
"""LES 3D de baixo Mach para um flare elevado em vento cruzado (PyTorch; GPU se houver).

Formulação (seções 4.1–4.6 do documento), no espírito do FDS:
  * baixo Mach, densidade variável: ρ = ρ(Z̃, s) da tabela β-PDF (química rápida + equilíbrio);
  * restrição de divergência  ∇·u = S_m/ρ − (1/ρ²)(dρ/dZ)[∇·(ρD_t∇Z) + S_m(1−Z)];
  * momento em forma advectiva (upwind de 3ª ordem), H = p̃/ρ, termo baroclínico defasado
    (H∇ρ/ρ), empuxo, Smagorinsky;
  * Poisson de coeficiente constante ∇²H = (∇·u* − D)/Δt, resolvido de forma direta pela
    diagonalização dos operadores 1D (autoproblema generalizado B v = λ M v em malha esticada);
  * fração de mistura com advecção TVD (Superbee), variância de submalha Z''² = C_Z Δ² |∇Z̃|²;
  * fonte de combustível volumétrica no tip com o fluxo de quantidade de movimento do jato real;
  * vento com perfil log e tensão de parede no solo; RK2 (SSP) com projeção em cada estágio.

Malha deslocada (MAC) e não uniforme por produto tensorial: fina (Δ_f) numa caixa em volta do
tip e da chama, crescendo geometricamente até Δ_max fora dela.
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass

import numpy as np
import scipy.linalg
import torch

from .props import G, BetaPDFTable

KAPPA = 0.41


# ----------------------------------------------------------------- utilidades
def _sl(a, axis, start, stop=None):
    idx = [slice(None)] * a.ndim
    idx[axis] = slice(start, stop)
    return a[tuple(idx)]


def _pad(a, axis, n=1):
    lo, hi = _sl(a, axis, 0, 1), _sl(a, axis, -1, None)
    return torch.cat([lo] * n + [a] + [hi] * n, dim=axis)


def _avg(a, axis):
    return 0.5 * (_sl(a, axis, 1, None) + _sl(a, axis, 0, -1))


def _to_faces(c, axis):
    """Centro → faces ao longo de axis (faces de borda recebem o valor da célula de borda)."""
    return _avg(_pad(c, axis), axis)


def _diff(a, axis):
    return _sl(a, axis, 1, None) - _sl(a, axis, 0, -1)


def _bshape(arr, axis, ndim=3):
    shape = [1] * ndim
    shape[axis] = -1
    return arr.reshape(shape)


def stretched_faces(lo, hi, f_lo, f_hi, d_fine, ratio=1.1, d_max=2.5):
    """Faces com Δ = d_fine em [f_lo, f_hi] e crescimento geométrico (ratio) até d_max fora."""
    nf = int(round((f_hi - f_lo) / d_fine))
    fine = f_lo + d_fine * np.arange(nf + 1)

    def grow(start, end, sign):
        out, x, d = [], start, d_fine
        while sign * (end - x) > 1e-9:
            d = min(d * ratio, d_max)
            x = x + sign * d
            out.append(x)
        if out:
            out[-1] = end
            if len(out) > 1 and abs(out[-1] - out[-2]) < 0.5 * d:
                out.pop(-2)
        return out

    up = grow(fine[-1], hi, +1)
    down = grow(fine[0], lo, -1)
    return np.concatenate([np.array(down[::-1]), fine, np.array(up)])


_PRESETS = {
    # GPU (Colab T4): Δ = 0,5 m perto da chama (~0,8 M células). Validado: L e inclinação
    # sob vento de 8,9 m/s a +2% e +10% de Chamberlain (1987)
    "gpu": dict(d_fine=0.5, d_max=3.0),
    # GPU, malha fina: Δ = 0,35 m (~1,8 M células)
    "gpu_fino": dict(d_fine=0.35),
    # GPU grande (A100/L4): Δ = 0,25 m (~4 M células)
    "gpu_ultra": dict(d_fine=0.25, ratio=1.08),
    # CPU: Δ = 0,7 m num domínio menor
    "cpu": dict(d_fine=0.7, d_max=3.5, x_range=(-30.0, 100.0), y_range=(-32.0, 32.0), z_top=90.0,
                receiver_dx=4.0, emitter_bin=2.0),
    # teste rápido (malha grossa: só para verificar o fluxo do notebook)
    "teste": dict(d_fine=1.5, d_max=3.0, x_range=(-24.0, 60.0), y_range=(-21.0, 21.0),
                  z_top=75.0, receiver_dx=6.0, emitter_bin=3.0),
}
# volume de referência da caixa fina [m³]: chamas maiores engrossam a malha na mesma proporção
# (mantém Δ relativo à chama e o número de células, ~1 M no preset "gpu")
FLAME_BOX_REF = 25000.0


@dataclass
class LESConfig:
    # domínio e malha
    x_range: tuple = (-40.0, 130.0)
    y_range: tuple = (-40.0, 40.0)
    z_top: float = 100.0
    d_fine: float = 0.35
    fine_x: tuple = (-4.0, 30.0)
    fine_y: tuple = (-6.0, 6.0)
    fine_z: tuple = (-2.0, 24.0)       # relativo à altura do tip
    ratio: float = 1.1
    d_max: float = 2.5
    # cenário
    H: float = 30.0            # altura do tip [m]
    u_ref: float = 8.9         # vento na altura do tip [m/s]
    z0: float = 0.03           # rugosidade aerodinâmica [m]
    mdot: float = 12.6         # vazão de combustível [kg/s]
    u_jet: float = 128.7       # velocidade real no tip (fluxo de quantidade de movimento)
    src_cells: int = 2         # lado da fonte, em células finas
    X_rad: float = 0.25        # fração radiante usada para escalar a emissão
    T_inf: float = 298.15
    RH: float = 0.5
    # modelos e numérica
    sgs: str = "smagorinsky"   # "smagorinsky" ou "wale"
    Cs: float = 0.10           # valor usual em escoamentos cisalhantes livres
    Cw: float = 0.5
    Sc_t: float = 0.7
    C_Z: float = 0.15
    cfl: float = 0.5
    dt_max: float = 0.05
    upwind: float = 0.0        # fração de upwind de 1ª ordem (0 = upwind de 3ª ordem puro)
    limiter: str = "superbee"  # limitador TVD de Z (Superbee, como no FDS); "mc" ou "vanleer"
    inflow_noise: float = 0.08
    jet_noise: float = 0.0     # impulsos aleatórios na fonte (desligado por padrão)
    baroclinic: bool = True
    # radiação no solo
    receiver_dx: float = 3.0
    receiver_z: float = 1.5
    receiver_box: tuple | None = None   # (x0, x1, y_meia) extra coberto por receptores, além do domínio
    emitter_bin: float = 1.5   # agrupa emissores em blocos deste tamanho [m]
    device: str = "auto"
    seed: int = 0

    @staticmethod
    def for_flame(name: str, box: dict, **kw) -> "LESConfig":
        """Preset com a caixa de malha fina e o domínio ajustados à chama prevista (semiempirical.flame_box)."""
        x0, x1 = box["x"]
        yh = box["y_half"]
        z0, z1 = box["z"]
        H = kw.get("H", 30.0)
        dom = dict(fine_x=(x0, x1), fine_y=(-yh, yh), fine_z=(z0, z1),
                   x_range=(min(-40.0, x0 - 30.0), max(130.0, x1 + 90.0)),
                   y_range=(-max(40.0, yh + 30.0), max(40.0, yh + 30.0)),
                   z_top=max(100.0, H + z1 + 40.0))
        if name == "teste":
            dom.update(x_range=(min(-24.0, x0 - 15.0), max(60.0, x1 + 30.0)), y_range=(-21.0, 21.0),
                       z_top=max(75.0, H + z1 + 15.0))
        elif name == "cpu":   # domínio menor que o da GPU (mesma lógica, folgas reduzidas)
            dom.update(x_range=(min(-30.0, x0 - 15.0), max(100.0, x1 + 60.0)),
                       y_range=(-max(32.0, yh + 20.0), max(32.0, yh + 20.0)), z_top=max(90.0, H + z1 + 30.0))
        # chamas grandes (vazões de emergência, tochas altas): Δ cresce com o tamanho da chama
        scale = max(1.0, ((x1 - x0) * 2 * yh * (z1 - z0) / FLAME_BOX_REF) ** (1 / 3))
        if scale > 1.0:
            base = LESConfig(**{**_PRESETS[name]})
            for k in ("d_fine", "d_max", "receiver_dx", "emitter_bin"):
                dom[k] = getattr(base, k) * scale
        dom.update(kw)
        return LESConfig.preset(name, **dom)

    @staticmethod
    def preset(name: str, **kw) -> "LESConfig":
        presets = _PRESETS
        cfg = presets[name].copy()
        if kw.get("u_ref", 1.0) <= 0.0 and name != "teste" and "fine_x" not in kw:
            # sem vento: chama vertical e mais longa → caixa fina centrada e mais alta
            cfg.update(x_range=(-50.0, 50.0), y_range=(-50.0, 50.0), z_top=110.0,
                       fine_x=(-7.0, 7.0), fine_y=(-7.0, 7.0), fine_z=(-2.0, 56.0))
        cfg.update(kw)
        return LESConfig(**cfg)


class FlareLES:
    def __init__(self, cfg: LESConfig, table: BetaPDFTable, Q: float):
        self.cfg = cfg
        self.Q = Q   # potência do flare (ṁ·PCI) [W], normaliza a radiação
        dev = cfg.device
        if dev == "auto":
            dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.dev = torch.device(dev)
        self.ft = torch.float32
        torch.manual_seed(cfg.seed)
        t = lambda a: torch.as_tensor(np.asarray(a, dtype=np.float64), dtype=self.ft, device=self.dev)  # noqa: E731
        self._t = t
        df = cfg.d_fine

        # ---- malha (x = 0, y = 0 e z = H são faces)
        snap = lambda a: round(a / df) * df  # noqa: E731
        H = snap(cfg.H)
        self.xf = stretched_faces(cfg.x_range[0], cfg.x_range[1], snap(cfg.fine_x[0]), snap(cfg.fine_x[1]),
                                  df, cfg.ratio, cfg.d_max)
        self.yf = stretched_faces(cfg.y_range[0], cfg.y_range[1], snap(cfg.fine_y[0]), snap(cfg.fine_y[1]),
                                  df, cfg.ratio, cfg.d_max)
        self.zf = stretched_faces(0.0, cfg.z_top, H + snap(cfg.fine_z[0]), H + snap(cfg.fine_z[1]),
                                  df, cfg.ratio, cfg.d_max)
        self.nx, self.ny, self.nz = len(self.xf) - 1, len(self.yf) - 1, len(self.zf) - 1
        self.n_cells = self.nx * self.ny * self.nz
        self.xc, self.yc, self.zc = [0.5 * (f[1:] + f[:-1]) for f in (self.xf, self.yf, self.zf)]
        self.wind = cfg.u_ref > 0.0
        # espaçamentos: largura das células (n), distância centro-centro nas faces (n+1)
        # e largura do volume de controle das faces (n+1)
        self.dc, self.df_, self.dcf = [], [], []
        for f, c in ((self.xf, self.xc), (self.yf, self.yc), (self.zf, self.zc)):
            dc = np.diff(f)
            self.dc.append(dc)
            self.df_.append(np.concatenate([[dc[0]], np.diff(c), [dc[-1]]]))
            self.dcf.append(np.concatenate([[dc[0]], 0.5 * (dc[1:] + dc[:-1]), [dc[-1]]]))
        self.Dc = [_bshape(t(a), ax) for ax, a in enumerate(self.dc)]
        self.Df = [_bshape(t(a), ax) for ax, a in enumerate(self.df_)]
        self.Dcf = [_bshape(t(a), ax) for ax, a in enumerate(self.dcf)]
        vol = np.einsum("i,j,k->ijk", *self.dc)
        self.vol = t(vol)
        self.delta = self.vol ** (1.0 / 3.0)
        self.inv_d2 = 1 / self.Dc[0] ** 2 + 1 / self.Dc[1] ** 2 + 1 / self.Dc[2] ** 2

        # ---- tabela termoquímica
        self.tab = table
        self.tab_rho = t(table.rho); self.tab_drho = t(table.drho_dZ); self.tab_T = t(table.T)
        self.tab_e = t(table.emission)
        self.nZt, self.ns = table.rho.shape
        self.s_max = float(table.s[-1])
        self.rho_inf = float(table.rho_inf)
        self.Z_st = float(table.Z_st)

        # ---- campos
        nx, ny, nz = self.nx, self.ny, self.nz
        z = lambda *s: torch.zeros(*s, dtype=self.ft, device=self.dev)  # noqa: E731
        self.u, self.v, self.w = z(nx + 1, ny, nz), z(nx, ny + 1, nz), z(nx, ny, nz + 1)
        self.Z, self.Hp = z(nx, ny, nz), z(nx, ny, nz)

        # ---- vento (perfil log ajustado para u_ref na altura do tip)
        self.u_star = KAPPA * cfg.u_ref / math.log((H + cfg.z0) / cfg.z0) if self.wind else 0.0
        U_prof = self.u_star / KAPPA * np.log((self.zc + cfg.z0) / cfg.z0)
        self.U_in = t(U_prof).view(1, 1, nz).expand(1, ny, nz).clone()
        if self.wind:
            self.u[:] = self.U_in
        self.Cd_wall = (KAPPA / math.log((0.5 * self.dc[2][0]) / cfg.z0)) ** 2
        self.noise = z(1, ny, nz)

        # ---- fonte de combustível (bloco src×src×1 logo acima do tip)
        i0 = int(np.argmin(np.abs(self.xf))); j0 = int(np.argmin(np.abs(self.yf)))
        n2 = cfg.src_cells // 2
        self.kH = int(np.argmin(np.abs(self.zf - H)))
        self.isrc = slice(i0 - n2, i0 - n2 + cfg.src_cells)
        self.jsrc = slice(j0 - n2, j0 - n2 + cfg.src_cells)
        Sm = z(nx, ny, nz)
        V_src = float(vol[self.isrc, self.jsrc, self.kH].sum())
        Sm[self.isrc, self.jsrc, self.kH] = cfg.mdot / V_src
        self.Sm = Sm
        self.Sm_u, self.Sm_v, self.Sm_w = _to_faces(Sm, 0), _to_faces(Sm, 1), _to_faces(Sm, 2)
        self.src_u, self.src_v = self.Sm_u.gt(0).to(self.ft), self.Sm_v.gt(0).to(self.ft)
        self.tip = np.array([0.0, 0.0, self.zf[self.kH]])
        self.src_area = float(self.dc[0][self.isrc].sum() * self.dc[1][self.jsrc].sum())

        # ---- Poisson (diagonalização 1D, autoproblema generalizado)
        bx = ("N" if self.wind else "D", "D")
        self.bc = (bx, ("D", "D"), ("N", "D"))
        self._poisson_setup()

        # ---- receptores no solo
        # (os receptores são só pontos de avaliação: podem ir além do domínio do escoamento para cobrir
        # toda a zona de 1,58 kW/m²)
        rx0, rx1, ry0, ry1 = self.xf[0], self.xf[-1], self.yf[0], self.yf[-1]
        if cfg.receiver_box is not None:
            bx0, bx1, byh = cfg.receiver_box
            rx0, rx1, ry0, ry1 = min(rx0, bx0), max(rx1, bx1), min(ry0, -byh), max(ry1, byh)
        rx = np.arange(rx0 + cfg.receiver_dx / 2, rx1, cfg.receiver_dx)
        ry = np.arange(ry0 + cfg.receiver_dx / 2, ry1, cfg.receiver_dx)
        self.rec_x, self.rec_y = rx, ry
        RX, RY = np.meshgrid(rx, ry, indexing="ij")
        self.receivers = t(np.stack([RX, RY, np.full_like(RX, cfg.receiver_z)], -1).reshape(-1, 3))

        # ---- coordenadas das células
        Xc, Yc, Zc = np.meshgrid(self.xc, self.yc, self.zc, indexing="ij")
        self.cell_xyz = t(np.stack([Xc, Yc, Zc], -1)).reshape(-1, 3)
        self.dist_tip = torch.linalg.norm(self.cell_xyz - t(self.tip), dim=-1).reshape(nx, ny, nz)
        self.j_mid = int(np.argmin(np.abs(self.yc)))

        # ---- estado do tempo e médias
        self.time = 0.0; self.step_n = 0; self.dt = cfg.dt_max; self.wall = 0.0
        self._numax = 0.0; self._last_cfl = 0.0
        self.history = {k: [] for k in ("t", "L", "tilt", "qmax", "Tmax", "dt", "cfl")}
        self.avg_n = 0.0
        self.avg_T, self.avg_I = z(nx, ny, nz), z(nx, ny, nz)
        self.avg_q = z(len(rx) * len(ry))
        self.q_inst = torch.zeros_like(self.avg_q)
        self._update_props(self.Z)

    # ---------------------------------------------------------------- Poisson
    def _lap1d(self, axis):
        dc, dfa = self.dc[axis], self.df_[axis]
        lo, hi = self.bc[axis]
        n = len(dc)
        # A = M⁻¹B com M = diag(Δc) e B simétrica: B[i,i±1] = 1/Δf
        B = np.zeros((n, n))
        off = 1.0 / dfa[1:-1]
        idx = np.arange(n - 1)
        B[idx, idx + 1] = off
        B[idx + 1, idx] = off
        diag = np.zeros(n)
        diag[:-1] -= off
        diag[1:] -= off
        if lo == "D":
            diag[0] -= 2.0 / dc[0]
        if hi == "D":
            diag[-1] -= 2.0 / dc[-1]
        B[np.arange(n), np.arange(n)] = diag
        lam, V = scipy.linalg.eigh(B, np.diag(dc))      # Vᵀ M V = I  →  V⁻¹ = Vᵀ M
        return self._t(lam), self._t(V), self._t(V.T * dc[None, :])

    def _poisson_setup(self):
        (lx, Vx, Wx), (ly, Vy, Wy), (lz, Vz, Wz) = (self._lap1d(0), self._lap1d(1), self._lap1d(2))
        self.V = (Vx, Vy, Vz)
        self.Vi = (Wx, Wy, Wz)
        self.lam = lx.view(-1, 1, 1) + ly.view(1, -1, 1) + lz.view(1, 1, -1)

    def poisson(self, rhs):
        Wx, Wy, Wz = self.Vi
        Vx, Vy, Vz = self.V
        r = torch.einsum("ai,ijk->ajk", Wx, rhs)
        r = torch.einsum("bj,ajk->abk", Wy, r)
        r = torch.einsum("ck,abk->abc", Wz, r)
        r = r / self.lam
        r = torch.einsum("ia,abc->ibc", Vx, r)
        r = torch.einsum("jb,ibc->ijc", Vy, r)
        return torch.einsum("kc,ijc->ijk", Vz, r)

    def _grad_face(self, Hc, axis):
        lo, hi = self.bc[axis]
        g_lo = _sl(Hc, axis, 0, 1) * (1.0 if lo == "N" else -1.0)
        g_hi = _sl(Hc, axis, -1, None) * (1.0 if hi == "N" else -1.0)
        return _diff(torch.cat([g_lo, Hc, g_hi], dim=axis), axis) / self.Df[axis]

    def divergence(self, u, v, w):
        return _diff(u, 0) / self.Dc[0] + _diff(v, 1) / self.Dc[1] + _diff(w, 2) / self.Dc[2]

    def project(self, u, v, w, D, coef):
        rhs = (self.divergence(u, v, w) - D) / coef
        Hc = self.poisson(rhs)
        return (u - coef * self._grad_face(Hc, 0), v - coef * self._grad_face(Hc, 1),
                w - coef * self._grad_face(Hc, 2), Hc)

    # ------------------------------------------------------------ propriedades
    def _lookup(self, tab, Z, s):
        fz = Z.clamp(0, 1) * (self.nZt - 1)
        fs = (s / self.s_max).clamp(0, 1) * (self.ns - 1)
        iz = fz.floor().clamp(max=self.nZt - 2).long()
        js = fs.floor().clamp(max=self.ns - 2).long()
        wz = fz - iz
        ws = fs - js
        f = tab.reshape(-1)
        b = iz * self.ns + js
        return ((1 - wz) * (1 - ws) * f[b] + wz * (1 - ws) * f[b + self.ns]
                + (1 - wz) * ws * f[b + 1] + wz * ws * f[b + self.ns + 1])

    def _cgrad(self, q, axis):
        """Derivada centrada de um campo de centro, ao longo de axis."""
        qp = _pad(q, axis)
        dist = _sl(self.Df[axis], axis, 0, -1) + _sl(self.Df[axis], axis, 1, None)
        return (_sl(qp, axis, 2, None) - _sl(qp, axis, 0, -2)) / dist

    def _svar(self, Z):
        g2 = self._cgrad(Z, 0) ** 2 + self._cgrad(Z, 1) ** 2 + self._cgrad(Z, 2) ** 2
        var = self.cfg.C_Z * self.delta ** 2 * g2
        return (var / (Z * (1 - Z) + 1e-8)).clamp(0, self.s_max)

    def _update_props(self, Z):
        s = self._svar(Z)
        self.s = s
        self.rho = self._lookup(self.tab_rho, Z, s)
        self.drho = self._lookup(self.tab_drho, Z, s)

    # --------------------------------------------------------- turbulência
    def _nu_t(self, u, v, w):
        """Viscosidade de submalha: Smagorinsky ou WALE (Nicoud & Ducros 1999)."""
        uc, vc, wc = _avg(u, 0), _avg(v, 1), _avg(w, 2)
        g = [[None] * 3 for _ in range(3)]
        for i, (q, qc) in enumerate(((u, uc), (v, vc), (w, wc))):
            for j in range(3):
                g[i][j] = _diff(q, j) / self.Dc[j] if j == i else self._cgrad(qc, j)
        SS = 0.0
        for i in range(3):
            for j in range(i, 3):
                Sij = 0.5 * (g[i][j] + g[j][i])
                SS = SS + (1.0 if i == j else 2.0) * Sij * Sij
        if self.cfg.sgs == "wale":
            g2 = [[sum(g[i][k] * g[k][j] for k in range(3)) for j in range(3)] for i in range(3)]
            tr = (g2[0][0] + g2[1][1] + g2[2][2]) / 3.0
            SdSd = 0.0
            for i in range(3):
                for j in range(i, 3):
                    Sd = 0.5 * (g2[i][j] + g2[j][i]) - (tr if i == j else 0.0)
                    SdSd = SdSd + (1.0 if i == j else 2.0) * Sd * Sd
            num = SdSd ** 1.5
            den = SS ** 2.5 + SdSd ** 1.25 + 1e-12
            return (self.cfg.Cw * self.delta) ** 2 * num / den
        return (self.cfg.Cs * self.delta) ** 2 * torch.sqrt(2.0 * SS)

    # ------------------------------------------------------------- momento
    def _spacings(self, comp, axis):
        """Espaçamento local h (advecção), distâncias h−/h+ e largura de controle (difusão)
        para a componente de velocidade comp ao longo de axis."""
        if comp == axis:   # variável nas faces ao longo de axis: vizinhas separadas por Δc
            dc = self.Dc[axis]
            hm = torch.cat([_sl(dc, axis, 0, 1), dc], dim=axis)
            hp = torch.cat([dc, _sl(dc, axis, -1, None)], dim=axis)
            return 0.5 * (hm + hp), hm, hp, self.Dcf[axis]
        dfa = self.Df[axis]
        return self.Dc[axis], _sl(dfa, axis, 0, -1), _sl(dfa, axis, 1, None), self.Dc[axis]

    def _adv_visc(self, comp, q, adv, mu_q, rho_q):
        """Upwind de 3ª ordem (= central de 4ª ordem + dissipação |a|h³/12 ∂⁴q) e ∇·(μ∇q)/ρ."""
        beta = self.cfg.upwind
        A = 0.0
        Vs = 0.0
        for axis, a in enumerate(adv):
            h, hm, hp, hc = self._spacings(comp, axis)
            qp = _pad(q, axis, 2)
            n = q.shape[axis]
            qm2, qm1 = _sl(qp, axis, 0, n), _sl(qp, axis, 1, n + 1)
            qp1, qp2 = _sl(qp, axis, 3, n + 3), _sl(qp, axis, 4, n + 4)
            dpos = (2 * qp1 + 3 * q - 6 * qm1 + qm2) / (6 * h)
            dneg = (-qp2 + 6 * qp1 - 3 * q - 2 * qm1) / (6 * h)
            d3 = torch.where(a >= 0, dpos, dneg)
            if beta > 0:
                d1 = torch.where(a >= 0, (q - qm1) / hm, (qp1 - q) / hp)
                d3 = (1 - beta) * d3 + beta * d1
            A = A + a * d3
            mp = _pad(mu_q, axis)
            mu_p = 0.5 * (mu_q + _sl(mp, axis, 2, None))
            mu_m = 0.5 * (mu_q + _sl(mp, axis, 0, -2))
            Vs = Vs + (mu_p * (qp1 - q) / hp - mu_m * (q - qm1) / hm) / hc
        return A - Vs / rho_q

    def _momentum_rhs(self, u, v, w, rho, Hc):
        cfg = self.cfg
        mu = rho * (self._nu_t(u, v, w) + 1.5e-5)
        uc, vc, wc = _avg(u, 0), _avg(v, 1), _avg(w, 2)
        ru, rv, rw = _to_faces(rho, 0), _to_faces(rho, 1), _to_faces(rho, 2)
        Fu = self._adv_visc(0, u, (u, _to_faces(vc, 0), _to_faces(wc, 0)), _to_faces(mu, 0), ru)
        Fv = self._adv_visc(1, v, (_to_faces(uc, 1), v, _to_faces(wc, 1)), _to_faces(mu, 1), rv)
        Fw = self._adv_visc(2, w, (_to_faces(uc, 2), _to_faces(vc, 2), w), _to_faces(mu, 2), rw)
        Fw = Fw + G * (rw - self.rho_inf) / rw                     # empuxo
        if cfg.baroclinic:                                          # −p̃∇(1/ρ) = H∇ρ/ρ
            Fu = Fu + _to_faces(Hc, 0) * _diff(_pad(rho, 0), 0) / self.Df[0] / ru
            Fv = Fv + _to_faces(Hc, 1) * _diff(_pad(rho, 1), 1) / self.Df[1] / rv
            Fw = Fw + _to_faces(Hc, 2) * _diff(_pad(rho, 2), 2) / self.Df[2] / rw
        # fonte de combustível: ρ Du/Dt += S_m (u_inj − u), com u_inj = (0, 0, u_jet)
        Fu = Fu + self.Sm_u * u / ru
        Fv = Fv + self.Sm_v * v / rv
        Fw = Fw - self.Sm_w * (cfg.u_jet - w) / rw
        if self.wind:                                               # tensão de parede (lei log)
            h0 = float(self.dc[2][0])
            u0, v0 = _sl(u, 2, 0, 1), _sl(v, 2, 0, 1)
            spd_u = torch.sqrt(u0 ** 2 + _to_faces(_sl(vc, 2, 0, 1), 0) ** 2) + 1e-6
            spd_v = torch.sqrt(_to_faces(_sl(uc, 2, 0, 1), 1) ** 2 + v0 ** 2) + 1e-6
            Fu = torch.cat([_sl(Fu, 2, 0, 1) + self.Cd_wall * spd_u * u0 / h0, _sl(Fu, 2, 1, None)], 2)
            Fv = torch.cat([_sl(Fv, 2, 0, 1) + self.Cd_wall * spd_v * v0 / h0, _sl(Fv, 2, 1, None)], 2)
        return Fu, Fv, Fw, mu

    # -------------------------------------------------- fração de mistura
    def _tvd_face(self, Z, a, axis):
        n = Z.shape[axis]
        Zp = _pad(Z, axis, 2)
        ZLL, ZL = _sl(Zp, axis, 0, n + 1), _sl(Zp, axis, 1, n + 2)
        ZR, ZRR = _sl(Zp, axis, 2, n + 3), _sl(Zp, axis, 3, n + 4)

        lim = self.cfg.limiter

        def vl(num, den):
            den = torch.where(den.abs() < 1e-12, torch.full_like(den, 1e-12), den)
            r = num / den
            if lim == "superbee":
                return torch.maximum(torch.clamp(2 * r, max=1.0), torch.clamp(r, max=2.0)).clamp(min=0.0)
            if lim == "mc":
                return torch.minimum(torch.minimum(2 * r, 0.5 * (1 + r)), torch.full_like(r, 2.0)).clamp(min=0.0)
            return (r + r.abs()) / (1 + r.abs())
        Zpos = ZL + 0.5 * vl(ZL - ZLL, ZR - ZL) * (ZR - ZL)
        Zneg = ZR + 0.5 * vl(ZR - ZRR, ZL - ZR) * (ZL - ZR)
        Zf = torch.where(a >= 0, Zpos, Zneg)
        lo, hi = _sl(Zf, axis, 0, 1), _sl(Zf, axis, -1, None)      # bordas: entra ar (Z = 0)
        lo = torch.where(_sl(a, axis, 0, 1) > 0, torch.zeros_like(lo), lo)
        hi = torch.where(_sl(a, axis, -1, None) < 0, torch.zeros_like(hi), hi)
        return torch.cat([lo, _sl(Zf, axis, 1, -1), hi], dim=axis)

    def _z_adv(self, Z, u, v, w):
        """−∇·(uZ) + Z∇·u com faces TVD."""
        adv = 0.0
        for axis, a in enumerate((u, v, w)):
            adv = adv + _diff(a * self._tvd_face(Z, a, axis), axis) / self.Dc[axis]
        return -adv + Z * self.divergence(u, v, w)

    def _z_diff_src(self, Z, mu):
        """∇·(ρD_t∇Z) + S_m(1 − Z)  [kg/(m3 s)]."""
        rhoD = mu / self.cfg.Sc_t
        dif = 0.0
        for axis in range(3):
            flux = _to_faces(rhoD, axis) * _diff(_pad(Z, axis), axis) / self.Df[axis]
            dif = dif + _diff(flux, axis) / self.Dc[axis]
        return dif + self.Sm * (1.0 - Z)

    def _divergence_target(self, rho, drho, DZ_rho):
        return self.Sm / rho - drho * DZ_rho / (rho * rho)

    # --------------------------------------------------------------- passo
    def _apply_bc(self, u, v, w):
        if self.wind:
            u = torch.cat([self.U_in * (1.0 + self.noise), u[1:]], 0)
        w = torch.cat([torch.zeros_like(w[:, :, :1]), w[:, :, 1:]], 2)
        return u, v, w

    def _compute_dt(self):
        uc, vc, wc = _avg(self.u, 0).abs(), _avg(self.v, 1).abs(), _avg(self.w, 2).abs()
        rate = (uc / self.Dc[0] + vc / self.Dc[1] + wc / self.Dc[2]).max().item()
        dt = self.cfg.cfl / max(rate, 1e-6)
        if self._numax > 0:
            dt = min(dt, 0.45 / self._numax)
        dt = min(dt, self.cfg.dt_max)
        return dt, rate * dt

    def step(self):
        t0 = time.perf_counter()
        dt, cfl = self._compute_dt()
        self.dt = dt
        cfg = self.cfg
        if self.wind and cfg.inflow_noise > 0:
            # ruído de entrada: AR(1) no tempo, suavizado na vertical
            eps = torch.randn_like(self.noise)
            eps = torch.nn.functional.avg_pool1d(eps.reshape(-1, 1, self.nz), 5, 1, 2).reshape(self.noise.shape)
            a = math.exp(-dt / 2.0)
            self.noise = a * self.noise + math.sqrt(1 - a * a) * cfg.inflow_noise * eps * 2.2
        if cfg.jet_noise > 0:
            # semente de turbulência do jato: impulsos horizontais aleatórios nas faces da fonte
            amp = cfg.jet_noise * cfg.u_jet ** 0.5 * 3.0 * math.sqrt(dt)
            self.u = self.u + self.src_u * torch.randn_like(self.u) * amp
            self.v = self.v + self.src_v * torch.randn_like(self.v) * amp
        u0, v0, w0, Z0, rho0, H0 = self.u, self.v, self.w, self.Z, self.rho, self.Hp
        # ---- estágio 1 (preditor)
        Fu, Fv, Fw, mu = self._momentum_rhs(u0, v0, w0, rho0, H0)
        RZ = self._z_adv(Z0, u0, v0, w0) + self._z_diff_src(Z0, mu) / rho0
        Z1 = (Z0 + dt * RZ).clamp(0, 1)
        self._update_props(Z1)
        rho1, drho1 = self.rho, self.drho
        D1 = self._divergence_target(rho1, drho1, self._z_diff_src(Z1, mu * (rho1 / rho0)))
        u1, v1, w1 = self._apply_bc(u0 - dt * Fu, v0 - dt * Fv, w0 - dt * Fw)
        u1, v1, w1, H1 = self.project(u1, v1, w1, D1, dt)
        # ---- estágio 2 (corretor)
        Fu, Fv, Fw, mu1 = self._momentum_rhs(u1, v1, w1, rho1, H1)
        RZ = self._z_adv(Z1, u1, v1, w1) + self._z_diff_src(Z1, mu1) / rho1
        Z2 = (0.5 * (Z0 + Z1 + dt * RZ)).clamp(0, 1)
        self._update_props(Z2)
        rho2, drho2 = self.rho, self.drho
        D2 = self._divergence_target(rho2, drho2, self._z_diff_src(Z2, mu1 * (rho2 / rho1)))
        u2, v2, w2 = self._apply_bc(0.5 * (u0 + u1 - dt * Fu), 0.5 * (v0 + v1 - dt * Fv),
                                    0.5 * (w0 + w1 - dt * Fw))
        u2, v2, w2, H2 = self.project(u2, v2, w2, D2, 0.5 * dt)
        self.u, self.v, self.w, self.Z, self.Hp = u2, v2, w2, Z2, H2
        self._numax = ((mu1 / rho1) / cfg.Sc_t * 2.0 * self.inv_d2).max().item()
        if not math.isfinite(self._numax) or not torch.isfinite(self.w).all():
            raise FloatingPointError(f"a simulação divergiu em t = {self.time:.2f} s")
        self.time += dt
        self.step_n += 1
        if self.dev.type == "cuda":
            torch.cuda.synchronize()
        self.wall += time.perf_counter() - t0
        self._last_cfl = cfl
        return dt

    # ---------------------------------------------------------- diagnósticos
    def fields(self):
        """T̃, emissão luminosa (não normalizada) e máscara da chama (Z̃ ≥ Z_st)."""
        T = self._lookup(self.tab_T, self.Z, self.s)
        e = self._lookup(self.tab_e, self.Z, self.s)
        return T, e, self.Z >= self.Z_st

    def ground_flux(self, e):
        """Fluxo incidente no solo (pior orientação, seção 3.8): soma volumétrica dos emissores
        com transmissividade de Wayne, potência total escalada para X_rad·Q (como no FDS)."""
        cfg = self.cfg
        P = (e * self.vol).reshape(-1)
        keep = P > 1e-4 * P.max().clamp(min=1e-30)
        P, xyz = P[keep], self.cell_xyz[keep]
        if len(P) == 0:
            return torch.zeros(len(self.receivers), dtype=self.ft, device=self.dev)
        # agrupa emissores em blocos (centroide ponderado pela potência)
        key = torch.floor(xyz / cfg.emitter_bin).long()
        key = key - key.min(0).values
        dims = key.max(0).values + 1
        lin = (key[:, 0] * dims[1] + key[:, 1]) * dims[2] + key[:, 2]
        uniq, inv = torch.unique(lin, return_inverse=True)
        Pb = torch.zeros(len(uniq), dtype=self.ft, device=self.dev).index_add_(0, inv, P)
        Xb = torch.zeros(len(uniq), 3, dtype=self.ft, device=self.dev).index_add_(0, inv, P[:, None] * xyz)
        Xb = Xb / Pb[:, None]
        scale = cfg.X_rad * self.Q / Pb.sum()
        psat = math.exp(20.386 - 5132.0 / cfg.T_inf)
        V = torch.zeros_like(self.receivers)
        for i0 in range(0, len(Pb), 2048):
            dv = Xb[None, i0:i0 + 2048] - self.receivers[:, None]
            r = torch.linalg.norm(dv, dim=-1).clamp(min=0.5)
            lw = torch.log10(cfg.RH * r * psat * 2.88651e2 / cfg.T_inf)
            lc = torch.log10(r * 273.0 / cfg.T_inf)
            tau = (1.006 - 0.01171 * lw - 0.02368 * lw ** 2 - 0.03188 * lc + 0.001164 * lc ** 2).clamp(0, 1)
            wgt = Pb[None, i0:i0 + 2048] * tau / (4 * math.pi * r ** 3)
            V += (wgt[..., None] * dv).sum(1)
        return torch.linalg.norm(V, dim=1) * scale

    def _centroid_tilt(self, mask):
        if not mask.any():
            return float("nan")
        c = self.cell_xyz[mask.reshape(-1)].mean(0).cpu().numpy() - self.tip
        return math.degrees(math.atan2(math.hypot(c[0], c[1]), max(c[2], 1e-6)))

    def diagnostics(self, averaging: bool):
        T, e, mask = self.fields()
        q = self.ground_flux(e)
        self.q_inst = q
        L = torch.where(mask, self.dist_tip, torch.zeros_like(self.dist_tip)).max().item()
        h = self.history
        h["t"].append(self.time); h["L"].append(L); h["tilt"].append(self._centroid_tilt(mask))
        h["qmax"].append(q.max().item()); h["Tmax"].append(T.max().item()); h["dt"].append(self.dt)
        h["cfl"].append(self._last_cfl)
        if averaging:
            self.avg_n += 1.0
            self.avg_T += T
            self.avg_I += mask.to(self.ft)
            self.avg_q += q
        return T, e, mask, q

    def mean_flame_length(self, threshold: float = 0.5):
        if self.avg_n == 0:
            return float("nan")
        I = self.avg_I / self.avg_n
        return torch.where(I >= threshold, self.dist_tip, torch.zeros_like(self.dist_tip)).max().item()

    def mean_tilt(self, threshold: float = 0.5):
        if self.avg_n == 0:
            return float("nan")
        return self._centroid_tilt(self.avg_I / self.avg_n >= threshold)

    def q_grid(self, q):
        return q.reshape(len(self.rec_x), len(self.rec_y)).cpu().numpy()

    def slice_y0(self, f):
        return f[:, self.j_mid, :].float().cpu().numpy()

    def mass_balance(self):
        """Massa no domínio e vazão volumétrica líquida pelas fronteiras."""
        m = (self.rho * self.vol).sum().item()
        A_x = torch.outer(self._t(self.dc[1]), self._t(self.dc[2]))
        A_y = torch.outer(self._t(self.dc[0]), self._t(self.dc[2]))
        A_z = torch.outer(self._t(self.dc[0]), self._t(self.dc[1]))
        out = ((self.u[-1] - self.u[0]) * A_x).sum() + ((self.v[:, -1] - self.v[:, 0]) * A_y).sum() \
            + (self.w[:, :, -1] * A_z).sum()
        return m, out.item()

    def summary(self) -> str:
        dmin = min(float(d.min()) for d in self.dc)
        dmax = max(float(d.max()) for d in self.dc)
        return (f"{self.nx}×{self.ny}×{self.nz} = {self.n_cells / 1e6:.2f} M células · Δ {dmin:.2f}–{dmax:.2f} m · "
                f"fonte {self.src_area:.2f} m² · {self.dev.type.upper()}")

In [ ]:
%%writefile flarekit/dashboard.py
"""Painel ao vivo da LES e gravação do vídeo (MP4) quadro a quadro.

O mesmo quadro desenhado para o vídeo é mostrado no notebook enquanto o solver roda,
de modo que os gráficos acompanham a solução em tempo real.
"""
from __future__ import annotations

import io
import math
import time

import numpy as np
from matplotlib import colors as mcolors
from matplotlib.animation import FFMpegWriter
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

from . import semiempirical as se

BG, PANEL, FG, MUTED, GRID = "#0d1117", "#131a22", "#e6edf3", "#8b98a5", "#263241"
ACCENT, LES_C, CHAM_C, PT_C = "#f0a35e", "#ffb347", "#7fd1ff", "#b9c2cc"
API_LEVELS = [(1.58, "#6cc08f"), (4.73, "#e0b24a"), (6.31, "#f08a4b"), (9.46, "#f2766b")]


def _style(ax, title=None):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.xaxis.label.set_color(MUTED)
    ax.yaxis.label.set_color(MUTED)
    if title:
        ax.set_title(title, color=FG, fontsize=11, loc="left", pad=6, fontweight="bold")


def frustum_outline_xz(ch: se.Chamberlain, tip):
    """Trapézio do frustum de Chamberlain no plano y = 0 (vento em +x)."""
    base, ax, e1, _ = se.frustum_axis(ch, np.asarray(tip, float))
    top = base + ch.R_l * ax
    pts = [base - ch.W1 / 2 * e1, top - ch.W2 / 2 * e1, top + ch.W2 / 2 * e1, base + ch.W1 / 2 * e1,
           base - ch.W1 / 2 * e1]
    return np.array([[p[0], p[2]] for p in pts])


class LiveDashboard:
    def __init__(self, les, sc: se.Scenario, title: str, view_x=(-25.0, 95.0), view_z=(0.0, 80.0),
                 dpi: int = 120, live: bool = True):
        self.les, self.sc, self.live = les, sc, live
        self.fig = Figure(figsize=(16, 9), dpi=dpi, facecolor=BG)
        FigureCanvasAgg(self.fig)
        gs = GridSpec(3, 5, figure=self.fig, left=0.045, right=0.985, top=0.865, bottom=0.07,
                      hspace=0.45, wspace=0.5)
        self.ax_s = self.fig.add_subplot(gs[0:2, 0:3])
        self.ax_m = self.fig.add_subplot(gs[0:2, 3:5])
        self.ax_l = self.fig.add_subplot(gs[2, 0:2])
        self.ax_t = self.fig.add_subplot(gs[2, 2:4])
        self.ax_i = self.fig.add_subplot(gs[2, 4])
        self.fig.text(0.045, 0.952, title, color=FG, fontsize=17, fontweight="bold")
        self.sub = self.fig.text(0.045, 0.918, "", color=MUTED, fontsize=10.5)
        self.clock = self.fig.text(0.985, 0.95, "", color=ACCENT, fontsize=20, ha="right", fontweight="bold",
                                   family="monospace")
        ch = sc.cham if les.wind else sc.cham0
        self.ch = ch

        # ---- vista lateral: emissão luminosa integrada na linha de visada ("câmera sintética")
        ax = self.ax_s
        _style(ax, "Vista lateral: emissão luminosa integrada na linha de visada  ·  envelope Z̃ = Z_st")
        img0 = self._side_view(les.fields()[1])
        self.lum_norm = mcolors.PowerNorm(0.45, vmin=0.0, vmax=1.0)
        self.mesh = ax.pcolormesh(les.xf, les.zf, img0.T, cmap="inferno", norm=self.lum_norm, shading="flat",
                                  rasterized=True)
        cb = self.fig.colorbar(self.mesh, ax=ax, pad=0.01, fraction=0.035)
        cb.set_label("luminosidade relativa", color=MUTED); cb.ax.tick_params(colors=MUTED, labelsize=8)
        cb.outline.set_edgecolor(GRID)
        self.lum_cb = cb
        ax.add_patch(Rectangle((-0.6, 0), 1.2, les.tip[2], color="#5c6773", zorder=3))
        fr = frustum_outline_xz(ch, les.tip)
        ax.plot(fr[:, 0], fr[:, 1], ls="--", color=CHAM_C, lw=1.6, zorder=4, label="Chamberlain (1987)")
        ax.plot([], [], color="#4dd0e1", lw=1.4, label="LES: envelope da chama")
        self.flame_c = None
        self.vmax_hist = []
        ax.set_xlim(*view_x); ax.set_ylim(*view_z); ax.set_aspect("equal")
        ax.set_xlabel("x [m] (vento →)"); ax.set_ylabel("z [m]")
        ax.legend(loc="upper right", fontsize=9, facecolor=PANEL, edgecolor=GRID, labelcolor=FG)
        if les.wind:
            ax.annotate("", xy=(view_x[0] + 12, view_z[1] - 6), xytext=(view_x[0] + 2, view_z[1] - 6),
                        arrowprops=dict(arrowstyle="-|>", color=CHAM_C, lw=2))
            ax.text(view_x[0] + 2, view_z[1] - 11, f"{les.cfg.u_ref:.1f} m/s", color=CHAM_C, fontsize=9)

        # ---- mapa de radiação no solo
        ax = self.ax_m
        _style(ax, "Radiação incidente no solo (1,5 m) [kW/m²]")
        q0 = les.q_grid(les.q_inst) / 1e3
        self.norm = mcolors.PowerNorm(0.6, vmin=0.0, vmax=10.0)
        self.qmesh = ax.pcolormesh(les.rec_x, les.rec_y, q0.T, cmap="magma", norm=self.norm, shading="nearest",
                                   rasterized=True)
        cb = self.fig.colorbar(self.qmesh, ax=ax, pad=0.01, fraction=0.04)
        cb.ax.tick_params(colors=MUTED, labelsize=8); cb.outline.set_edgecolor(GRID)
        for lev, c in API_LEVELS:
            cb.ax.axhline(lev, color=c, lw=2)
        self.q_cont = None
        ax.plot([0], [0], marker="^", color="white", ms=9, mec="black", zorder=5)
        ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
        ax.set_xlim(les.rec_x[0], les.rec_x[-1]); ax.set_ylim(les.rec_y[0], les.rec_y[-1])
        self.map_note = ax.text(0.02, 0.03, "", transform=ax.transAxes, color=FG, fontsize=9)
        handles = [Line2D([], [], color=c, lw=2, label=f"{lev:.2f}") for lev, c in API_LEVELS]
        ax.legend(handles=handles, title="API 521", loc="upper right", fontsize=8, title_fontsize=8,
                  facecolor=PANEL, edgecolor=GRID, labelcolor=FG).get_title().set_color(FG)

        # ---- fluxo ao longo do vento (y = 0) vs modelos semi-empíricos
        ax = self.ax_l
        _style(ax, "Fluxo no solo ao longo do vento (y = 0)")
        jl = int(np.argmin(np.abs(les.rec_y)))
        self.jl = jl
        xr = les.rec_x
        rec = np.stack([xr, np.zeros_like(xr), np.full_like(xr, les.cfg.receiver_z)], 1)
        self.q_cham_line = se.q_chamberlain(rec, ch, les.tip, sc.T_inf, sc.RH) / 1e3
        center = se.flame_center_chamberlain(ch, les.tip)
        self.q_pt_line = se.q_point_source(rec, center, les.cfg.X_rad, sc.Q, sc.T_inf, sc.RH) / 1e3
        ax.plot(xr, self.q_cham_line, "--", color=CHAM_C, lw=1.8, label="Chamberlain (frustum)")
        ax.plot(xr, self.q_pt_line, ":", color=PT_C, lw=1.8, label="Fonte pontual (API 521)")
        self.l_inst, = ax.plot(xr, np.zeros_like(xr), color=LES_C, lw=1.0, alpha=0.45, label="LES instantâneo")
        self.l_mean, = ax.plot(xr, np.zeros_like(xr), color=LES_C, lw=2.6, label="LES média")
        for lev, c in API_LEVELS:
            ax.axhline(lev, color=c, lw=0.8, alpha=0.6)
        ymax = 1.6 * max(self.q_cham_line.max(), self.q_pt_line.max())
        ax.set_ylim(0, ymax); ax.set_xlim(xr[0], xr[-1])
        ax.set_xlabel("x [m]"); ax.set_ylabel("q'' [kW/m²]")
        ax.grid(color=GRID, lw=0.5)
        ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID, labelcolor=FG, ncol=2, loc="upper right")

        # ---- comprimento da chama no tempo
        ax = self.ax_t
        _style(ax, "Comprimento da chama no tempo")
        refL = ch.L_b
        ax.axhline(refL, color=CHAM_C, ls="--", lw=1.6, label=f"Chamberlain {refL:.1f} m")
        if not les.wind:
            ax.axhline(sc.L_api, color=PT_C, ls=":", lw=1.6, label=f"API 521 {sc.L_api:.1f} m")
        self.t_inst, = ax.plot([], [], color=LES_C, lw=1.0, alpha=0.5, label="LES instantâneo")
        self.t_mean, = ax.plot([], [], color=LES_C, lw=2.6, label="LES média (I = 0,5)")
        ax.set_ylim(0, 1.8 * refL); ax.set_xlabel("t [s]"); ax.set_ylabel("L [m]")
        ax.grid(color=GRID, lw=0.5)
        ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID, labelcolor=FG, loc="upper right")
        self.mean_hist_t, self.mean_hist_L = [], []

        # ---- painel de informações
        ax = self.ax_i
        ax.set_facecolor(PANEL); ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_color(GRID)
        self.info = ax.text(0.04, 0.96, "", transform=ax.transAxes, va="top", color=FG, fontsize=8.4,
                            family="monospace", linespacing=1.45)
        self.writer = None
        self.handle = None

    def _side_view(self, e):
        """Emissão luminosa integrada em y (vista lateral, sem reabsorção), em unidades de X_rad·Q por
        área projetada; o painel mostra o valor relativo ao percentil 99,7 recente."""
        les = self.les
        tot = (e * les.vol).sum().clamp(min=1e-30)
        scale = les.cfg.X_rad * les.Q / tot
        col = (e * les.Dc[1]).sum(1) * scale / 1e3
        return col.float().cpu().numpy()

    # ------------------------------------------------------------------ vídeo
    def start_video(self, path: str, fps: int = 24):
        self.writer = FFMpegWriter(fps=fps, bitrate=9000, codec="libx264",
                                   extra_args=["-pix_fmt", "yuv420p", "-preset", "medium"])
        self.writer.setup(self.fig, path, dpi=self.fig.dpi)
        self.fps = fps

    def finish_video(self, hold_s: float = 2.5):
        if self.writer is not None:
            for _ in range(int(hold_s * self.fps)):
                self.writer.grab_frame(facecolor=BG)
            self.writer.finish()
            self.writer = None

    # --------------------------------------------------------------- quadro
    def update(self, T, e, mask, q, averaging: bool, sub: str = ""):
        les, ch = self.les, self.ch
        img = self._side_view(e)
        self.vmax_hist.append(float(np.percentile(img, 99.7)))
        vmax = max(np.median(self.vmax_hist[-30:]), 1e-6)
        self.mesh.set_array(np.clip(img / vmax, 0.0, 1.0).T.ravel())
        if self.flame_c is not None:
            self.flame_c.remove()
        Zp = les.Z.max(1).values.float().cpu().numpy()
        self.flame_c = self.ax_s.contour(les.xc, les.zc, Zp.T, levels=[les.Z_st], colors=["#4dd0e1"],
                                         linewidths=1.4)
        qg = les.q_grid(q) / 1e3
        qm = les.q_grid(les.avg_q / les.avg_n) / 1e3 if les.avg_n > 0 else None
        show = qm if (averaging and qm is not None) else qg
        self.qmesh.set_array(show.T.ravel())
        if self.q_cont is not None:
            self.q_cont.remove()
        levels = [lev for lev, _ in API_LEVELS if lev < show.max()]
        cols = [c for lev, c in API_LEVELS if lev < show.max()]
        self.q_cont = self.ax_m.contour(les.rec_x, les.rec_y, show.T, levels=levels, colors=cols, linewidths=1.8) \
            if levels else None
        self.map_note.set_text("média temporal" if (averaging and qm is not None) else "instantâneo")
        self.l_inst.set_ydata(qg[:, self.jl])
        if qm is not None:
            self.l_mean.set_ydata(qm[:, self.jl])
        h = les.history
        self.t_inst.set_data(h["t"], h["L"])
        if averaging and les.avg_n > 0:
            self.mean_hist_t.append(les.time)
            self.mean_hist_L.append(les.mean_flame_length())
            self.t_mean.set_data(self.mean_hist_t, self.mean_hist_L)
        self.ax_t.set_xlim(0, max(5.0, les.time * 1.05))
        Lm = les.mean_flame_length()
        tm = les.mean_tilt()
        qmax_m = float(qm.max()) if qm is not None else float("nan")
        fmt = lambda v, f="{:.1f}": "  —  " if not math.isfinite(v) else f.format(v)  # noqa: E731
        lines = [
            f"t     {les.time:6.2f} s",
            f"passo {les.step_n:6d}",
            f"Δt    {les.dt * 1e3:6.1f} ms  CFL {les._last_cfl:.2f}",
            f"wall  {les.wall:6.0f} s",
            "",
            "VALIDAÇÃO     LES   Cham.",
            f"L [m]      {fmt(Lm):>6} {ch.L_b:6.1f}",
            f"incl. [°]  {fmt(tm):>6} {ch.alpha:6.1f}",
            f"q máx      {fmt(qmax_m, '{:.2f}'):>6} {self.q_cham_line.max():6.2f}",
            "(q em kW/m², médias)",
            "",
            f"X_rad = {les.cfg.X_rad:.3f} (F_s)",
        ]
        self.info.set_text("\n".join(lines))
        self.clock.set_text(f"t = {les.time:5.1f} s")
        self.sub.set_text(sub)
        if self.writer is not None:
            self.writer.grab_frame(facecolor=BG)
        else:
            self.fig.canvas.draw()
        if self.live:
            self._show()

    def _show(self):
        try:
            from IPython.display import Image, display
        except ImportError:
            return
        buf = io.BytesIO()
        from PIL import Image as PILImage
        rgba = np.asarray(self.fig.canvas.buffer_rgba())
        PILImage.fromarray(rgba[..., :3]).save(buf, format="JPEG", quality=82)
        img = Image(data=buf.getvalue(), format="jpeg")
        if self.handle is None:
            self.handle = display(img, display_id=True)
        else:
            self.handle.update(img)

    def save_png(self, path: str):
        self.fig.savefig(path, dpi=self.fig.dpi, facecolor=BG)


def run_live(les, sc, title: str, t_end: float, t_avg: float, frame_dt: float = 0.1,
             video_path: str | None = "flare_les.mp4", live: bool = True, fps: int = 24,
             max_wall_s: float | None = None, log_every: int = 0):
    """Roda a LES até t_end, atualizando o painel a cada frame_dt (tempo físico) e gravando o vídeo."""
    dash = LiveDashboard(les, sc, title, live=live)
    if video_path:
        dash.start_video(video_path, fps=fps)
    sub = (f"{les.summary()}  ·  Smagorinsky + Z̃/β-PDF (equilíbrio) + fuligem  ·  "
           f"média a partir de t = {t_avg:.0f} s")
    next_frame = 0.0
    t_start = time.time()
    try:
        while les.time < t_end:
            les.step()
            if les.time >= next_frame:
                averaging = les.time >= t_avg
                T, e, mask, q = les.diagnostics(averaging)
                dash.update(T, e, mask, q, averaging, sub)
                next_frame += frame_dt
                if log_every and len(les.history["t"]) % log_every == 0:
                    print(f"t = {les.time:6.2f} s · L = {les.history['L'][-1]:5.1f} m · "
                          f"{les.wall / les.step_n * 1e3:5.0f} ms/passo", flush=True)
            if max_wall_s and time.time() - t_start > max_wall_s:
                print(f"Tempo de parede máximo atingido em t = {les.time:.1f} s.")
                break
    finally:
        dash.finish_video()
    return dash

In [ ]:
%%writefile flarekit/safety.py
"""Critérios de segurança (seção 5 do documento): zonas do API 521, dose, probits, fuga e
temperatura de superfícies expostas."""
from __future__ import annotations

import math

import numpy as np
from scipy.optimize import brentq
from scipy.special import erf

from .props import SIGMA

API_521 = [
    (9.46, "máximo com acesso de pessoal (poucos segundos, só fuga)"),
    (6.31, "ações de emergência de até ~30 s, sem abrigo"),
    (4.73, "ações de emergência de 2–3 min, sem abrigo"),
    (1.58, "exposição contínua com roupa adequada"),
]

PROBITS = {
    "queimadura 1º grau (Tsao & Perry)": (-39.83, 3.0186),
    "queimadura 2º grau (Tsao & Perry)": (-43.14, 3.0186),
    "fatalidade (TNO Green Book)": (-36.38, 2.56),
    "fatalidade (Eisenberg)": (-38.48, 2.56),
}


def probability(Y):
    return 0.5 * (1.0 + erf((np.asarray(Y) - 5.0) / math.sqrt(2.0)))


def dose_escape(q0_W: float, r0: float, t_react: float, u_esc: float, q_safe_W: float = 1580.0) -> dict:
    """Dose com reação e fuga radial (dedução da seção 5.2):
    D = q0^(4/3) {t_r + (3/5)(r0/u)[1 − (q_s/q0)^(5/6)]}."""
    frac = 0.0 if q0_W <= q_safe_W else 1.0 - (q_safe_W / q0_W) ** (5.0 / 6.0)
    t_eff = t_react + 0.6 * r0 / u_esc * frac
    D = t_eff * q0_W ** (4.0 / 3.0)
    out = {"t_eff_s": t_eff, "dose_TDU": D / 1e4}
    for name, (a, b) in PROBITS.items():
        out[name] = float(probability(a + b * math.log(D)))
    return out


def steel_temperature(q_inc_W: float, T_inf: float = 298.15, alpha: float = 0.9, eps: float = 0.9,
                      h: float = 15.0) -> float:
    """Placa fina isolada atrás, regime permanente: α q = h(T−T∞) + εσ(T⁴−T∞⁴). Retorna °C."""
    f = lambda T: alpha * q_inc_W - h * (T - T_inf) - eps * SIGMA * (T ** 4 - T_inf ** 4)  # noqa: E731
    return brentq(f, T_inf, 3000.0) - 273.15


def zone_extents(q_kW: np.ndarray, rec_x: np.ndarray, rec_y: np.ndarray, stack=(0.0, 0.0)) -> list[dict]:
    """Para cada nível do API 521: área no solo e distância máxima a partir do pé do flare
    ("truncado" = a zona chega à borda da grade de receptores; os valores são limites inferiores)."""
    dx = float(rec_x[1] - rec_x[0]) if len(rec_x) > 1 else 1.0
    dy = float(rec_y[1] - rec_y[0]) if len(rec_y) > 1 else 1.0
    RX, RY = np.meshgrid(rec_x, rec_y, indexing="ij")
    R = np.hypot(RX - stack[0], RY - stack[1])
    out = []
    for lev, desc in API_521:
        m = q_kW >= lev
        # zona encostando na borda dos receptores: área e alcance são só limites inferiores
        trunc = bool(m.any() and (m[0].any() or m[-1].any() or m[:, 0].any() or m[:, -1].any()))
        out.append({"nivel_kW_m2": lev, "descricao": desc, "area_m2": float(m.sum() * dx * dy), "truncado": trunc,
                    "raio_max_m": float(R[m].max()) if m.any() else 0.0,
                    "x_min_m": float(RX[m].min()) if m.any() else float("nan"),
                    "x_max_m": float(RX[m].max()) if m.any() else float("nan")})
    return out

In [ ]:
%%writefile flarekit/report.py
"""Resultados finais: tabela de validação, figura-resumo (16:9) e composição do vídeo final."""
from __future__ import annotations

import math
import shutil
import subprocess

import numpy as np
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec

from . import safety
from . import semiempirical as se
from .dashboard import API_LEVELS, BG, CHAM_C, FG, GRID, LES_C, MUTED, PANEL, PT_C, _style, frustum_outline_xz


def validation_rows(les, sc: se.Scenario, refs: dict) -> list[dict]:
    ch = sc.cham if les.wind else sc.cham0
    Lm, tm = les.mean_flame_length(), les.mean_tilt()
    qm = (les.avg_q / max(les.avg_n, 1)).max().item() / 1e3
    xr = les.rec_x
    rec = np.stack([xr, np.zeros_like(xr), np.full_like(xr, les.cfg.receiver_z)], 1)
    q_ch = se.q_chamberlain(rec, ch, les.tip, sc.T_inf, sc.RH).max() / 1e3
    rows = [
        {"grandeza": "Comprimento da chama L [m]", "LES": Lm, "referência": ch.L_b, "fonte": "Chamberlain (1987)"},
        {"grandeza": "Inclinação da chama [°]", "LES": tm, "referência": ch.alpha, "fonte": "Chamberlain (1987)"},
        {"grandeza": "Fluxo máximo no solo [kW/m²]", "LES": qm, "referência": q_ch, "fonte": "Chamberlain (frustum)"},
    ]
    if not les.wind:
        rows += [
            {"grandeza": "L vs API 521 [m]", "LES": Lm, "referência": sc.L_api, "fonte": "API 521 (Beychok)"},
            {"grandeza": "L vs Delichatsios [m]", "LES": Lm, "referência": refs["delichatsios"]["L"],
             "fonte": "Delichatsios (1993)"},
            {"grandeza": "L vs Heskestad [m]", "LES": Lm, "referência": refs["L_heskestad"], "fonte": "Heskestad"},
        ]
    for r in rows:
        r["erro_%"] = 100.0 * (r["LES"] - r["referência"]) / r["referência"] if r["referência"] else float("nan")
    return rows


def print_validation(rows):
    print(f"{'Grandeza':34s} {'LES':>8s} {'Ref.':>8s} {'Erro':>8s}  Fonte")
    for r in rows:
        print(f"{r['grandeza']:34s} {r['LES']:8.2f} {r['referência']:8.2f} {r['erro_%']:7.1f}%  {r['fonte']}")


def summary_figure(les, sc: se.Scenario, refs: dict, path: str, title: str, dpi: int = 120):
    ch = sc.cham if les.wind else sc.cham0
    fig = Figure(figsize=(16, 9), dpi=dpi, facecolor=BG)
    FigureCanvasAgg(fig)
    gs = GridSpec(2, 3, figure=fig, left=0.085, right=0.985, top=0.86, bottom=0.06, hspace=0.34, wspace=0.30,
                  height_ratios=[1.2, 1.0])
    fig.text(0.045, 0.952, title, color=FG, fontsize=17, fontweight="bold")
    fig.text(0.045, 0.918, f"Médias de {les.avg_n:.0f} quadros (t ≥ início da média) · {les.summary()}",
             color=MUTED, fontsize=10.5)

    # temperatura média + chama média (intermitência 0,5) + Chamberlain
    ax = fig.add_subplot(gs[0, 0])
    _style(ax, "Temperatura média (máx. em y) e chama média (I = 0,5)")
    Tm = (les.avg_T / les.avg_n).max(1).values.float().cpu().numpy()
    Im = (les.avg_I / les.avg_n).max(1).values.float().cpu().numpy()
    m = ax.pcolormesh(les.xf, les.zf, Tm.T, cmap="inferno", vmin=290, vmax=1500, shading="flat", rasterized=True)
    ax.contour(les.xc, les.zc, Im.T, levels=[0.5], colors=["#4dd0e1"], linewidths=2)
    fr = frustum_outline_xz(ch, les.tip)
    ax.plot(fr[:, 0], fr[:, 1], "--", color=CHAM_C, lw=1.8, label="Chamberlain (1987)")
    ax.plot([], [], color="#4dd0e1", lw=2, label="LES: chama média (I = 0,5)")
    from matplotlib.patches import Rectangle
    ax.add_patch(Rectangle((-0.6, 0), 1.2, les.tip[2], color="#5c6773"))
    # janela em volta da chama (frustum + região quente da LES), com um pedaço da chaminé
    hot = Tm > 0.5 * (Tm.max() + 300.0)
    xs, zs = list(fr[:, 0]) + [0.0], list(fr[:, 1]) + [float(les.tip[2])]
    if hot.any():
        ii, kk = np.nonzero(hot)
        xs += [les.xc[ii].min(), les.xc[ii].max()]
        zs += [les.zc[kk].max()]
    zt = float(les.tip[2])
    ztop = max(zs) + 8.0
    x0, x1 = min(-15.0, min(xs) - 8.0), max(xs) + 8.0
    zbot = max(0.0, zt - 0.35 * (ztop - zt) - 8.0)
    span = max(x1 - x0, 1.25 * (ztop - zbot))
    x1 = x0 + span
    zbot = max(0.0, min(zbot, ztop - span / 1.25))
    ax.set_xlim(x0, x1); ax.set_ylim(zbot, ztop); ax.set_aspect("equal")
    ax.set_xlabel("x [m]"); ax.set_ylabel("z [m]")
    ax.legend(fontsize=8.5, facecolor=PANEL, edgecolor=GRID, labelcolor=FG, loc="upper left")
    cb = fig.colorbar(m, ax=ax, fraction=0.04, pad=0.01)
    cb.ax.tick_params(colors=MUTED, labelsize=8); cb.outline.set_edgecolor(GRID)

    # radiação média no solo e zonas do API 521
    ax = fig.add_subplot(gs[0, 1:])
    _style(ax, "Radiação média no solo e zonas do API 521")
    qm = les.q_grid(les.avg_q / les.avg_n) / 1e3
    from matplotlib import colors as mcolors
    from .render import qnorm
    mm = ax.pcolormesh(les.rec_x, les.rec_y, qm.T, cmap="magma", norm=qnorm(max(float(qm.max()), 0.5)),
                       shading="nearest", rasterized=True)
    levels = [lev for lev, _ in API_LEVELS if lev < qm.max()]
    cols = [c for lev, c in API_LEVELS if lev < qm.max()]
    if levels:
        ax.contour(les.rec_x, les.rec_y, qm.T, levels=levels, colors=cols, linewidths=2)
    ax.plot([0], [0], marker="^", color="white", ms=9, mec="black")
    ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    cb = fig.colorbar(mm, ax=ax, fraction=0.03, pad=0.01)
    cb.ax.tick_params(colors=MUTED, labelsize=8); cb.outline.set_edgecolor(GRID)
    zones = safety.zone_extents(qm, les.rec_x, les.rec_y)

    # validação
    ax = fig.add_subplot(gs[1, 0])
    _style(ax, "Validação (LES vs. modelos de referência)")
    rows = validation_rows(les, sc, refs)
    short = {"Comprimento da chama L [m]": "L chama [m]", "Inclinação da chama [°]": "Inclinação [°]",
             "Fluxo máximo no solo [kW/m²]": "q máx [kW/m²]", "L vs API 521 [m]": "L × API [m]",
             "L vs Delichatsios [m]": "L × Delichatsios", "L vs Heskestad [m]": "L × Heskestad"}
    names = [short.get(r["grandeza"], r["grandeza"]) for r in rows]
    # razão LES/referência (grandezas de escalas muito diferentes no mesmo eixo)
    y = np.arange(len(rows))
    ratio = np.array([100.0 * r["LES"] / r["referência"] if r["referência"] else np.nan for r in rows])
    ax.barh(y, ratio, 0.55, color=[LES_C if abs(v - 100) <= 25 else "#f2766b" for v in ratio], zorder=3)
    ax.axvline(100, color=CHAM_C, lw=1.6, ls="--", label="referência = 100%")
    ax.axvspan(75, 125, color=CHAM_C, alpha=0.08)
    xmax = max(160.0, np.nanmax(ratio) * 1.45)
    for i, r in enumerate(rows):
        ax.text(min(ratio[i], xmax) + 0.02 * xmax, i, f"{r['LES']:.3g} vs {r['referência']:.3g} ({r['erro_%']:+.0f}%)",
                va="center", color=FG, fontsize=8.5, zorder=4)
    ax.set_xlim(0, xmax)
    ax.set_xlabel("LES / referência [%]  (faixa sombreada: ±25%)")
    ax.set_yticks(y); ax.set_yticklabels(names, color=FG, fontsize=9)
    ax.set_ylim(len(rows) - 0.4, -0.9)
    ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID, labelcolor=FG, loc="upper right")
    ax.grid(color=GRID, lw=0.5, axis="x")

    # segurança no ponto mais exposto
    def text_box(cell, lines):
        ax = fig.add_subplot(cell)
        ax.set_facecolor(PANEL); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_color(GRID)
        ax.text(0.05, 0.95, "\n".join(lines), transform=ax.transAxes, va="top", color=FG, fontsize=9.5,
                family="monospace", linespacing=1.5)

    i, j = np.unravel_index(np.argmax(qm), qm.shape)
    xq, yq, q0 = les.rec_x[i], les.rec_y[j], qm[i, j]
    center = se.flame_center_chamberlain(ch, les.tip)
    r0 = float(np.linalg.norm(center - np.array([xq, yq, les.cfg.receiver_z])))
    d1 = safety.dose_escape(q0 * 1e3, r0, 5.0, 2.5)
    d2 = safety.dose_escape(q0 * 1e3, r0, 30.0, 2.5)
    text_box(gs[1, 1], [
        "PONTO MAIS EXPOSTO NO SOLO",
        f"x = {xq:.0f} m, y = {yq:.0f} m: q = {q0:.2f} kW/m²",
        f"T aço exposto (regime): {safety.steel_temperature(q0 * 1e3):.0f} °C",
        "",
        "Dose com fuga a 2,5 m/s:",
        f" reage em 5 s : {d1['dose_TDU']:5.0f} TDU",
        f"   1º grau {100 * d1['queimadura 1º grau (Tsao & Perry)']:5.1f}%"
        f"  2º grau {100 * d1['queimadura 2º grau (Tsao & Perry)']:5.2f}%",
        f" reage em 30 s: {d2['dose_TDU']:5.0f} TDU",
        f"   1º grau {100 * d2['queimadura 1º grau (Tsao & Perry)']:5.1f}%"
        f"  2º grau {100 * d2['queimadura 2º grau (Tsao & Perry)']:5.2f}%",
        f"   fatal (TNO) {100 * d2['fatalidade (TNO Green Book)']:5.2f}%",
    ])
    zl = ["ZONAS API 521 (LES, média)", "", "nível       área [m²]  alcance [m]"]
    zl += [f"{z['nivel_kW_m2']:5.2f} kW/m² {'≥' if z.get('truncado') else ' '}{z['area_m2']:8.0f}"
           f"   {'≥' if z.get('truncado') else ' '}{z['raio_max_m']:6.0f}" for z in zones]
    if any(z.get("truncado") for z in zones):
        zl += ["", "≥: a zona passa da grade de", "   receptores (limite inferior)"]
    text_box(gs[1, 2], zl)
    fig.savefig(path, dpi=dpi, facecolor=BG)
    return fig, rows, zones


def compose_video(main_mp4: str, summary_png: str, out_mp4: str, seconds: float = 6.0, fps: int = 24) -> str:
    """Anexa a figura-resumo ao fim do vídeo da simulação (ffmpeg)."""
    ff = shutil.which("ffmpeg")
    if ff is None:
        raise RuntimeError("ffmpeg não encontrado")
    cmd = [ff, "-y", "-i", main_mp4, "-loop", "1", "-t", f"{seconds}", "-framerate", f"{fps}", "-i", summary_png,
           "-filter_complex",
           "[0:v]scale=1920:1080,setsar=1,fps=%d[a];[1:v]scale=1920:1080,setsar=1,fps=%d,format=yuv420p[b];"
           "[a][b]concat=n=2:v=1:a=0[v]" % (fps, fps),
           "-map", "[v]", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "17", "-preset", "slow", out_mp4]
    subprocess.run(cmd, check=True, capture_output=True)
    return out_mp4

In [ ]:
%%writefile flarekit/wind.py
"""Clima de vento do local: dados de atlas/reanálise → perfil de camada limite → rosa dos ventos →
pegadas de radiação por setor e mapas de probabilidade de excedência.

Convenções
----------
* Direção meteorológica: de onde o vento VEM, em graus a partir do Norte, sentido horário
  (0 = N, 90 = E). O vento sopra para θ + 180°.
* Coordenadas do local: E (leste) e N (norte), origem no pé do flare.
* Coordenadas da LES/Chamberlain: x a jusante (para onde o vento sopra), y à esquerda de x.
  Com d = (−sen θ, −cos θ) o versor a jusante em (E, N):
      E = x·d_E − y·d_N,   N = x·d_N + y·d_E      (rotação ortonormal)
      x = E·d_E + N·d_N,   y = −E·d_N + N·d_E
"""
from __future__ import annotations

import json
import math
from dataclasses import dataclass, field
from urllib.request import Request, urlopen

import numpy as np
from scipy.ndimage import map_coordinates

from . import semiempirical as se

KAPPA = 0.41
SECTORS = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSO", "SO", "OSO", "O", "ONO", "NO", "NNO"]
SPEED_EDGES = np.array([0.5, 2.0, 4.0, 6.0, 8.0, 10.0, 13.0, np.inf])   # m/s na altura do tip; < 0.5 = calmaria
API_LEVELS = (1.58, 4.73, 6.31, 9.46)


def sector_name(theta_deg: float) -> str:
    return SECTORS[int(((theta_deg % 360) + 11.25) // 22.5) % 16]


# ===================================================================== séries
@dataclass
class WindSeries:
    """Série horária de vento em duas alturas (velocidade em m/s, direção meteorológica em graus)."""
    source: str
    lat: float
    lon: float
    z1: float
    z2: float
    U1: np.ndarray
    U2: np.ndarray
    D1: np.ndarray
    D2: np.ndarray
    period: str = ""
    synthetic: bool = False

    def clean(self) -> "WindSeries":
        ok = (np.isfinite(self.U1) & np.isfinite(self.U2) & np.isfinite(self.D1) & np.isfinite(self.D2)
              & (self.U1 >= 0) & (self.U2 >= 0) & (self.U1 < 80) & (self.U2 < 80))
        return WindSeries(self.source, self.lat, self.lon, self.z1, self.z2, self.U1[ok], self.U2[ok],
                          self.D1[ok] % 360, self.D2[ok] % 360, self.period, self.synthetic)


def _get_text(url: str, timeout: float = 90.0, headers: dict | None = None) -> str:
    h = {"User-Agent": "flarekit/0.3 (pesquisa; Google Colab)"}
    h.update(headers or {})
    with urlopen(Request(url, headers=h), timeout=timeout) as r:
        return r.read().decode("utf-8", errors="replace")


def _get_json(url: str, timeout: float = 90.0) -> dict:
    return json.loads(_get_text(url, timeout))


def direction_from_uv(u, v):
    """Direção meteorológica (de onde vem) a partir das componentes leste (u) e norte (v)."""
    return np.degrees(np.arctan2(-np.asarray(u), -np.asarray(v))) % 360


# ------------------------------------------------------------- Global Wind Atlas
def parse_gwa_lib(text: str) -> dict:
    """Clima de vento generalizado do Global Wind Atlas (formato WAsP .lib):
    cabeçalho; 'n_rugosidades n_alturas n_setores'; rugosidades z0 [m]; alturas [m]; e, para cada
    rugosidade, frequências por setor [%] seguidas, para cada altura, de A [m/s] e k por setor.
    O setor 0 é centrado no Norte."""
    lines = text.strip().splitlines()
    header = lines[0]
    nums = []
    for ln in lines[1:]:
        for tok in ln.replace(",", " ").split():
            try:
                nums.append(float(tok))
            except ValueError:
                pass
    nr, nh, ns = (int(round(x)) for x in nums[:3])
    i = 3
    z0s = np.array(nums[i:i + nr]); i += nr
    hs = np.array(nums[i:i + nh]); i += nh
    freq = np.zeros((nr, ns)); A = np.zeros((nr, nh, ns)); k = np.zeros((nr, nh, ns))
    for r in range(nr):
        freq[r] = nums[i:i + ns]; i += ns
        for h in range(nh):
            A[r, h] = nums[i:i + ns]; i += ns
            k[r, h] = nums[i:i + ns]; i += ns
    lat = lon = None
    if "<coordinates>" in header:
        try:
            c = header.split("<coordinates>")[1].split("</coordinates>")[0].split(",")
            lon, lat = float(c[0]), float(c[1])
        except (IndexError, ValueError):
            pass
    return {"header": header, "z0": z0s, "heights": hs, "freq": freq / 100.0, "A": A, "k": k,
            "n_sectors": ns, "lat": lat, "lon": lon}


def fetch_gwa(lat: float, lon: float) -> dict:
    """Ponto do Global Wind Atlas (endpoint não documentado usado por clientes abertos, ex. wind-stats)."""
    url = f"https://globalwindatlas.info/api/gwa/custom/Lib/?lat={lat:.4f}&long={lon:.4f}"
    return parse_gwa_lib(_get_text(url, headers={"Referer": "https://globalwindatlas.info"}))


def gwa_series(gwc: dict, lat: float, lon: float, z0_site: float = 0.3, n: int = 60000,
               z1: float = 10.0, z2: float = 50.0, seed: int = 7) -> WindSeries:
    """Amostra horas equivalentes do clima generalizado do GWA para a classe de rugosidade mais
    próxima de z0_site: setor pela frequência, direção uniforme no setor e velocidades nas duas
    alturas com o mesmo quantil da Weibull do setor (perfil coerente)."""
    rng = np.random.default_rng(seed)
    z0s = np.maximum(gwc["z0"], 1e-4)
    r = int(np.argmin(np.abs(np.log(z0s) - math.log(max(z0_site, 1e-4)))))
    f = gwc["freq"][r] / gwc["freq"][r].sum()
    ns = gwc["n_sectors"]
    width = 360.0 / ns
    hs = gwc["heights"]

    def weib_at(z, sec):
        lh = np.log(hs)
        A = np.array([np.interp(math.log(z), lh, gwc["A"][r, :, j]) for j in range(ns)])
        k = np.array([np.interp(math.log(z), lh, gwc["k"][r, :, j]) for j in range(ns)])
        return A[sec], k[sec]
    sec = rng.choice(ns, size=n, p=f)
    D = (sec * width + rng.uniform(-width / 2, width / 2, n)) % 360
    u = rng.uniform(1e-6, 1 - 1e-6, n)
    A1, k1 = weib_at(z1, sec); A2, k2 = weib_at(z2, sec)
    U1 = A1 * (-np.log(1 - u)) ** (1 / k1)
    U2 = A2 * (-np.log(1 - u)) ** (1 / k2)
    src = f"Global Wind Atlas (clima generalizado, z0 = {gwc['z0'][r]:g} m)"
    return WindSeries(src, lat, lon, z1, z2, U1, U2, D, D, f"{n} horas equivalentes", synthetic=False)


# ------------------------------------------------------------- NASA POWER
NASA_PARAMS = "WS10M,WD10M,WS50M,WD50M,U10M,V10M,U50M,V50M"


def parse_nasa_power(js: dict, lat: float, lon: float, period: str = "") -> WindSeries:
    """JSON horário da NASA POWER: properties.parameter.{PARAM} → {AAAAMMDDHH: valor}, fill_value −999.
    Se as componentes U/V vierem, a direção é recalculada delas (convenção meteorológica garantida)."""
    par = js["properties"]["parameter"]
    fill = js.get("header", {}).get("fill_value", -999.0)
    keys = sorted(par["WS10M"].keys())

    def arr(name):
        a = np.array([par[name].get(k, fill) for k in keys], dtype=float)
        a[(a == fill) | (a <= -998)] = np.nan
        return a
    D1, D2 = arr("WD10M"), arr("WD50M")
    if all(k in par for k in ("U10M", "V10M", "U50M", "V50M")):
        D1 = direction_from_uv(arr("U10M"), arr("V10M"))
        D2 = direction_from_uv(arr("U50M"), arr("V50M"))
    return WindSeries("NASA POWER (MERRA-2), horário", lat, lon, 10.0, 50.0, arr("WS10M"), arr("WS50M"),
                      D1, D2, period).clean()


def fetch_nasa_power(lat: float, lon: float, start: str = "20140101", end: str = "20231231") -> WindSeries:
    """Vento horário a 10 m e 50 m (NASA POWER, sem chave), um ano por pedido. start/end em AAAAMMDD."""
    y0, y1 = int(start[:4]), int(end[:4])
    parts = []
    for y in range(y0, y1 + 1):
        s = start if y == y0 else f"{y}0101"
        e = end if y == y1 else f"{y}1231"
        url = (f"https://power.larc.nasa.gov/api/temporal/hourly/point?parameters={NASA_PARAMS}"
               f"&community=RE&longitude={lon:.4f}&latitude={lat:.4f}&start={s}&end={e}"
               "&format=JSON&time-standard=UTC")
        parts.append(parse_nasa_power(_get_json(url, timeout=300.0), lat, lon))
    cat = lambda k: np.concatenate([getattr(p, k) for p in parts])  # noqa: E731
    return WindSeries("NASA POWER (MERRA-2), horário", lat, lon, 10.0, 50.0, cat("U1"), cat("U2"), cat("D1"),
                      cat("D2"), f"{start[:4]}–{end[:4]}").clean()


# ------------------------------------------------------------- Open-Meteo (ERA5)
def parse_open_meteo(js: dict, lat: float, lon: float, period: str = "") -> WindSeries:
    """JSON do arquivo histórico Open-Meteo: hourly.{wind_speed_10m, wind_speed_100m, wind_direction_10m,
    wind_direction_100m} (pedido com wind_speed_unit=ms); nulos viram NaN."""
    h = js["hourly"]
    g = lambda k: np.array([np.nan if v is None else v for v in h[k]], dtype=float)  # noqa: E731
    return WindSeries("Open-Meteo (ERA5), horário", lat, lon, 10.0, 100.0, g("wind_speed_10m"),
                      g("wind_speed_100m"), g("wind_direction_10m"), g("wind_direction_100m"), period).clean()


def fetch_open_meteo(lat: float, lon: float, start: str = "2014-01-01", end: str = "2023-12-31") -> WindSeries:
    """ERA5 horário a 10 m e 100 m. Uso não comercial gratuito; uso comercial exige chave."""
    url = ("https://archive-api.open-meteo.com/v1/archive?"
           f"latitude={lat:.4f}&longitude={lon:.4f}&start_date={start}&end_date={end}"
           "&hourly=wind_speed_10m,wind_speed_100m,wind_direction_10m,wind_direction_100m"
           "&wind_speed_unit=ms&timezone=GMT&models=era5")
    return parse_open_meteo(_get_json(url, timeout=300.0), lat, lon, f"{start[:4]}–{end[:4]}")


def synthetic_series(lat: float, lon: float, n: int = 8760, seed: int = 1) -> WindSeries:
    """Série SINTÉTICA de exemplo (só para rodar sem internet): rosa com predominância de SE, como a
    descrita para Campinas na literatura, Weibull k = 2, z0 = 0,3 m. Não é dado medido de nenhum local."""
    rng = np.random.default_rng(seed)
    freq = np.array([2, 2, 3, 6, 10, 14, 20, 11, 6, 4, 3, 3, 4, 4, 4, 4], float)
    freq /= freq.sum()
    sec = rng.choice(16, size=n, p=freq)
    D2 = (sec * 22.5 + rng.uniform(-11.25, 11.25, n)) % 360
    U2 = 5.0 * rng.weibull(2.0, n)
    z0 = 0.3
    U1 = U2 * math.log(10 / z0) / math.log(50 / z0)
    D1 = (D2 + rng.normal(0, 8, n)) % 360
    return WindSeries("rosa SINTÉTICA de exemplo (sem internet)", lat, lon, 10.0, 50.0, U1, U2, D1, D2,
                      "1 ano sintético", synthetic=True)


def get_wind_series(lat: float, lon: float, source: str = "auto", z0_site: float = 0.3,
                    start: str = "20140101", end: str = "20231231", verbose: bool = True,
                    H: float | None = None) -> WindSeries:
    """source: 'gwa' (Global Wind Atlas), 'nasa' (NASA POWER), 'open-meteo' (ERA5), 'sintetico' ou
    'auto' (GWA → NASA → Open-Meteo → sintético). H (altura do tip): no GWA a segunda altura amostrada
    passa a ser H (até 200 m), de modo que o vento no tip vem do próprio atlas, sem extrapolar."""
    order = {"auto": ["gwa", "nasa", "open-meteo", "sintetico"], "gwa": ["gwa"], "nasa": ["nasa"],
             "open-meteo": ["open-meteo"], "sintetico": ["sintetico"]}[source]
    errors = []
    for s in order:
        try:
            if s == "gwa":
                gwc = fetch_gwa(lat, lon)
                z2 = float(np.clip(H, 50.0, max(gwc["heights"]))) if H else 50.0
                return gwa_series(gwc, lat, lon, z0_site, z2=z2)
            if s == "nasa":
                return fetch_nasa_power(lat, lon, start, end)
            if s == "open-meteo":
                return fetch_open_meteo(lat, lon, f"{start[:4]}-{start[4:6]}-{start[6:]}",
                                        f"{end[:4]}-{end[4:6]}-{end[6:]}")
            return synthetic_series(lat, lon)
        except Exception as e:  # noqa: BLE001 — tenta a próxima fonte
            errors.append(f"{s}: {type(e).__name__}: {str(e)[:120]}")
            if verbose:
                print(f"[vento] {s} indisponível ({type(e).__name__}); tentando a próxima fonte")
    raise RuntimeError("Nenhuma fonte de vento disponível: " + " | ".join(errors))


# ===================================================================== perfil
def fit_log_profile(U1, U2, z1: float, z2: float, umin: float = 1.0) -> dict:
    """Ajusta U(z) = (u*/κ) ln(z/z0) às médias nas duas alturas (horas com U > umin em ambas).
    Também devolve o expoente da lei de potência α = ln(U2/U1)/ln(z2/z1)."""
    m = (U1 > umin) & (U2 > umin)
    a, b = float(np.mean(U1[m])), float(np.mean(U2[m]))
    alpha = math.log(b / a) / math.log(z2 / z1)
    if b > a:
        z0 = math.exp((b * math.log(z1) - a * math.log(z2)) / (b - a))
    else:
        z0 = 1e-4
    z0_clamped = float(np.clip(z0, 2e-4, 1.0))
    return {"z0": z0_clamped, "z0_raw": z0, "alpha": alpha, "U1_mean": a, "U2_mean": b,
            "u_star_mean": KAPPA * b / math.log(z2 / z0_clamped)}


def speed_at_height(U1, U2, z1, z2, z, z0):
    """Velocidade horária na altura z pela lei log, ancorada no nível de dados mais próximo."""
    if abs(math.log(z / z1)) <= abs(math.log(z / z2)):
        return U1 * math.log(z / z0) / math.log(z1 / z0)
    return U2 * math.log(z / z0) / math.log(z2 / z0)


def direction_at_height(D1, D2, z1, z2, z):
    """Interpolação circular da direção em ln z entre as duas alturas."""
    w = float(np.clip(math.log(z / z1) / math.log(z2 / z1), 0.0, 1.0))
    a1, a2 = np.radians(D1), np.radians(D2)
    s = (1 - w) * np.sin(a1) + w * np.sin(a2)
    c = (1 - w) * np.cos(a1) + w * np.cos(a2)
    return np.degrees(np.arctan2(s, c)) % 360


def weibull_fit(U):
    """Weibull por máxima verossimilhança (scipy), com localização fixa em 0."""
    from scipy.stats import weibull_min
    U = U[U > 0.1]
    k, _, c = weibull_min.fit(U, floc=0)
    return float(k), float(c)


@dataclass
class WindClimate:
    series: WindSeries
    H: float
    z0: float
    alpha: float
    U_H: np.ndarray
    D_H: np.ndarray
    freq: np.ndarray            # (16, nbins) frequências conjuntas (somam 1 − calmaria)
    calm: float
    bin_speed: np.ndarray       # velocidade representativa de cada classe [m/s]
    weibull_k: float
    weibull_c: float
    profile_fit: dict = field(default_factory=dict)

    @classmethod
    def from_series(cls, s: WindSeries, H: float, edges=SPEED_EDGES) -> "WindClimate":
        fit = fit_log_profile(s.U1, s.U2, s.z1, s.z2)
        z0 = fit["z0"]
        UH = speed_at_height(s.U1, s.U2, s.z1, s.z2, H, z0)
        DH = direction_at_height(s.D1, s.D2, s.z1, s.z2, H)
        n = len(UH)
        calm = float(np.mean(UH < edges[0]))
        sec = (((DH % 360) + 11.25) // 22.5).astype(int) % 16
        nb = len(edges) - 1
        freq = np.zeros((16, nb))
        bspeed = np.zeros(nb)
        for b in range(nb):
            m = (UH >= edges[b]) & (UH < edges[b + 1])
            bspeed[b] = float(np.mean(UH[m])) if m.any() else (edges[b] + min(edges[b + 1], edges[b] + 3)) / 2
            for k in range(16):
                freq[k, b] = np.count_nonzero(m & (sec == k)) / n
        k, c = weibull_fit(UH)
        return cls(s, H, z0, fit["alpha"], UH, DH, freq, calm, bspeed, k, c, fit)

    # ------------------------------------------------------------ resumos
    @property
    def sector_freq(self):
        return self.freq.sum(1)

    @property
    def dominant_sector(self) -> int:
        return int(np.argmax(self.sector_freq))

    def sector_speeds(self, k: int):
        sec = (((self.D_H % 360) + 11.25) // 22.5).astype(int) % 16
        return self.U_H[(sec == k) & (self.U_H >= SPEED_EDGES[0])]

    def design_wind(self, mode: str = "dominante_p90", design_speed: float = 8.9):
        """Vento para a LES: (velocidade na altura do tip, direção de onde vem)."""
        k = self.dominant_sector
        theta = 22.5 * k
        u = self.sector_speeds(k)
        if mode == "dominante_media":
            return float(np.mean(u)), theta
        if mode == "dominante_p90":
            return float(np.percentile(u, 90)), theta
        if mode == "projeto":
            return design_speed, theta
        raise ValueError(mode)

    def profile(self, z, U_H: float | None = None):
        """Perfil log com o z0 ajustado, escalado para U_H na altura do tip."""
        U_H = float(np.mean(self.U_H)) if U_H is None else U_H
        return U_H * np.log(np.asarray(z) / self.z0) / math.log(self.H / self.z0)

    def describe(self) -> str:
        s = self.series
        k = self.dominant_sector
        return (f"{s.source} · {s.period} · lat {s.lat:.3f}, lon {s.lon:.3f}\n"
                f"perfil log: z0 = {self.z0:.3f} m (α = {self.alpha:.3f}); médias {s.U1.mean():.2f} m/s a "
                f"{s.z1:.0f} m e {s.U2.mean():.2f} m/s a {s.z2:.0f} m\n"
                f"a {self.H:.0f} m: média {self.U_H.mean():.2f} m/s, P90 {np.percentile(self.U_H, 90):.2f} m/s, "
                f"Weibull k = {self.weibull_k:.2f}, c = {self.weibull_c:.2f} m/s, calmaria {100 * self.calm:.1f}%\n"
                f"setor predominante: {SECTORS[k]} ({100 * self.sector_freq[k]:.1f}% do tempo)")


# ===================================================================== pegadas
@dataclass
class SiteGrid:
    E: np.ndarray
    N: np.ndarray

    @classmethod
    def square(cls, R: float = 120.0, n: int = 121) -> "SiteGrid":
        x = np.linspace(-R, R, n)
        return cls(x, x)


def downwind_footprint(sc: se.Scenario, U: float, R: float, n: int, z_rec: float = 1.5):
    """q [kW/m²] de Chamberlain no referencial a jusante (x, y) numa grade quadrada de meia-largura R."""
    ch = se.chamberlain(sc.fuel, sc.mdot, sc.tip["u_j"], sc.tip["rho_j"], sc.rho_inf, U,
                        fs_factor=getattr(sc, "fs_factor", 1.0))
    x = np.linspace(-R, R, n)
    X, Y = np.meshgrid(x, x, indexing="ij")
    rec = np.stack([X.ravel(), Y.ravel(), np.full(X.size, z_rec)], 1)
    q = se.q_chamberlain(rec, ch, sc.tip_xyz, sc.T_inf, sc.RH).reshape(X.shape) / 1e3
    return x, q, ch


def rotate_to_site(xd: np.ndarray, yd: np.ndarray, q_down: np.ndarray, grid: SiteGrid, theta_from: float,
                   fill: float = 0.0) -> np.ndarray:
    """Amostra uma pegada do referencial a jusante (grade regular xd × yd) nos pontos (E, N) do local
    para vento vindo de theta_from."""
    th = math.radians(theta_from)
    dE, dN = -math.sin(th), -math.cos(th)
    EE, NN = np.meshgrid(grid.E, grid.N, indexing="ij")
    x = EE * dE + NN * dN
    y = -EE * dN + NN * dE
    ix = np.interp(x, xd, np.arange(len(xd)), left=-1, right=len(xd))
    iy = np.interp(y, yd, np.arange(len(yd)), left=-1, right=len(yd))
    out = map_coordinates(q_down, [ix, iy], order=1, mode="constant", cval=fill)
    return out


def exceedance_maps(sc: se.Scenario, cl: WindClimate, grid: SiteGrid, levels=API_LEVELS, sub: int = 3,
                    q_solar: float = 0.0):
    """Probabilidade (dado que o flare queima) de cada ponto exceder cada nível, ponderada pela rosa
    dos ventos, e envoltória (máximo sobre as condições com frequência > 0).

    Cada setor é dividido em `sub` subdireções (evita mapas em raios). Calmaria → chama vertical.
    P usa a velocidade média de cada classe; a envoltória inclui também a maior velocidade observada
    (o fluxo máximo cresce com o vento, e a média da classe mais alta subestimaria o pior caso).
    q_solar [kW/m²] é somado ao fluxo do flare quando o critério é de radiação total (0 = só o flare)."""
    R = max(abs(grid.E).max(), abs(grid.N).max()) * 1.45
    nd = 2 * int(R / 3.0) + 1
    P = {L: np.zeros((len(grid.E), len(grid.N))) for L in levels}
    Psec = {L: np.zeros((16, len(grid.E), len(grid.N))) for L in levels}
    env = np.zeros((len(grid.E), len(grid.N)))
    q_mean = np.zeros((len(grid.E), len(grid.N)))   # fluxo esperado (média ponderada pela rosa)
    foot = {}

    def subdirs(k):
        return [22.5 * k + (j - (sub - 1) / 2) * 22.5 / sub for j in range(sub)]

    for b, U in enumerate(cl.bin_speed):
        if cl.freq[:, b].sum() <= 0:
            continue
        xd, qd, _ = downwind_footprint(sc, U, R, nd)
        qd = qd + q_solar
        foot[b] = (xd, qd)
        for k in range(16):
            f = cl.freq[k, b]
            if f <= 0:
                continue
            for th in subdirs(k):
                q = rotate_to_site(xd, xd, qd, grid, th)
                for L in levels:
                    hit = (f / sub) * (q >= L)
                    P[L] += hit
                    Psec[L][k] += hit
                env = np.maximum(env, q)
                q_mean += (f / sub) * q
    # pior caso: maior velocidade observada, nas direções em que a classe mais alta ocorre
    top = [b for b in range(cl.freq.shape[1]) if cl.freq[:, b].sum() > 0]
    U_max = float(np.max(cl.U_H)) if len(cl.U_H) else 0.0
    if top and U_max > cl.bin_speed[top[-1]] + 0.1:
        xd, qd, _ = downwind_footprint(sc, U_max, R, nd)
        qd = qd + q_solar
        foot["U_max"] = (xd, qd)
        for k in range(16):
            if cl.freq[k, top[-1]] > 0:
                for th in subdirs(k):
                    env = np.maximum(env, rotate_to_site(xd, xd, qd, grid, th))
    if cl.calm > 0:
        xd, qd, _ = downwind_footprint(sc, 0.0, R, nd)
        qd = qd + q_solar
        q = rotate_to_site(xd, xd, qd, grid, 0.0)
        for L in levels:
            P[L] += cl.calm * (q >= L)
        env = np.maximum(env, q)
        q_mean += cl.calm * q
        foot["calm"] = (xd, qd)
    return {"P": P, "P_sector": Psec, "envelope": env, "q_mean": q_mean, "footprints": foot, "grid": grid,
            "q_solar": q_solar, "U_max": U_max}


def receiver_box(sc: se.Scenario, U: float, level: float = 1.58, margin: float = 15.0) -> tuple:
    """(x0, x1, y_meia) que cobre a zona q ≥ level no referencial a jusante, com folga: união da zona
    de Chamberlain com a da fonte pontual do API 521 (mais conservadora a barlavento, onde a chama curta
    da LES irradia mais que o frustum). Use em LESConfig(receiver_box=...) para não cortar a zona."""
    r_pt = math.sqrt(sc.cham.F_s * sc.Q / (4 * math.pi * level * 1e3))   # alcance da fonte pontual (τ = 1)
    R = max(60.0, 3.5 * sc.H + sc.cham.L_b, sc.cham.L_b + 1.2 * r_pt)
    for _ in range(4):
        x, q, ch = downwind_footprint(sc, U, R, 2 * int(R / 3.0) + 1)
        X, Y = np.meshgrid(x, x, indexing="ij")
        c = se.flame_center_chamberlain(ch, sc.tip_xyz)
        rec = np.stack([X.ravel(), Y.ravel(), np.full(X.size, 1.5)], 1)
        qp = se.q_point_source(rec, c, ch.F_s, sc.Q, sc.T_inf, sc.RH).reshape(X.shape) / 1e3
        m = (q >= level) | (qp >= level)
        if not (m[0].any() or m[-1].any() or m[:, 0].any() or m[:, -1].any()):
            break
        R *= 1.5                                     # a zona encostou na borda: amplia e refaz
    if not m.any():
        return (-margin, margin, margin)
    return (float(X[m].min() - margin), float(X[m].max() + margin), float(np.abs(Y[m]).max() + margin))


def les_footprint_downwind(rec_x, rec_y, q_les, sc: se.Scenario, U: float, R: float, n: int,
                           blend: float = 10.0):
    """Pegada média da LES num quadrado regular a jusante; fora do domínio de receptores da LES usa
    Chamberlain para a mesma velocidade (a LES cobre a região de maior fluxo). Uma faixa de `blend` metros
    junto à borda dos receptores mistura linearmente as duas, para não criar degrau nas isolinhas."""
    xd, qd, _ = downwind_footprint(sc, U, R, n)
    X, Y = np.meshgrid(xd, xd, indexing="ij")
    d = np.minimum.reduce([X - rec_x[0], rec_x[-1] - X, Y - rec_y[0], rec_y[-1] - Y])
    b = max(1e-6, min(blend, 0.25 * min(rec_x[-1] - rec_x[0], rec_y[-1] - rec_y[0])))
    w = np.clip(d / b, 0.0, 1.0)
    ix = np.interp(X, rec_x, np.arange(len(rec_x)))
    iy = np.interp(Y, rec_y, np.arange(len(rec_y)))
    ql = map_coordinates(np.asarray(q_les, float), [ix, iy], order=1, mode="nearest")
    return xd, w * ql + (1.0 - w) * qd

In [ ]:
%%writefile flarekit/render.py
"""Gravação dos quadros da LES e renderização do vídeo final.

A LES grava, a cada `frame_dt` de tempo simulado, só o que o vídeo precisa (vista lateral luminosa,
envelope da chama, radiação no solo e séries temporais). O vídeo é montado depois, em segmentos:

  1. abertura (cartão de título);
  2. chama em zoom, em tela cheia ("câmera sintética");
  3. painel com a chama ampliada + validação ao vivo;
  4. varredura das direções do vento (rosa dos ventos do atlas → pegada de radiação girando →
     probabilidade de excedência acumulada);
  5. resumos.

Separar simulação e renderização permite refazer o vídeo (outro zoom, outra paleta) sem rodar a LES.
"""
from __future__ import annotations

import io
import math
import shutil
import subprocess
from pathlib import Path

import numpy as np
from matplotlib import colors as mcolors
from matplotlib.animation import FFMpegWriter
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from scipy.ndimage import gaussian_filter, map_coordinates

from . import semiempirical as se
from .dashboard import API_LEVELS, BG, CHAM_C, FG, GRID, LES_C, MUTED, PANEL, PT_C, _style, frustum_outline_xz

FIRE = mcolors.LinearSegmentedColormap.from_list(
    "fogo", ["#000000", "#1a0303", "#5c0d05", "#a3240a", "#dd5410", "#f7941d", "#ffd15c", "#fff6d8"])
ENVELOPE_C = "#4dd0e1"
W, H_PX, DPI = 16, 9, 120


# ===================================================================== gravação
def _f16(a):
    """float16 sem estouro (máx. 65504): nos primeiros instantes a emissão normalizada por X_rad·Q fica
    concentrada em poucas células e a luminosidade projetada pode passar desse limite."""
    return np.clip(np.asarray(a, np.float32), -6.5e4, 6.5e4).astype(np.float16)


class Recorder:
    """Guarda os quadros da LES (float16) e os resultados médios finais."""

    def __init__(self, les, sc: se.Scenario, title: str, wind_label: str = ""):
        self.static = dict(
            xf=les.xf, zf=les.zf, xc=les.xc, zc=les.zc, yc=les.yc, rec_x=les.rec_x, rec_y=les.rec_y,
            tip=les.tip, Z_st=les.Z_st, X_rad=les.cfg.X_rad, Q=les.Q, u_ref=les.cfg.u_ref,
            receiver_z=les.cfg.receiver_z, summary=les.summary(), title=title, wind_label=wind_label,
            T_inf=sc.T_inf, RH=sc.RH)
        self.frames: list[dict] = []
        self._qsum = None
        self._nq = 0
        self.final: dict = {}

    def snap(self, les, e, q, averaging: bool, T=None):
        tot = float((e * les.vol).sum())
        scale = les.cfg.X_rad * les.Q / max(tot, 1e-30)
        ey = e * les.Dc[1]
        lum = (ey.sum(1) * scale / 1e3).float().cpu().numpy()      # kW/m² projetado
        # temperatura média na linha de visada, ponderada pela emissão (cor da chama no render HD)
        tl = None if T is None else ((ey * T).sum(1) / ey.sum(1).clamp(min=1e-30)).float().cpu().numpy()
        zp = les.Z.max(1).values.float().cpu().numpy()
        qg = les.q_grid(q) / 1e3
        if averaging:
            self._qsum = qg if self._qsum is None else self._qsum + qg
            self._nq += 1
        qmean = self._qsum / self._nq if self._nq else None
        h = les.history
        self.frames.append(dict(
            t=les.time, step=les.step_n, dt=les.dt, cfl=les._last_cfl, wall=les.wall,
            lum=_f16(lum), zp=_f16(zp), q=_f16(qg), tl=None if tl is None else _f16(tl),
            qmean=None if qmean is None else _f16(qmean), averaging=averaging,
            L=h["L"][-1], tilt=h["tilt"][-1], Lmean=les.mean_flame_length(), tiltmean=les.mean_tilt()))

    def finalize(self, les):
        n = max(les.avg_n, 1.0)
        self.final = dict(
            q_mean=les.q_grid(les.avg_q / n) / 1e3,
            I_proj=(les.avg_I / n).max(1).values.float().cpu().numpy(),
            T_proj=(les.avg_T / n).max(1).values.float().cpu().numpy(),
            L_mean=les.mean_flame_length(), tilt_mean=les.mean_tilt(), n_avg=les.avg_n,
            steps=les.step_n, wall=les.wall)

    # -------------------------------------------------------------- disco
    def save(self, path: str):
        keys = ["t", "step", "dt", "cfl", "wall", "L", "tilt", "Lmean", "tiltmean", "averaging"]
        out = {f"s_{k}": np.asarray(v) for k, v in self.static.items()}
        for k in keys:
            out[f"f_{k}"] = np.array([f[k] for f in self.frames], dtype=float)
        for k in ("lum", "zp", "q"):
            out[f"f_{k}"] = np.stack([f[k] for f in self.frames])
        if all(f.get("tl") is not None for f in self.frames):
            out["f_tl"] = np.stack([f["tl"] for f in self.frames])
        qm = [f["qmean"] if f["qmean"] is not None else np.full_like(f["q"], np.nan) for f in self.frames]
        out["f_qmean"] = np.stack(qm)
        for k, v in self.final.items():
            out[f"r_{k}"] = np.asarray(v)
        np.savez_compressed(path, **out)

    @classmethod
    def load(cls, path: str) -> "Recorder":
        d = np.load(path, allow_pickle=False)
        rec = cls.__new__(cls)
        rec.static = {k[2:]: (d[k].item() if d[k].ndim == 0 else d[k]) for k in d.files if k.startswith("s_")}
        A = {k: d[f"f_{k}"] for k in ("t", "step", "dt", "cfl", "wall", "L", "tilt", "Lmean", "tiltmean",
                                       "averaging", "lum", "zp", "q", "qmean", "tl") if f"f_{k}" in d.files}
        rec.frames = []
        for i in range(len(A["t"])):
            f = {k: float(A[k][i]) for k in ("t", "step", "dt", "cfl", "wall", "L", "tilt", "Lmean", "tiltmean")}
            f["averaging"] = bool(A["averaging"][i])
            f["lum"], f["zp"], f["q"] = A["lum"][i], A["zp"][i], A["q"][i]
            f["tl"] = A["tl"][i] if "tl" in A else None
            qm = A["qmean"][i]
            f["qmean"] = None if np.isnan(qm.astype(float)).all() else qm
            rec.frames.append(f)
        rec.final = {k[2:]: (d[k].item() if d[k].ndim == 0 else d[k]) for k in d.files if k.startswith("r_")}
        rec._qsum, rec._nq = None, 0
        return rec


# ===================================================================== utilidades
def flame_view(rec: Recorder, ch: se.Chamberlain, pad: float = 6.0, aspect: float = 16 / 9,
               with_model: bool = True):
    """Janela de zoom em volta da chama: envelope médio da LES (I ≥ 0,2), unido ao frustum de Chamberlain
    quando with_model (painéis que desenham o frustum)."""
    s = rec.static
    fr = frustum_outline_xz(ch, s["tip"])
    xs, zs = list(fr[:, 0]), list(fr[:, 1])
    Ip = rec.final.get("I_proj")
    if Ip is not None:
        m = Ip >= 0.2
        if m.any():
            if not with_model:
                xs, zs = [0.0], [float(s["tip"][2])]
            ii, kk = np.nonzero(m)
            xs += [s["xc"][ii].min(), s["xc"][ii].max()]
            zs += [s["zc"][kk].min(), s["zc"][kk].max()]
    x0, x1 = min(xs) - pad, max(xs) + pad
    z0, z1 = min(zs) - pad, max(zs) + pad
    z1 += 0.15 * (z1 - z0)                          # folga no topo para título e relógio
    z0 = max(z0, s["tip"][2] - 0.45 * (z1 - z0))   # mostra um pedaço da chaminé
    cx, cz, w, h = 0.5 * (x0 + x1), 0.5 * (z0 + z1), x1 - x0, z1 - z0
    if w / h < aspect:
        w = h * aspect
    else:
        h = w / aspect
    return (cx - w / 2, cx + w / 2, cz - h / 2, cz + h / 2)


class Resampler:
    """Reamostra campos (nx, nz) da malha esticada para uma grade uniforme (zoom suave)."""

    def __init__(self, xc, zc, view, nx_out=960, nz_out=540, smooth=0.7):
        self.smooth = smooth   # filtro gaussiano (em células) antes da interpolação bilinear: suaviza os
        x0, x1, z0, z1 = view  # degraus de célula no zoom sem o "ringing" de uma spline cúbica
        self.xs = np.linspace(x0, x1, nx_out)
        self.zs = np.linspace(z0, z1, nz_out)
        ix = np.interp(self.xs, xc, np.arange(len(xc)))
        iz = np.interp(self.zs, zc, np.arange(len(zc)))
        IX, IZ = np.meshgrid(ix, iz, indexing="ij")
        self.coords = np.stack([IX, IZ])
        self.inside_x = (self.xs >= xc[0]) & (self.xs <= xc[-1])
        self.inside_z = (self.zs >= zc[0]) & (self.zs <= zc[-1])

    def __call__(self, f):
        f = np.asarray(f, dtype=np.float32)
        if self.smooth:
            f = gaussian_filter(f, self.smooth, mode="nearest")
        out = map_coordinates(f, self.coords, order=1, mode="nearest")
        out[~self.inside_x, :] = 0.0
        out[:, ~self.inside_z] = 0.0
        return out


class LumNorm:
    """Normalização da luminosidade pelo percentil 99,7 recente (evita cintilação de escala)."""

    def __init__(self, window: int = 40):
        self.hist, self.window = [], window

    def __call__(self, img):
        img = np.nan_to_num(np.asarray(img, np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        self.hist.append(float(np.percentile(img, 99.7)))
        v = max(np.median(self.hist[-self.window:]), 1e-9)
        return np.clip(img / v, 0.0, 1.0)


def _writer(fig, path, fps):
    w = FFMpegWriter(fps=fps, codec="libx264", bitrate=12000,
                     extra_args=["-pix_fmt", "yuv420p", "-preset", "medium", "-crf", "16"])
    w.setup(fig, path, dpi=DPI)
    return w


def qnorm(qmax: float) -> mcolors.PowerNorm:
    """Escala de cor para fluxo no solo [kW/m²] ajustada ao máximo (tochas altas ficam bem abaixo de 10)."""
    for v in (1, 2, 3, 5, 8, 10, 15, 20, 30, 50, 100):
        if v >= 1.1 * qmax:
            break
    return mcolors.PowerNorm(0.6, 0, v)


def _fit_text(fig, txt, max_frac: float, min_size: float = 7.0):
    """Reduz a fonte de um texto até caber em max_frac da largura da figura."""
    r = fig.canvas.get_renderer()
    while txt.get_fontsize() > min_size and txt.get_window_extent(r).width > max_frac * fig.bbox.width:
        txt.set_fontsize(txt.get_fontsize() - 0.5)
    return txt


def _new_fig():
    fig = Figure(figsize=(W, H_PX), dpi=DPI, facecolor=BG)
    FigureCanvasAgg(fig)
    return fig


def _wind_arrow(ax, label, x=0.03, y=0.93, length=0.10, fontsize=12):
    """Seta do vento (sempre da esquerda para a direita na vista lateral), em coordenadas do eixo."""
    ax.annotate("", xy=(x + length, y), xytext=(x, y), xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", color=CHAM_C, lw=2.5), zorder=6)
    ax.text(x, y - 0.025, label, color=CHAM_C, fontsize=fontsize, va="top", transform=ax.transAxes, zorder=6)


# ===================================================================== segmentos
def render_title(path: str, title: str, subtitle: str, credit: str = "", seconds: float = 3.0, fps: int = 30,
                 note: str = ""):
    fig = _new_fig()
    _fit_text(fig, fig.text(0.5, 0.58, title, ha="center", color=FG, fontsize=40, fontweight="bold"), 0.94)
    _fit_text(fig, fig.text(0.5, 0.48, subtitle, ha="center", color=LES_C, fontsize=20), 0.94)
    if credit:
        fig.text(0.5, 0.38, credit, ha="center", color=MUTED, fontsize=14)
    if note:
        _fit_text(fig, fig.text(0.5, 0.08, note, ha="center", color=MUTED, fontsize=13, style="italic"), 0.94)
    w = _writer(fig, path, fps)
    for _ in range(int(seconds * fps)):
        w.grab_frame(facecolor=BG)
    w.finish()
    return path


def render_hero(rec: Recorder, ch: se.Chamberlain, path: str, title: str, subtitle: str, fps: int = 30,
                every: int = 1, view=None):
    """Chama em tela cheia, ampliada: vista lateral da emissão luminosa da LES."""
    s = rec.static
    view = view or flame_view(rec, ch, pad=4.0, with_model=False)   # zoom só na chama da LES
    rs = Resampler(s["xc"], s["zc"], view)
    norm = LumNorm()
    fig = _new_fig()
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_facecolor("black")
    ax.set_axis_off()
    im = ax.imshow(np.zeros((len(rs.zs), len(rs.xs))), origin="lower", extent=view, cmap=FIRE,
                   norm=mcolors.PowerNorm(0.5, 0, 1), interpolation="bilinear", aspect="equal")
    ax.set_xlim(view[0], view[1]); ax.set_ylim(view[2], view[3])
    tip = s["tip"]
    ax.add_patch(Rectangle((-0.45, view[2] - 5), 0.9, tip[2] - view[2] + 5, color="#59616b", zorder=4))
    if s["u_ref"] > 0:
        _wind_arrow(ax, "vento", x=0.03, y=0.80, length=0.08, fontsize=14)
    fig.text(0.03, 0.93, title, color=FG, fontsize=26, fontweight="bold")
    fig.text(0.03, 0.885, subtitle, color=MUTED, fontsize=15)
    clock = fig.text(0.97, 0.93, "", color=LES_C, fontsize=26, ha="right", family="monospace", fontweight="bold")
    readout = fig.text(0.97, 0.885, "", color=FG, fontsize=15, ha="right", family="monospace")
    fig.text(0.97, 0.03, "LES 3D · emissão luminosa integrada na linha de visada (câmera sintética)",
             color=MUTED, fontsize=12, ha="right")
    # barra de escala de 10 m
    sb_x = view[1] - 0.03 * (view[1] - view[0]) - 10.0     # canto inferior direito, longe da chaminé
    sb_z = view[2] + 0.09 * (view[3] - view[2])
    ax.plot([sb_x, sb_x + 10], [sb_z, sb_z], color=FG, lw=3, zorder=6)
    ax.text(sb_x + 5, sb_z + 0.02 * (view[3] - view[2]), "10 m", color=FG, fontsize=13, ha="center", zorder=6)
    w = _writer(fig, path, fps)
    for f in rec.frames[::every]:
        im.set_data(norm(rs(f["lum"])).T)
        clock.set_text(f"t = {f['t']:5.1f} s")
        readout.set_text(f"L = {f['L']:4.1f} m")
        w.grab_frame(facecolor=BG)
    w.finish()
    return path


class ZoomDashboard:
    """Painel com a chama ampliada (2/3 da tela) + mapa, fluxo no solo e comprimento da chama."""

    def __init__(self, rec: Recorder, sc: se.Scenario, ch: se.Chamberlain, title: str, subtitle: str, view=None,
                 t_end: float | None = None):
        s = rec.static
        self.rec, self.ch, self.t_end = rec, ch, t_end
        self.view = view or flame_view(rec, ch, aspect=1.2)
        self.rs = Resampler(s["xc"], s["zc"], self.view, 720, 600)
        self.norm = LumNorm()
        fig = self.fig = _new_fig()
        gs = GridSpec(3, 3, figure=fig, left=0.035, right=0.985, top=0.875, bottom=0.07, hspace=0.5, wspace=0.28,
                      width_ratios=[1, 1, 0.95])
        self.ax = fig.add_subplot(gs[:, 0:2])
        self.ax_m = fig.add_subplot(gs[0, 2])
        self.ax_l = fig.add_subplot(gs[1, 2])
        self.ax_t = fig.add_subplot(gs[2, 2])
        fig.text(0.035, 0.955, title, color=FG, fontsize=18, fontweight="bold")
        _fit_text(fig, fig.text(0.035, 0.918, subtitle, color=MUTED, fontsize=11), 0.95)
        self.clock = fig.text(0.985, 0.95, "", color=LES_C, fontsize=22, ha="right", family="monospace",
                              fontweight="bold")
        # chama
        ax = self.ax
        _style(ax, "LES: emissão luminosa (câmera sintética) · envelope Z̃ = Z_st · Chamberlain (1987)")
        v = self.view
        self.im = ax.imshow(np.zeros((len(self.rs.zs), len(self.rs.xs))), origin="lower", extent=v, cmap=FIRE,
                            norm=mcolors.PowerNorm(0.5, 0, 1), interpolation="bilinear", aspect="equal")
        tip = s["tip"]
        ax.add_patch(Rectangle((-0.45, v[2] - 5), 0.9, tip[2] - v[2] + 5, color="#59616b", zorder=4))
        fr = frustum_outline_xz(ch, tip)
        ax.plot(fr[:, 0], fr[:, 1], "--", color=CHAM_C, lw=1.8, zorder=5, label="Chamberlain (1987)")
        ax.plot([], [], color=ENVELOPE_C, lw=1.6, label="LES: envelope da chama")
        ax.set_xlim(v[0], v[1]); ax.set_ylim(v[2], v[3])
        ax.set_xlabel("x [m] (a jusante)"); ax.set_ylabel("z [m]")
        ax.legend(loc="upper right", fontsize=10, facecolor=PANEL, edgecolor=GRID, labelcolor=FG)
        if s["u_ref"] > 0:
            _wind_arrow(ax, s.get("wind_label") or f"{s['u_ref']:.1f} m/s")
        self.env = None
        self.box = ax.text(0.015, 0.03, "", transform=ax.transAxes, color=FG, fontsize=11, family="monospace",
                           va="bottom", bbox=dict(facecolor=PANEL, edgecolor=GRID, alpha=0.85, pad=6), zorder=7)
        # mapa no solo (escala de cor pelo maior fluxo previsto: LES média, Chamberlain ou fonte pontual)
        xr = s["rec_x"]
        recp = np.stack([xr, np.zeros_like(xr), np.full_like(xr, float(s["receiver_z"]))], 1)
        self.q_ch = se.q_chamberlain(recp, ch, tip, sc.T_inf, sc.RH) / 1e3
        self.q_pt = se.q_point_source(recp, se.flame_center_chamberlain(ch, tip), float(s["X_rad"]), sc.Q,
                                      sc.T_inf, sc.RH) / 1e3
        q_ref = max(self.q_ch.max(), self.q_pt.max(),
                    float(np.nanmax(rec.final["q_mean"])) if rec.final.get("q_mean") is not None else 0.0)
        ax = self.ax_m
        _style(ax, "Radiação no solo [kW/m²]")
        q0 = rec.frames[0]["q"].astype(float)
        self.qmesh = ax.pcolormesh(s["rec_x"], s["rec_y"], q0.T, cmap="magma",
                                   norm=qnorm(1.3 * q_ref), shading="nearest", rasterized=True)
        cb = fig.colorbar(self.qmesh, ax=ax, pad=0.01, fraction=0.04)
        cb.ax.tick_params(colors=MUTED, labelsize=8); cb.outline.set_edgecolor(GRID)
        for lev, c in API_LEVELS:                       # níveis do API 521 marcados na barra de cores
            cb.ax.axhline(lev, color=c, lw=2)
        ax.plot([0], [0], marker="^", color="white", ms=7, mec="black", zorder=5)
        ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
        self.qc = None
        self.map_note = ax.text(0.02, 0.04, "", transform=ax.transAxes, color=FG, fontsize=8)
        # fluxo ao longo do vento
        ax = self.ax_l
        _style(ax, "Fluxo no solo, y = 0 [kW/m²]")
        ax.plot(xr, self.q_ch, "--", color=CHAM_C, lw=1.6, label="Chamberlain")
        ax.plot(xr, self.q_pt, ":", color=PT_C, lw=1.6, label="API 521 (pontual)")
        self.l_i, = ax.plot(xr, np.zeros_like(xr), color=LES_C, lw=0.9, alpha=0.45)
        self.l_m, = ax.plot(xr, np.zeros_like(xr), color=LES_C, lw=2.4, label="LES média")
        for lev, c in API_LEVELS:
            ax.axhline(lev, color=c, lw=0.7, alpha=0.6)
        ax.set_xlim(xr[0], xr[-1]); ax.set_ylim(0, 1.6 * q_ref)
        ax.grid(color=GRID, lw=0.5)
        ax.legend(fontsize=7.5, facecolor=PANEL, edgecolor=GRID, labelcolor=FG, ncol=3, loc="upper right")
        self.jl = int(np.argmin(np.abs(s["rec_y"])))
        # comprimento da chama
        ax = self.ax_t
        _style(ax, "Comprimento da chama [m]")
        ax.axhline(ch.L_b, color=CHAM_C, ls="--", lw=1.6, label=f"Chamberlain {ch.L_b:.1f} m")
        self.t_i, = ax.plot([], [], color=LES_C, lw=0.9, alpha=0.5)
        self.t_m, = ax.plot([], [], color=LES_C, lw=2.4, label="LES média")
        ax.set_ylim(0, 1.8 * ch.L_b); ax.set_xlabel("t [s]")
        ax.grid(color=GRID, lw=0.5)
        ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID, labelcolor=FG, loc="upper right")
        self.ts, self.Ls, self.tm, self.Lm = [], [], [], []

    def draw(self, i: int):
        f = self.rec.frames[i]
        s = self.rec.static
        ch = self.ch
        self.im.set_data(self.norm(self.rs(f["lum"])).T)
        if self.env is not None:
            self.env.remove()
        zp = f["zp"].astype(float)
        self.env = self.ax.contour(s["xc"], s["zc"], zp.T, levels=[s["Z_st"]], colors=[ENVELOPE_C],
                                   linewidths=1.5, zorder=5)
        show = f["qmean"] if (f["averaging"] and f["qmean"] is not None) else f["q"]
        show = show.astype(float)
        self.qmesh.set_array(show.T.ravel())
        if self.qc is not None:
            self.qc.remove()
        lv = [(l, c) for l, c in API_LEVELS if l < show.max()]
        self.qc = self.ax_m.contour(s["rec_x"], s["rec_y"], show.T, levels=[l for l, _ in lv],
                                    colors=[c for _, c in lv], linewidths=1.5) if lv else None
        self.map_note.set_text("média temporal" if f["averaging"] else "instantâneo")
        self.l_i.set_ydata(f["q"].astype(float)[:, self.jl])
        if f["qmean"] is not None:
            self.l_m.set_ydata(f["qmean"].astype(float)[:, self.jl])
        self.ts.append(f["t"]); self.Ls.append(f["L"])
        self.t_i.set_data(self.ts, self.Ls)
        if f["averaging"] and math.isfinite(f["Lmean"]):
            self.tm.append(f["t"]); self.Lm.append(f["Lmean"])
            self.t_m.set_data(self.tm, self.Lm)
        fmt = lambda v, p="{:5.1f}": "  —  " if not math.isfinite(v) else p.format(v)  # noqa: E731
        qmax = float(f["qmean"].astype(float).max()) if f["qmean"] is not None else float("nan")
        self.box.set_text("\n".join([
            "              LES   Chamberlain",
            f"L [m]       {fmt(f['Lmean'])}   {ch.L_b:5.1f}",
            f"incl. [°]   {fmt(f['tiltmean'])}   {ch.alpha:5.1f}",
            f"q máx [kW/m²] {fmt(qmax, '{:4.2f}')}   {self.q_ch.max():4.2f}",
        ]))
        self.ax_t.set_xlim(0, max(self.t_end or 0.0, 1.02 * f["t"], 5.0))
        self.clock.set_text(f"t = {f['t']:5.1f} s")

    def render(self, path: str, fps: int = 30, every: int = 1):
        w = _writer(self.fig, path, fps)
        for i in range(0, len(self.rec.frames), every):
            self.draw(i)
            w.grab_frame(facecolor=BG)
        w.finish()
        return path

    def png(self, i: int = -1) -> bytes:
        self.draw(i % len(self.rec.frames))
        self.fig.canvas.draw()
        buf = io.BytesIO()
        from PIL import Image as PILImage
        PILImage.fromarray(np.asarray(self.fig.canvas.buffer_rgba())[..., :3]).save(buf, format="JPEG", quality=85)
        return buf.getvalue()


def render_still(src_png: str, path: str, seconds: float = 5.0, fps: int = 30):
    """Segmento de vídeo a partir de uma imagem fixa (1920×1080)."""
    ff = shutil.which("ffmpeg")
    subprocess.run([ff, "-y", "-loop", "1", "-framerate", str(fps), "-t", f"{seconds}", "-i", src_png,
                    "-vf", "scale=1920:1080,setsar=1,format=yuv420p", "-c:v", "libx264", "-crf", "16",
                    "-preset", "medium", path], check=True, capture_output=True)
    return path


def assemble(segments: list[str], out: str, fps: int = 30, size: tuple = (1920, 1080), crf: int = 16) -> str:
    """Concatena os segmentos em um MP4 final na resolução `size` (os de outra resolução são reescalados
    com Lanczos, preservando o aspecto)."""
    ff = shutil.which("ffmpeg")
    if ff is None:
        raise RuntimeError("ffmpeg não encontrado")
    segs = [p for p in segments if p and Path(p).exists()]
    cmd = [ff, "-y"]
    for p in segs:
        cmd += ["-i", p]
    W, H = size
    chains = "".join(f"[{i}:v]scale={W}:{H}:flags=lanczos:force_original_aspect_ratio=decrease,"
                     f"pad={W}:{H}:(ow-iw)/2:(oh-ih)/2,setsar=1,fps={fps},format=yuv420p[v{i}];"
                     for i in range(len(segs)))
    cmd += ["-filter_complex", chains + "".join(f"[v{i}]" for i in range(len(segs)))
            + f"concat=n={len(segs)}:v=1:a=0[v]", "-map", "[v]", "-c:v", "libx264", "-crf", str(crf),
            "-preset", "slow", "-tune", "film", "-movflags", "+faststart", out]
    subprocess.run(cmd, check=True, capture_output=True)
    return out


class RecordedLES:
    """Substitui o objeto FlareLES a partir de um flare_quadros.npz (refazer figuras e vídeo sem rodar a LES):
    expõe só o que validation_rows, summary_figure e as células do notebook usam."""

    def __init__(self, rec: Recorder):
        import torch
        from types import SimpleNamespace
        s, f = rec.static, rec.final
        n = float(f.get("n_avg", 1.0)) or 1.0
        self.rec = rec
        self.wind = float(s["u_ref"]) > 0
        self.tip, self.rec_x, self.rec_y = np.asarray(s["tip"]), np.asarray(s["rec_x"]), np.asarray(s["rec_y"])
        self.xf, self.zf, self.xc, self.zc = (np.asarray(s[k]) for k in ("xf", "zf", "xc", "zc"))
        self.avg_n = n
        self.avg_q = torch.from_numpy(np.asarray(f["q_mean"], float) * 1e3 * n).reshape(-1)
        self.avg_T = torch.from_numpy(np.asarray(f["T_proj"], float)[:, None, :] * n)
        self.avg_I = torch.from_numpy(np.asarray(f["I_proj"], float)[:, None, :] * n)
        self.cfg = SimpleNamespace(receiver_z=float(s["receiver_z"]), X_rad=float(s["X_rad"]))
        self.dev = SimpleNamespace(type="cuda" if "CUDA" in str(s["summary"]) else "cpu")
        self.step_n = int(f.get("steps", 0))
        self.wall = float(f.get("wall", 0.0))

    def mean_flame_length(self):
        return float(self.rec.final["L_mean"])

    def mean_tilt(self):
        return float(self.rec.final["tilt_mean"])

    def q_grid(self, q):
        return q.reshape(len(self.rec_x), len(self.rec_y)).cpu().numpy()

    def summary(self):
        return str(self.rec.static["summary"])


def run_and_record(les, sc: se.Scenario, title: str, t_end: float, t_avg: float, frame_dt: float = 0.1,
                   live_every: int = 5, dash_title: str = "", dash_sub: str = "", wind_label: str = "",
                   log_every: int = 50):
    """Roda a LES gravando um quadro a cada frame_dt; mostra o painel ao vivo a cada live_every quadros."""
    rec = Recorder(les, sc, title, wind_label)
    ch = sc.cham if les.wind else sc.cham0
    handle = None
    dash = None
    next_frame = 0.0
    try:
        from IPython.display import Image, display
    except ImportError:
        display = None
    try:
        while les.time < t_end:
            les.step()
            if les.time >= next_frame:
                averaging = les.time >= t_avg
                T, e, mask, q = les.diagnostics(averaging)
                rec.snap(les, e, q, averaging, T)
                next_frame += frame_dt
                n = len(rec.frames)
                if live_every and n % live_every == 0 and display is not None:
                    if dash is None:
                        dash = ZoomDashboard(rec, sc, ch, dash_title or title, dash_sub, t_end=t_end)
                    img = Image(data=dash.png(n - 1), format="jpeg")
                    if handle is None:
                        handle = display(img, display_id=True)
                    else:
                        handle.update(img)
                if log_every and n % log_every == 0:
                    print(f"t = {les.time:6.2f} s · L = {les.history['L'][-1]:5.1f} m · "
                          f"{les.wall / les.step_n * 1e3:5.0f} ms/passo", flush=True)
    except (KeyboardInterrupt, FloatingPointError, RuntimeError) as exc:
        # interrompido (ou divergiu): mantém os quadros já gravados para o vídeo
        print(f"LES interrompida em t = {les.time:.1f} s ({type(exc).__name__}: {exc}); seguindo com "
              f"{len(rec.frames)} quadros", flush=True)
    rec.finalize(les)
    return rec


# ===================================================================== vento
SPEED_COLORS = ["#1b3a5c", "#245b8a", "#2f7fb8", "#4aa3d8", "#7cc4ea", "#b5e0f6", "#e8f6ff"]


def _rose(ax, cl, highlight: int | None = None):
    from .wind import SECTORS, SPEED_EDGES
    ax.set_facecolor(PANEL)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    th = np.radians(np.arange(16) * 22.5)
    width = np.radians(22.5) * 0.9
    bottom = np.zeros(16)
    nb = cl.freq.shape[1]
    bars = []
    for b in range(nb):
        f = 100 * cl.freq[:, b]
        lab = (f"{SPEED_EDGES[b]:.1f}–{SPEED_EDGES[b + 1]:.0f}" if np.isfinite(SPEED_EDGES[b + 1])
               else f"> {SPEED_EDGES[b]:.0f}")
        bars.append(ax.bar(th, f, width=width, bottom=bottom, color=SPEED_COLORS[b % len(SPEED_COLORS)],
                           edgecolor=PANEL, lw=0.5, label=lab))
        bottom = bottom + f
    ax.set_xticks(th)
    ax.set_xticklabels(SECTORS, color=FG, fontsize=8)
    ax.tick_params(axis="y", colors=MUTED, labelsize=7)
    ax.grid(color=GRID, lw=0.6)
    ax.spines["polar"].set_color(GRID)
    hl = None
    if highlight is not None:
        hl = ax.bar([th[highlight]], [bottom[highlight] * 1.06], width=width * 1.05, bottom=0, fill=False,
                    edgecolor="white", lw=2.2)
    return bottom, hl


def _site_map(ax, grid, title):
    _style(ax, title)
    ax.set_aspect("equal")
    ax.set_xlabel("E [m]"); ax.set_ylabel("N [m]")
    ax.plot([0], [0], marker="^", color="white", ms=8, mec="black", zorder=6)
    R = grid.E.max()
    ax.annotate("N", xy=(0.93, 0.95), xytext=(0.93, 0.83), xycoords="axes fraction", color=FG, ha="center",
                fontsize=11, fontweight="bold", arrowprops=dict(arrowstyle="-|>", color=FG, lw=1.8))
    ax.set_xlim(-R, R); ax.set_ylim(-R, R)


def _zoom_radius(maps, level: float = 1.58) -> float:
    """Raio do mapa a mostrar: 1,3 × alcance da envoltória no menor nível API (mín. 40 m); se nenhum
    nível é atingido (tocha alta), usa o alcance de metade do fluxo máximo."""
    grid = maps["grid"]
    EE, NN = np.meshgrid(grid.E, grid.N, indexing="ij")
    env = maps["envelope"]
    m = env >= (level if env.max() >= level else 0.5 * env.max())
    r = float(np.hypot(EE, NN)[m].max()) if m.any() else 0.0
    return float(min(grid.E.max(), max(40.0, 10.0 * math.ceil(1.3 * r / 10.0))))


def sweep_level(maps) -> float | None:
    """Maior nível do API 521 (entre 4,73 e 1,58 kW/m²) com probabilidade > 0 em algum ponto; None se
    nenhum é atingido no solo."""
    for L in (4.73, 1.58):
        if L in maps["P"] and maps["P"][L].max() > 0:
            return L
    return None


def render_wind_sweep(cl, maps, les_down, path: str, U_les: float, title: str, level: float | None = None,
                      seconds: float = 14.0, hold: float = 2.5, fps: int = 30):
    """Varredura das direções: rosa dos ventos, pegada da LES girando e P(q ≥ nível) acumulada
    (nível automático; se nenhum nível do API 521 é atingido, mostra o máximo acumulado de q)."""
    from .wind import SECTORS, rotate_to_site
    grid = maps["grid"]
    xd, qd = les_down
    level = level if level is not None else sweep_level(maps)
    Psec = maps["P_sector"][level] if level is not None else None
    fig = _new_fig()
    gs = GridSpec(1, 3, figure=fig, left=0.03, right=0.975, top=0.83, bottom=0.08, wspace=0.28,
                  width_ratios=[0.9, 1, 1])
    fig.text(0.03, 0.94, title, color=FG, fontsize=18, fontweight="bold")
    fig.text(0.03, 0.895, f"{cl.series.source} · {cl.series.period} · perfil log z0 = {cl.z0:.2f} m · "
             f"vento na altura do tip ({cl.H:.0f} m)", color=MUTED, fontsize=11)
    ax_r = fig.add_subplot(gs[0, 0], projection="polar")
    ax_r.set_title("Rosa dos ventos [% do tempo]", color=FG, fontsize=11, fontweight="bold", pad=18)
    _, hl = _rose(ax_r, cl, 0)
    leg = ax_r.legend(loc="upper center", bbox_to_anchor=(0.5, -0.09), fontsize=8, ncol=4, facecolor=PANEL,
                      edgecolor=GRID, labelcolor=FG, title="velocidade no tip [m/s]", title_fontsize=8)
    leg.get_title().set_color(MUTED)
    ax_f = fig.add_subplot(gs[0, 1])
    _site_map(ax_f, grid, f"Pegada da LES ({U_les:.1f} m/s) girando com o vento [kW/m²]")
    q0 = rotate_to_site(xd, xd, qd, grid, 0.0)
    qn = qnorm(max(float(qd.max()), 0.3))
    m1 = ax_f.pcolormesh(grid.E, grid.N, q0.T, cmap="magma", norm=qn, shading="nearest", rasterized=True)
    cb = fig.colorbar(m1, ax=ax_f, fraction=0.045, pad=0.02)
    cb.ax.tick_params(colors=MUTED, labelsize=8); cb.outline.set_edgecolor(GRID)
    ax_p = fig.add_subplot(gs[0, 2])
    if level is not None:
        _site_map(ax_p, grid, f"P(q ≥ {level:.2f} kW/m²) acumulada [%]")
        Pmax = max(1e-6, 100 * float(maps["P"][level].max()))
        m2 = ax_p.pcolormesh(grid.E, grid.N, np.ma.masked_all((len(grid.N), len(grid.E))), cmap="YlOrRd",
                             norm=mcolors.Normalize(0, Pmax), shading="nearest", rasterized=True)
    else:
        _site_map(ax_p, grid, "q máximo acumulado nas direções [kW/m²]")
        m2 = ax_p.pcolormesh(grid.E, grid.N, np.zeros((len(grid.N), len(grid.E))), cmap="magma", norm=qn,
                             shading="nearest", rasterized=True)
        note = ("nenhum nível do API 521 (≥ 1,58 kW/m²)\né atingido no solo" if maps["envelope"].max() < 1.58
                else "P = 0 nas classes de vento; só a envoltória\n(vento máximo observado) atinge 1,58 kW/m²")
        ax_p.text(0.03, 0.04, note, transform=ax_p.transAxes, color=FG, fontsize=9.5,
                  bbox=dict(facecolor=PANEL, edgecolor=GRID, alpha=0.85, pad=4))
    q_acc = np.zeros((len(grid.E), len(grid.N)))
    cb = fig.colorbar(m2, ax=ax_p, fraction=0.045, pad=0.02)
    cb.ax.tick_params(colors=MUTED, labelsize=8); cb.outline.set_edgecolor(GRID)
    Rz = _zoom_radius(maps)
    for a in (ax_f, ax_p):
        a.set_xlim(-Rz, Rz); a.set_ylim(-Rz, Rz)
    lab = fig.text(0.5, 0.78, "", color=LES_C, fontsize=17, fontweight="bold", ha="center")
    arrow = [None]
    cont = [None, None]
    w = _writer(fig, path, fps)
    n = int(seconds * fps)
    for i in range(n + int(hold * fps)):
        theta = 360.0 * min(i, n - 1) / n
        kr = int((theta + 11.25) // 22.5)          # sem "dar a volta": controla o acúmulo
        k = kr % 16
        q = rotate_to_site(xd, xd, qd, grid, theta)
        m1.set_array(q.T.ravel())
        if cont[0] is not None:
            cont[0].remove()
        qs = float(maps.get("q_solar", 0.0))      # mesmo critério dos mapas de probabilidade
        lv = [(L - qs, c) for L, c in API_LEVELS if 0 < L - qs < q.max()]
        cont[0] = ax_f.contour(grid.E, grid.N, q.T, levels=[L for L, _ in lv], colors=[c for _, c in lv],
                               linewidths=1.5) if lv else None
        done = 16 if i >= n else min(16, kr + 1)
        if cont[1] is not None:
            cont[1].remove()
        if Psec is not None:
            Pacc = 100 * Psec[:done].sum(0)
            m2.set_array(np.ma.masked_less(Pacc, 0.05).T.ravel())
            lv = [p for p in (1, 5, 10, 20) if p < Pacc.max()]
            cont[1] = ax_p.contour(grid.E, grid.N, Pacc.T, levels=lv, colors="white", linewidths=0.9) if lv else None
        else:
            q_acc = np.maximum(q_acc, q)
            m2.set_array(q_acc.T.ravel())
            cont[1] = None
        if hl is not None:
            hl.remove()
        tops = 100 * cl.freq.sum(1)
        hl = ax_r.bar([np.radians(22.5 * k)], [tops[k] * 1.08], width=np.radians(22.5) * 0.95, bottom=0,
                      fill=False, edgecolor="white", lw=2.2)
        if arrow[0] is not None:
            arrow[0].remove()
        R = Rz
        th = math.radians(theta)
        dE, dN = -math.sin(th), -math.cos(th)
        arrow[0] = ax_f.annotate("", xy=(0.55 * R * dE, 0.55 * R * dN), xytext=(-0.85 * R * dE, -0.85 * R * dN),
                                 arrowprops=dict(arrowstyle="-|>", color=CHAM_C, lw=2.2, alpha=0.9))
        lab.set_text(f"vento de {SECTORS[k]} ({theta:5.1f}°) · {100 * cl.sector_freq[k]:4.1f}% do tempo")
        w.grab_frame(facecolor=BG)
    w.finish()
    return path


def risk_summary_figure(cl, maps, sc, path: str, title: str):
    """Resumo do clima de vento e das zonas probabilísticas (PNG 1920×1080)."""
    grid = maps["grid"]
    fig = _new_fig()
    fig.text(0.035, 0.955, title, color=FG, fontsize=18, fontweight="bold")
    d = cl.describe().split("\n")
    fig.text(0.035, 0.918, "  ·  ".join(d[:2]), color=MUTED, fontsize=10)
    fig.text(0.035, 0.893, "  ·  ".join(d[2:]), color=MUTED, fontsize=10)
    gl = GridSpec(2, 1, figure=fig, left=0.045, right=0.29, top=0.80, bottom=0.07, hspace=0.42,
                  height_ratios=[1.15, 1])
    ax = fig.add_subplot(gl[0], projection="polar")
    ax.set_title("Rosa dos ventos na altura do tip [% do tempo]", color=FG, fontsize=11, fontweight="bold",
                 pad=22)
    _rose(ax, cl, cl.dominant_sector)
    ax = fig.add_subplot(gl[1])
    _style(ax, "Perfil de camada limite (lei log)")
    z = np.linspace(max(cl.z0 * 2, 1.0), max(120.0, 1.5 * cl.H), 200)
    s = cl.series
    ax.plot(cl.profile(z), z, color=LES_C, lw=2.2, label=f"lei log, z0 = {cl.z0:.2f} m")
    ax.plot([s.U1.mean(), s.U2.mean()], [s.z1, s.z2], "o", color=CHAM_C, ms=8, label="médias dos dados")
    ax.axhline(cl.H, color=PT_C, ls=":", lw=1.4, label=f"tip ({cl.H:.0f} m)")
    ax.set_xlabel("U médio [m/s]"); ax.set_ylabel("z [m]"); ax.grid(color=GRID, lw=0.5)
    ax.legend(fontsize=8.5, facecolor=PANEL, edgecolor=GRID, labelcolor=FG, loc="upper left")
    gr = GridSpec(2, 2, figure=fig, left=0.35, right=0.975, top=0.845, bottom=0.035, hspace=0.16, wspace=0.22,
                  height_ratios=[3.6, 1])
    env = maps["envelope"]
    Rz = _zoom_radius(maps)
    reached = [L for L in (4.73, 1.58) if maps["P"][L].max() > 0]
    # tochas altas podem não atingir nenhum nível no solo: aí mostram-se os próprios fluxos
    panels = (["P4.73", "P1.58"] if len(reached) == 2 else ["P1.58", "env"] if reached else ["env", "mean"])
    qn = qnorm(max(float(env.max()), 0.5))
    for j, kind in enumerate(panels):
        ax = fig.add_subplot(gr[0, j])
        if kind.startswith("P"):
            L = float(kind[1:])
            _site_map(ax, grid, f"P(q ≥ {L:.2f} kW/m²) com o flare queimando [%]")
            P = 100 * maps["P"][L]
            m = ax.pcolormesh(grid.E, grid.N, np.ma.masked_less(P, 0.05).T, cmap="YlOrRd", shading="nearest",
                              rasterized=True, vmin=0, vmax=max(1.0, float(P.max())))
            lv = [p for p in (1, 5, 10, 20, 50) if p < P.max()]
            if lv:
                cs = ax.contour(grid.E, grid.N, P.T, levels=lv, colors="white", linewidths=1.0)
                ax.clabel(cs, fmt="%d%%", fontsize=8, colors="white")
            if env.max() > L:
                ax.contour(grid.E, grid.N, env.T, levels=[L], colors=[CHAM_C], linewidths=1.8, linestyles="--")
            ax.plot([], [], "--", color=CHAM_C, label="envoltória (pior direção)")
            ax.plot([], [], color="white", lw=1.0, label="isolinhas de P [%]")
        else:
            field = env if kind == "env" else maps["q_mean"]
            _site_map(ax, grid, "q máximo em qualquer direção [kW/m²]" if kind == "env"
                      else "q médio ponderado pela rosa dos ventos [kW/m²]")
            m = ax.pcolormesh(grid.E, grid.N, field.T, cmap="magma", norm=qn, shading="nearest", rasterized=True)
            lv = [(l, c) for l, c in API_LEVELS if l < field.max()]
            if lv:
                ax.contour(grid.E, grid.N, field.T, levels=[l for l, _ in lv], colors=[c for _, c in lv],
                           linewidths=1.6)
            ax.plot([], [], color="none", label=f"máximo: {field.max():.2f} kW/m²")
        ax.set_xlim(-Rz, Rz); ax.set_ylim(-Rz, Rz)
        cb = fig.colorbar(m, ax=ax, fraction=0.045, pad=0.02)
        cb.ax.tick_params(colors=MUTED, labelsize=8); cb.outline.set_edgecolor(GRID)
        ax.legend(loc="lower left", fontsize=8.5, facecolor=PANEL, edgecolor=GRID, labelcolor=FG)
    EE, NN = np.meshgrid(grid.E, grid.N, indexing="ij")
    Rr = np.hypot(EE, NN)

    def reach(mask):
        return float(Rr[mask].max()) if mask.any() else 0.0

    def edge(mask):   # a zona encosta na borda do mapa: o alcance é só um limite inferior
        return bool(mask[0].any() or mask[-1].any() or mask[:, 0].any() or mask[:, -1].any())

    rows, cells = [], []
    for lev, _ in API_LEVELS:
        ms = (env >= lev, maps["P"][lev] >= 0.01, maps["P"][lev] >= 0.10)
        rows.append((lev, *(reach(m) for m in ms)))
        cells.append([f"{lev:.2f}"] + [("≥ " if edge(m) else "") + f"{reach(m):.0f}" for m in ms])
    ax = fig.add_subplot(gr[1, :])
    ax.set_axis_off()
    tb = ax.table(cellText=cells, colLabels=["nível [kW/m²]", "envoltória [m]", "P ≥ 1% [m]", "P ≥ 10% [m]"],
                  loc="center", cellLoc="center", colLoc="center", bbox=[0.0, 0.0, 0.55, 1.0])
    tb.auto_set_font_size(False)
    tb.set_fontsize(10)
    for (r, c), cell in tb.get_celld().items():
        cell.set_edgecolor(GRID)
        cell.set_facecolor(PANEL if r else "#1d2733")
        cell.get_text().set_color(API_LEVELS[r - 1][1] if (c == 0 and r) else FG)
    qs = float(maps.get("q_solar", 0.0))
    crit = (f"com a radiação do flare + {qs:.2f} kW/m² de solar" if qs > 0
            else "só com a radiação do flare (sem solar)")
    ax.text(0.58, 0.5, "Alcance máximo (a partir da base do flare) em que o nível\n"
            "é atingido: na pior direção (envoltória, até a maior velocidade\n"
            f"observada, {maps.get('U_max', 0.0):.1f} m/s) e com probabilidade ≥ 1% / ≥ 10% do\n"
            "tempo de queima, ponderada pela rosa. Níveis do API 521\n"
            f"comparados {crit}.",
            transform=ax.transAxes, color=MUTED, fontsize=9.5, va="center")
    fig.savefig(path, dpi=DPI, facecolor=BG)
    return rows

In [ ]:
%%writefile flarekit/hd.py
"""Renderização HD da chama (pós-processamento dos quadros gravados da LES).

O campo da LES tem Δ ≈ 0,5 m; em 1080p isso dá ~20 px por célula. Para um vídeo nítido, cada quadro é
composto em PyTorch (GPU quando houver):

* cor física: emissão espectral da fuligem (ε ∝ λ^-1,39) à temperatura média da linha de visada,
  integrada nas funções de cor CIE 1931 → sRGB linear;
* brilho: luminosidade integrada da LES (câmera sintética) × fração visível da emissão;
* realce de detalhe abaixo da malha (opcional): ruído fractal advectado com o vento que deforma e modula
  a chama só na escala da célula — efeito visual, não altera nenhum resultado da simulação;
* tone mapping fílmico (ACES), bloom em três escalas, céu noturno e chaminé iluminada pela chama;
* textos com PIL e codificação H.264 de alta qualidade pelo ffmpeg (sem passar pelo matplotlib).
"""
from __future__ import annotations

import math
import shutil
import subprocess
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

RES = {"1080p": (1920, 1080), "1440p": (2560, 1440), "4k": (3840, 2160)}


# ----------------------------------------------------------------------------- cor
def _cie_cmf(lam_nm: np.ndarray) -> np.ndarray:
    """Funções de cor CIE 1931 2° (ajuste analítico de Wyman, Sloan & Shirley, 2013)."""
    def g(x, mu, s1, s2):
        s = np.where(x < mu, s1, s2)
        return np.exp(-0.5 * ((x - mu) / s) ** 2)
    x = 1.056 * g(lam_nm, 599.8, 37.9, 31.0) + 0.362 * g(lam_nm, 442.0, 16.0, 26.7) \
        - 0.065 * g(lam_nm, 501.1, 20.4, 26.2)
    y = 0.821 * g(lam_nm, 568.8, 46.9, 40.5) + 0.286 * g(lam_nm, 530.9, 16.3, 31.1)
    z = 1.217 * g(lam_nm, 437.0, 11.8, 36.0) + 0.681 * g(lam_nm, 459.0, 26.0, 13.8)
    return np.stack([x, y, z], -1)


XYZ_TO_SRGB = np.array([[3.2406, -1.5372, -0.4986], [-0.9689, 1.8758, 0.0415], [0.0557, -0.2040, 1.0570]])


def soot_color_lut(T0: float = 700.0, T1: float = 3000.0, n: int = 232, alpha: float = 1.39):
    """Tabela T → (cor sRGB linear normalizada, peso de brilho visível relativo a 1800 K).

    Emissão espectral da fuligem ∝ λ^-α B_λ(T) (Hottel–Broughton); o peso é a luminância visível por
    unidade de potência total emitida (∝ T^(4+α)), que deixa o núcleo quente mais branco e brilhante."""
    lam = np.linspace(380e-9, 780e-9, 401)
    cmf = _cie_cmf(lam * 1e9)
    T = np.linspace(T0, T1, n)
    c2 = 1.4388e-2
    rgb = np.zeros((n, 3))
    Y = np.zeros(n)
    for i, t in enumerate(T):
        spec = lam ** (-5.0 - alpha) / np.expm1(c2 / (lam * t))
        xyz = (spec[:, None] * cmf).sum(0)
        rgb[i] = np.clip(XYZ_TO_SRGB @ xyz, 0.0, None)
        Y[i] = xyz[1] / t ** (4.0 + alpha)
    rgb /= rgb.max(1, keepdims=True)
    w = Y / np.interp(1800.0, T, Y)
    return T, rgb, w


def aces(x: torch.Tensor) -> torch.Tensor:
    """Tone mapping fílmico ACES (aproximação de Narkowicz)."""
    return ((x * (2.51 * x + 0.03)) / (x * (2.43 * x + 0.59) + 0.14)).clamp(0.0, 1.0)


def to_srgb(x: torch.Tensor) -> torch.Tensor:
    return torch.where(x <= 0.0031308, 12.92 * x, 1.055 * x.clamp(min=0.0) ** (1 / 2.4) - 0.055)


# --------------------------------------------------------------------- utilidades
def _gauss_kernel(sigma: float, dev, dtype):
    r = max(1, int(3 * sigma))
    x = torch.arange(-r, r + 1, device=dev, dtype=dtype)
    k = torch.exp(-0.5 * (x / sigma) ** 2)
    return k / k.sum()


def blur(img: torch.Tensor, sigma: float) -> torch.Tensor:
    """Desfoque gaussiano separável de (C, H, W); sigmas grandes via pirâmide (mais rápido)."""
    if sigma < 0.5:
        return img
    x = img[None]
    down = 1
    while sigma / down > 12 and min(x.shape[-2:]) > 64:
        x = F.avg_pool2d(x, 2)
        down *= 2
    k = _gauss_kernel(sigma / down, img.device, img.dtype)
    C = x.shape[1]
    pad = len(k) // 2
    x = F.conv2d(F.pad(x, (pad, pad, 0, 0), mode="replicate"), k.view(1, 1, 1, -1).expand(C, 1, 1, -1), groups=C)
    x = F.conv2d(F.pad(x, (0, 0, pad, pad), mode="replicate"), k.view(1, 1, -1, 1).expand(C, 1, -1, 1), groups=C)
    if down > 1:
        x = F.interpolate(x, size=img.shape[-2:], mode="bilinear", align_corners=False)
    return x[0]


def fractal_noise(n: int, lam_min: float, lam_max: float, beta: float, gen: torch.Generator, dev) -> torch.Tensor:
    """Ruído periódico n×n com espectro ∝ k^-β entre os comprimentos de onda lam_min e lam_max (em px)."""
    w = torch.randn(n, n, generator=gen)
    k = torch.fft.fftfreq(n)
    kk = torch.sqrt(k[:, None] ** 2 + k[None, :] ** 2)
    band = (kk >= 1.0 / lam_max) & (kk <= 1.0 / lam_min)
    amp = torch.where(band, kk.clamp(min=1e-6) ** (-beta / 2), torch.zeros_like(kk))
    f = torch.fft.ifft2(torch.fft.fft2(w) * amp).real
    return ((f - f.mean()) / f.std()).to(dev)


def _font(size: int, bold: bool = False):
    from PIL import ImageFont
    import matplotlib
    name = "DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf"
    path = Path(matplotlib.get_data_path()) / "fonts" / "ttf" / name
    try:
        return ImageFont.truetype(str(path), size)
    except OSError:
        return ImageFont.load_default()


def _mono(size: int, bold: bool = False):
    from PIL import ImageFont
    import matplotlib
    name = "DejaVuSansMono-Bold.ttf" if bold else "DejaVuSansMono.ttf"
    try:
        return ImageFont.truetype(str(Path(matplotlib.get_data_path()) / "fonts" / "ttf" / name), size)
    except OSError:
        return _font(size, bold)


class FFmpegPipe:
    """Escreve quadros RGB (H, W, 3) uint8 direto no ffmpeg (H.264, CRF baixo, yuv420p)."""

    def __init__(self, path: str, size: tuple, fps: int = 30, crf: int = 14):
        ff = shutil.which("ffmpeg")
        if ff is None:
            raise RuntimeError("ffmpeg não encontrado")
        W, H = size
        self.p = subprocess.Popen(
            [ff, "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}",
             "-r", str(fps), "-i", "-", "-c:v", "libx264", "-preset", "slow", "-crf", str(crf),
             "-tune", "film", "-pix_fmt", "yuv420p", "-movflags", "+faststart", path],
            stdin=subprocess.PIPE)
        self.path = path

    def write(self, frame: np.ndarray):
        self.p.stdin.write(np.ascontiguousarray(frame, dtype=np.uint8).tobytes())

    def close(self):
        self.p.stdin.close()
        if self.p.wait() != 0:
            raise RuntimeError(f"ffmpeg falhou ao gravar {self.path}")
        return self.path


# ------------------------------------------------------------------- render HD
def hd_view(rec, aspect: float = 16 / 9, pad: float = 4.0, top_room: float = 0.18):
    """Janela da câmera: envelope médio da LES (I ≥ 0,2) com folga e um pedaço da chaminé."""
    s = rec.static
    tip = s["tip"]
    xs, zs = [0.0], [float(tip[2])]
    Ip = rec.final.get("I_proj")
    if Ip is not None and (np.asarray(Ip) >= 0.2).any():
        ii, kk = np.nonzero(np.asarray(Ip) >= 0.2)
        xs += [s["xc"][ii].min(), s["xc"][ii].max()]
        zs += [s["zc"][kk].min(), s["zc"][kk].max()]
    else:   # sem médias: usa o maior quadro
        lum = np.asarray(rec.frames[-1]["lum"], float)
        m = lum > 0.05 * lum.max()
        ii, kk = np.nonzero(m)
        xs += [s["xc"][ii].min(), s["xc"][ii].max()]
        zs += [s["zc"][kk].min(), s["zc"][kk].max()]
    x0, x1 = min(xs) - pad, max(xs) + pad
    z0, z1 = min(zs) - pad, max(zs) + pad
    z1 += top_room * (z1 - z0)
    z0 = max(z0, float(tip[2]) - 0.45 * (z1 - z0))
    cx, cz, w, h = 0.5 * (x0 + x1), 0.5 * (z0 + z1), x1 - x0, z1 - z0
    if w / h < aspect:
        w = h * aspect
    else:
        h = w / aspect
    return (cx - w / 2, cx + w / 2, cz - h / 2, cz + h / 2)


class HDFlameRenderer:
    """Compõe quadros HD da chama a partir dos quadros gravados (Recorder)."""

    def __init__(self, rec, size=(1920, 1080), view=None, detail: float = 0.35, exposure: float = 1.0,
                 bloom: float = 0.6, wind_speed: float | None = None, device: str = "auto", seed: int = 3):
        # detail: 0 = só a LES; ~0,35 = realce discreto; > 0,6 fica artificial
        self.rec = rec
        s = rec.static
        self.W, self.H = size
        self.dev = torch.device("cuda" if (device == "auto" and torch.cuda.is_available()) else
                                ("cpu" if device == "auto" else device))
        self.view = view or hd_view(rec, aspect=self.W / self.H)
        x0, x1, z0, z1 = self.view
        self.ppm = self.W / (x1 - x0)                     # pixels por metro
        self.detail, self.exposure, self.bloom = detail, exposure, bloom
        self.U = float(s["u_ref"]) if wind_speed is None else wind_speed
        t = lambda a: torch.as_tensor(np.asarray(a, np.float32), device=self.dev)  # noqa: E731
        # posições físicas dos pixels (centros), z para cima
        xs = x0 + (np.arange(self.W) + 0.5) / self.ppm
        zs = z1 - (np.arange(self.H) + 0.5) / self.ppm
        self.X = t(np.broadcast_to(xs[None, :], (self.H, self.W)))
        self.Zc = t(np.broadcast_to(zs[:, None], (self.H, self.W)))
        # mapeamento físico → índice da malha esticada da LES (para grid_sample)
        self.xc, self.zc = np.asarray(s["xc"], float), np.asarray(s["zc"], float)
        self._xi = (t(self.xc), t(np.arange(len(self.xc))))
        self._zi = (t(self.zc), t(np.arange(len(self.zc))))
        # cor da fuligem
        T, rgb, w = soot_color_lut()
        self.lut_T0, self.lut_dT = float(T[0]), float(T[1] - T[0])
        self.lut_rgb, self.lut_w = t(rgb), t(w)
        # ruído fractal periódico (detalhe abaixo da malha), em metros
        gen = torch.Generator().manual_seed(seed)
        self.tile_m = 12.0
        n = 512
        dcell = float(np.median(np.diff(self.xc)[np.abs(self.xc[:-1]) < 20])) if len(self.xc) > 2 else 0.5
        self.dcell = dcell
        px_per_m = n / self.tile_m
        # deformação: estruturas de 1–6 células (línguas de chama); brilho: 0,6–3 células
        warp = (1.0 * dcell * px_per_m, 6.0 * dcell * px_per_m)
        fine = (0.6 * dcell * px_per_m, 3.0 * dcell * px_per_m)
        self.noise = [fractal_noise(n, *warp, 3.0, gen, self.dev) for _ in range(4)] + \
                     [fractal_noise(n, *fine, 3.0, gen, self.dev) for _ in range(2)]
        self.tl_fallback = None
        self.norm_hist: list[float] = []
        self.background = self._background()

    # ---------------------------------------------------------------- partes
    def _interp_index(self, v: torch.Tensor, pair) -> torch.Tensor:
        xp, ip = pair
        j = torch.searchsorted(xp, v.clamp(xp[0], xp[-1]).contiguous()).clamp(1, len(xp) - 1)
        x0, x1 = xp[j - 1], xp[j]
        return ip[j - 1] + (v - x0) / (x1 - x0)

    def _sample(self, field: np.ndarray, X: torch.Tensor, Z: torch.Tensor, smooth: float = 0.7) -> torch.Tensor:
        """Amostra um campo (nx, nz) da malha da LES nos pontos físicos (X, Z) (bicúbico, pré-filtrado)."""
        f = torch.as_tensor(np.asarray(field, np.float32), device=self.dev)
        if smooth:
            f = blur(f[None], smooth)[0]
        ix = self._interp_index(X, self._xi)
        iz = self._interp_index(Z, self._zi)
        nx, nz = f.shape
        gx = 2 * (iz + 0.5) / nz - 1        # grid_sample: x ↔ última dimensão (z da LES)
        gy = 2 * (ix + 0.5) / nx - 1
        grid = torch.stack([gx, gy], -1)[None]
        out = F.grid_sample(f[None, None], grid, mode="bicubic", padding_mode="border", align_corners=False)[0, 0]
        inside = (X >= self.xc[0]) & (X <= self.xc[-1]) & (Z >= self.zc[0]) & (Z <= self.zc[-1])
        return torch.where(inside, out, torch.zeros_like(out))

    def _noise(self, k: int, X: torch.Tensor, Z: torch.Tensor, t: float) -> torch.Tensor:
        """Ruído advectado com o escoamento (vento em x, empuxo em z) e que evolui no tempo."""
        u, w = max(self.U, 1.0), 0.35 * max(self.U, 3.0)
        ph = 2 * math.pi * t / 1.2
        out = 0.0
        for j, c in ((2 * k, math.cos(ph)), (2 * k + 1, math.sin(ph))):
            gx = 2 * (((X - u * t) / self.tile_m) % 1.0) - 1
            gy = 2 * (((Z - w * t) / self.tile_m) % 1.0) - 1
            out = out + c * F.grid_sample(self.noise[j][None, None], torch.stack([gx, gy], -1)[None],
                                          mode="bilinear", padding_mode="reflection", align_corners=False)[0, 0]
        return out

    def _background(self) -> torch.Tensor:
        H, W = self.H, self.W
        y = torch.linspace(0, 1, H, device=self.dev)[:, None, None]
        top = torch.tensor([0.004, 0.006, 0.014], device=self.dev)
        bot = torch.tensor([0.012, 0.016, 0.030], device=self.dev)
        return (top * (1 - y) + bot * y).expand(H, W, 3).permute(2, 0, 1).contiguous()

    def _stack_mask(self) -> torch.Tensor:
        tip = self.rec.static["tip"]
        half = max(0.45, 0.012 * (self.view[1] - self.view[0]))
        return ((self.X.abs() <= half) & (self.Zc <= float(tip[2]))).float()

    # ---------------------------------------------------------------- quadro
    def frame(self, f) -> torch.Tensor:
        """Imagem linear (3, H, W) do quadro f (antes do tone mapping)."""
        t = float(f["t"])
        X, Z = self.X, self.Zc
        a = self.detail
        lum = np.asarray(f["lum"], np.float32)
        if a > 0:   # deformação suave (1–6 células), concentrada nas bordas da chama, não no núcleo
            s0 = (self._sample(lum, X, Z, smooth=2.0) / max(float(lum.max()), 1e-12)).clamp(0, 1) ** 0.5
            d = 1.2 * a * self.dcell * (0.25 + 3.0 * s0 * (1 - s0))
            X = X + d * self._noise(0, X, Z, t)
            Z = Z + d * self._noise(1, X, Z, t)
        L = self._sample(np.log(np.maximum(lum, 1e-6 * max(lum.max(), 1e-12))), X, Z)
        I = torch.exp(L) * (L > math.log(1e-5 * max(lum.max(), 1e-12)))
        tl = f.get("tl")
        if tl is not None:
            T = self._sample(np.asarray(tl, np.float32), X, Z, smooth=1.0)
        else:      # quadros antigos sem temperatura: estima pela intensidade (núcleo mais quente)
            T = 1350.0 + 650.0 * (I / I.max().clamp(min=1e-12)) ** 0.5
        if a > 0:   # modulação de brilho só nas bordas da chama
            s = (I / I.max().clamp(min=1e-12)) ** 0.5
            I = I * torch.exp(0.45 * a * self._noise(2, X, Z, t) * (4 * s * (1 - s) + 0.15))
        idx = ((T.clamp(self.lut_T0, self.lut_T0 + self.lut_dT * (len(self.lut_w) - 1)) - self.lut_T0)
               / self.lut_dT)
        i0 = idx.floor().long().clamp(0, len(self.lut_w) - 2)
        fr = (idx - i0)[..., None]
        rgb = self.lut_rgb[i0] * (1 - fr) + self.lut_rgb[i0 + 1] * fr
        wv = self.lut_w[i0] * (1 - fr[..., 0]) + self.lut_w[i0 + 1] * fr[..., 0]
        I = I * wv.clamp(0.25, 4.0)
        # exposição automática estável (mediana dos últimos quadros do percentil 99,7)
        q = torch.quantile(I.flatten()[:: max(1, I.numel() // 400000)], 0.997).item()
        self.norm_hist.append(q)
        ref = max(float(np.median(self.norm_hist[-40:])), 1e-12)
        E = (2.2 * self.exposure / ref) * I
        flame = (E[..., None] * rgb).permute(2, 0, 1)
        img = self.background.clone()
        # chaminé iluminada pela chama (silhueta + reflexo quente)
        sm = self._stack_mask()
        tipz = float(self.rec.static["tip"][2])
        lit = 0.02 * float(E.max().clamp(max=6.0)) / (1.0 + ((tipz - self.Zc) / 6.0) ** 2)
        base = torch.tensor([0.030, 0.033, 0.040], device=self.dev)[:, None, None]
        warm = torch.tensor([1.0, 0.68, 0.42], device=self.dev)[:, None, None]
        img = img * (1 - sm) + (base + warm * lit[None]) * sm
        img = img + flame
        if self.bloom > 0:
            bright = (flame - 0.6).clamp(min=0.0)
            b = 0.0
            for sig, wgt in ((0.004, 0.55), (0.015, 0.3), (0.05, 0.15)):
                b = b + wgt * blur(bright, sig * self.H)
            img = img + self.bloom * b
        return img

    def to_uint8(self, img: torch.Tensor) -> np.ndarray:
        out = to_srgb(aces(img)).clamp(0, 1)
        return (out.permute(1, 2, 0) * 255.0 + 0.5).to(torch.uint8).cpu().numpy()


# -------------------------------------------------------------- sobreposições
class Overlay:
    """Textos e elementos gráficos desenhados com PIL sobre o quadro HD."""

    def __init__(self, size, view, title: str, subtitle: str, caption: str, wind: bool = True):
        from PIL import Image, ImageDraw
        self.W, self.H = size
        self.s = self.H / 1080.0
        self.view = view
        self.static = Image.new("RGBA", size, (0, 0, 0, 0))
        d = ImageDraw.Draw(self.static)
        s = self.s
        d.text((int(58 * s), int(36 * s)), title, font=_font(int(44 * s), True), fill=(236, 240, 246, 255))
        d.text((int(60 * s), int(98 * s)), subtitle, font=_font(int(24 * s)), fill=(150, 160, 175, 255))
        if wind:
            y = int(200 * s)
            d.line([(int(60 * s), y), (int(185 * s), y)], fill=(120, 200, 255, 255), width=max(2, int(3 * s)))
            d.polygon([(int(200 * s), y), (int(182 * s), y - int(9 * s)), (int(182 * s), y + int(9 * s))],
                      fill=(120, 200, 255, 255))
            d.text((int(60 * s), y + int(16 * s)), "vento", font=_font(int(22 * s)), fill=(120, 200, 255, 255))
        # barra de escala de 10 m
        x0, x1, z0, z1 = view
        ppm = self.W / (x1 - x0)
        L = 10 * ppm
        xr = self.W - int(60 * s)
        yb = self.H - int(110 * s)
        d.line([(int(xr - L), yb), (xr, yb)], fill=(235, 238, 242, 255), width=max(2, int(4 * s)))
        d.text((int(xr - L / 2), yb - int(36 * s)), "10 m", font=_font(int(22 * s)), fill=(235, 238, 242, 255),
               anchor="mm")
        d.text((self.W - int(60 * s), self.H - int(48 * s)), caption, font=_font(int(18 * s)),
               fill=(140, 148, 160, 255), anchor="rm")
        self.f_clock = _mono(int(46 * s), True)
        self.f_read = _mono(int(26 * s))

    def compose(self, rgb: np.ndarray, t: float, L: float) -> np.ndarray:
        from PIL import Image, ImageDraw
        im = Image.fromarray(rgb).convert("RGBA")
        im.alpha_composite(self.static)
        d = ImageDraw.Draw(im)
        s = self.s
        d.text((self.W - int(60 * s), int(36 * s)), f"t = {t:5.1f} s", font=self.f_clock,
               fill=(255, 176, 64, 255), anchor="ra")
        d.text((self.W - int(60 * s), int(104 * s)), f"L = {L:4.1f} m", font=self.f_read,
               fill=(236, 240, 246, 255), anchor="ra")
        return np.asarray(im.convert("RGB"))


def render_flame_hd(rec, path: str, title: str, subtitle: str, resolution: str = "1080p", fps: int = 30,
                    every: int = 1, detail: float = 0.35, exposure: float = 1.0, bloom: float = 0.6,
                    caption: str | None = None, t_range: tuple | None = None, device: str = "auto",
                    log_every: int = 100) -> str:
    """Segmento HD da chama: resolução '1080p', '1440p' ou '4k'; detail = realce visual abaixo da malha
    (0 desliga); every = usa 1 a cada `every` quadros gravados (acelera o vídeo)."""
    size = RES.get(resolution, resolution if isinstance(resolution, tuple) else RES["1080p"])
    r = HDFlameRenderer(rec, size=size, detail=detail, exposure=exposure, bloom=bloom, device=device)
    if caption is None:
        caption = ("LES 3D · emissão da fuligem integrada na linha de visada (câmera sintética)"
                   + (" · realce visual de detalhes abaixo da malha" if detail > 0 else ""))
    ov = Overlay(size, r.view, title, subtitle, caption, wind=float(rec.static["u_ref"]) > 0)
    frames = rec.frames[::every]
    if t_range is not None:
        frames = [f for f in frames if t_range[0] <= f["t"] <= t_range[1]]
    pipe = FFmpegPipe(path, size, fps)
    with torch.no_grad():
        for n, f in enumerate(frames):
            pipe.write(ov.compose(r.to_uint8(r.frame(f)), float(f["t"]), float(f["L"])))
            if log_every and (n + 1) % log_every == 0:
                print(f"  HD: {n + 1}/{len(frames)} quadros", flush=True)
    return pipe.close()


def flame_hd_png(rec, path: str, i: int = -1, resolution: str = "1080p", **kw) -> str:
    """Um quadro HD isolado (PNG), para conferência ou capa do post."""
    from PIL import Image
    size = RES.get(resolution, RES["1080p"])
    r = HDFlameRenderer(rec, size=size, **kw)
    with torch.no_grad():
        for f in rec.frames[max(0, (i % len(rec.frames)) - 40): (i % len(rec.frames)) + 1]:
            img = r.frame(f)   # aquece a exposição automática
    Image.fromarray(r.to_uint8(img)).save(path)
    return path


# ------------------------------------------------------------- cartões de texto
def _card_background(size, backdrop=None, place: float = 0.78, strength: float = 0.5,
                     ramp_from: float = 0.42) -> np.ndarray:
    """Fundo escuro em gradiente; com `backdrop` (quadro HD da chama, sem textos), a chama é deslocada
    para a fração `place` da largura e entra esmaecida, sem competir com o texto à esquerda."""
    W, H = size
    bg = np.zeros((H, W, 3), np.float32)
    yy = np.linspace(0, 1, H)[:, None]
    for c, (a, b) in enumerate(zip((8, 11, 20), (14, 20, 34))):
        bg[..., c] = a + (b - a) * yy
    if backdrop is not None:
        from PIL import Image
        im = np.asarray(Image.fromarray(np.asarray(backdrop, np.uint8)).resize((W, H), Image.LANCZOS), np.float32)
        lum = im.mean(2)
        wgt = np.clip(lum - np.percentile(lum, 50), 0, None)
        cx = float((wgt.sum(0) * np.arange(W)).sum() / max(wgt.sum(), 1e-6))
        sh = int(place * W - cx)
        out = np.zeros_like(im)
        if sh >= 0:
            out[:, sh:] = im[:, :W - sh]
        else:
            out[:, :W + sh] = im[:, -sh:]
        ramp = np.clip((np.linspace(0, 1, W) - ramp_from) / 0.35, 0, 1)[None, :, None] ** 1.5
        bg = np.maximum(bg, strength * ramp * out)
    return np.clip(bg, 0, 255)


def render_title_card(path: str, title: str, subtitle: str, credit: str = "", note: str = "",
                      size=(1920, 1080), fps: int = 30, seconds: float = 3.5, backdrop=None,
                      accent=(255, 176, 64)) -> str:
    """Abertura em HD (PIL): título, subtítulo, crédito e nota, com entrada suave."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    base = Image.fromarray(_card_background(size, backdrop, place=0.5, strength=0.38,
                                            ramp_from=-1.0).astype(np.uint8)).convert("RGBA")
    probe = ImageDraw.Draw(base)

    def fit(text, bold, size0, maxw):
        sz = size0
        while sz > 12 and probe.textlength(text, font=_font(int(sz * s), bold)) > maxw:
            sz -= 2
        return _font(int(sz * s), bold)
    items = [(title, fit(title, True, 78, 0.9 * W), (240, 244, 250), 0.50, 0.0),
             (subtitle, fit(subtitle, False, 36, 0.9 * W), accent, 0.60, 0.35)]
    if credit:
        items.append((credit, fit(credit, False, 28, 0.9 * W), (170, 178, 190), 0.69, 0.7))
    if note:
        items.append((note, fit(note, False, 22, 0.9 * W), (130, 138, 150), 0.93, 1.0))
    pipe = FFmpegPipe(path, size, fps, crf=16)
    n = int(seconds * fps)
    for i in range(n):
        t = i / fps
        im = base.copy()
        d = ImageDraw.Draw(im)
        for text, font, col, yf, delay in items:
            a = float(np.clip((t - delay) / 0.6, 0, 1) * np.clip((seconds - t) / 0.4, 0, 1))
            ease = 1 - (1 - a) ** 3
            d.text((W / 2, yf * H + (1 - ease) * 20 * s), text, font=font, fill=tuple(col) + (int(255 * a),),
                   anchor="mm")
        pipe.write(np.asarray(im.convert("RGB")))
    return pipe.close()


def render_text_cards(path: str, cards: list[dict], size=(1920, 1080), fps: int = 30,
                      seconds: float | list = 7.0, fade: float = 0.6, accent=(255, 176, 64),
                      backdrop=None) -> str:
    """Cartões de texto (kicker, título, tópicos, nota opcional) com entrada suave, em HD.

    cards = [{"kicker": "O DESAFIO", "headline": "...", "bullets": ["...", "..."], "footnote": "..."}]
    backdrop: imagem RGB opcional (ex.: um quadro HD da chama) mostrada esmaecida à direita."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    pipe = FFmpegPipe(path, size, fps, crf=16)
    f_k, f_h, f_b, f_n = _font(int(30 * s), True), _font(int(62 * s), True), _font(int(38 * s)), _font(int(24 * s))
    secs = seconds if isinstance(seconds, (list, tuple)) else [seconds] * len(cards)
    bg = _card_background(size, backdrop)

    def wrap(d, text, font, width):
        words, lines, cur = text.split(), [], ""
        for w in words:
            nxt = (cur + " " + w).strip()
            if d.textlength(nxt, font=font) <= width:
                cur = nxt
            else:
                lines.append(cur)
                cur = w
        return lines + ([cur] if cur else [])

    for card, sec in zip(cards, secs):
        n = int(sec * fps)
        base = Image.fromarray(bg.astype(np.uint8)).convert("RGBA")
        layers = []   # (y_alvo, imagem RGBA, atraso em s)
        x0, wmax = int(150 * s), int(W - 300 * s)
        y = int(230 * s)
        probe = ImageDraw.Draw(base)

        def layer(lines, font, color, gap, delay, bullet=False):
            nonlocal y
            h = int(font.size * 1.25)
            im = Image.new("RGBA", (W, h * len(lines) + int(gap)), (0, 0, 0, 0))
            d = ImageDraw.Draw(im)
            for k, ln in enumerate(lines):
                xo = x0 + (int(48 * s) if bullet else 0)
                if bullet and k == 0:
                    d.ellipse([x0 + int(6 * s), int(h * 0.42 - 8 * s), x0 + int(22 * s), int(h * 0.42 + 8 * s)],
                              fill=accent + (255,))
                d.text((xo, k * h), ln, font=font, fill=color)
            layers.append((y, im, delay))
            y += im.height

        layer([card.get("kicker", "").upper()], f_k, accent + (255,), 18 * s, 0.0)
        layer(wrap(probe, card["headline"], f_h, wmax), f_h, (238, 242, 248, 255), 46 * s, 0.25)
        for j, bl in enumerate(card.get("bullets", [])):
            layer(wrap(probe, bl, f_b, wmax - int(48 * s)), f_b, (200, 208, 220, 255), 22 * s, 0.7 + 0.45 * j,
                  bullet=True)
        foot = card.get("footnote")
        y_top = int(230 * s)
        shift = int((H - (y - y_top)) / 2 - y_top - 20 * s)      # centraliza o bloco na vertical
        layers = [(yl + shift, L, dl) for (yl, L, dl) in layers]
        bar = (y_top + shift, y + shift)
        icard = cards.index(card)
        for i in range(n):
            t = i / fps
            im = base.copy()
            dr = ImageDraw.Draw(im)
            dr.rectangle([x0 - int(40 * s), bar[0], x0 - int(32 * s), bar[1]], fill=accent + (255,))
            if len(cards) > 1:   # indicador de progresso (bolinhas)
                for j in range(len(cards)):
                    cx = W - int((60 + 28 * (len(cards) - 1 - j)) * s)
                    r = int((7 if j == icard else 5) * s)
                    col = accent + (255,) if j == icard else (90, 98, 112, 255)
                    dr.ellipse([cx - r, H - int(60 * s) - r, cx + r, H - int(60 * s) + r], fill=col)
            for (yl, L, delay) in layers:
                a = float(np.clip((t - delay) / fade, 0, 1))
                a *= float(np.clip((sec - t) / fade, 0, 1))
                if a <= 0:
                    continue
                ease = 1 - (1 - a) ** 3
                Lc = L.copy()
                Lc.putalpha(Image.eval(L.getchannel("A"), lambda v, a=a: int(v * a)))
                im.alpha_composite(Lc, (0, int(yl + (1 - ease) * 24 * s)))
            if foot:
                a = float(np.clip((t - 1.2) / fade, 0, 1) * np.clip((sec - t) / fade, 0, 1))
                dr.text((x0, H - int(90 * s)), foot, font=f_n, fill=(140, 148, 160, int(255 * a)))
            pipe.write(np.asarray(im.convert("RGB")))
    return pipe.close()

In [ ]:
import importlib, flarekit
print('flarekit pronto')

## 1 · Local e clima de vento

In [ ]:
#@title 1 · Local e clima de vento (atlas eólico)
import numpy as np
import matplotlib.pyplot as plt
from flarekit import wind

from flarekit.props import pressure_at_altitude

LOCAL = "REPLAN · Paulínia (SP)"   #@param {type:"string"}
LAT = -22.7283                     #@param {type:"number"}
LON = -47.1317                     #@param {type:"number"}
ALTITUDE = 600.0                   #@param {type:"number"}
ALTURA = 115.0                     #@param {type:"number"}
FONTE_VENTO = "auto"               #@param ["auto", "gwa", "nasa", "open-meteo", "sintetico"]
Z0_LOCAL = 0.5                     #@param {type:"number"}
MODO_VENTO_LES = "dominante_p90"   #@param ["dominante_p90", "dominante_media", "projeto"]

# REPLAN: centro da refinaria 22°43′42″ S, 47°07′54″ O; ALTITUDE do terreno [m] (média de Paulínia) e
# ALTURA do tip [m] (maior tocha, ~115 m segundo fontes abertas): confira com os dados do projeto.
# auto: Global Wind Atlas → NASA POWER → Open-Meteo (ERA5) → rosa sintética (só sem internet)
# Z0_LOCAL escolhe a classe de rugosidade do GWA (NBR 6123: cat. II 0,07 m · III 0,3 m · IV 1,0 m;
# 0,5 m ≈ entorno industrial/suburbano)
P_LOCAL = pressure_at_altitude(ALTITUDE)
serie = wind.get_wind_series(LAT, LON, FONTE_VENTO, z0_site=Z0_LOCAL, H=ALTURA)
clima = wind.WindClimate.from_series(serie, ALTURA)
print(f"Altitude {ALTITUDE:.0f} m → pressão {P_LOCAL/1e3:.1f} kPa")
print(clima.describe())
if serie.synthetic:
    print("\nATENÇÃO: sem acesso às fontes de vento; usando uma rosa SINTÉTICA de exemplo.")

U_LES, TH_LES = clima.design_wind(MODO_VENTO_LES)
SETOR = wind.sector_name(TH_LES)
wind_label = f"vento de {SETOR} · {U_LES:.1f} m/s a {ALTURA:.0f} m"
if serie.synthetic:
    wind_label += " (rosa sintética)"
print(f"\nLES: {wind_label} (modo {MODO_VENTO_LES})")

fig = plt.figure(figsize=(12, 4.2))
ax = fig.add_subplot(1, 2, 1, projection="polar")
ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
bottom = np.zeros(16)
for b in range(clima.freq.shape[1]):
    ax.bar(np.radians(np.arange(16) * 22.5), 100 * clima.freq[:, b], width=np.radians(20), bottom=bottom)
    bottom += 100 * clima.freq[:, b]
ax.set_xticks(np.radians(np.arange(16) * 22.5)); ax.set_xticklabels(wind.SECTORS, fontsize=8)
ax.set_title(f"Rosa dos ventos a {ALTURA:.0f} m [% do tempo]")
ax2 = fig.add_subplot(1, 2, 2)
z = np.linspace(1, max(120.0, 1.6 * ALTURA), 100)
ax2.plot(clima.profile(z), z, label=f"lei log, z0 = {clima.z0:.2f} m")
ax2.plot(clima.profile(z, U_LES), z, "--", label=f"perfil da LES ({U_LES:.1f} m/s no tip)")
ax2.plot([serie.U1.mean(), serie.U2.mean()], [serie.z1, serie.z2], "o", label="médias dos dados")
ax2.axhline(ALTURA, ls=":", c="k"); ax2.set_xlabel("U [m/s]"); ax2.set_ylabel("z [m]"); ax2.legend()
ax2.set_title("Perfil de camada limite")
plt.tight_layout(); plt.show()

## 2 · Cenário e referências

In [ ]:
#@title 2 · Combustível, cenário e modelos de referência (API 521, Chamberlain, correlações)
from flarekit.props import (FLARE_GAS_PRESETS, FUELS, flare_gas, lhv_volumetric, mixture, parse_composition,
                            stoichiometry, state_relation, beta_pdf_table)
from flarekit.semiempirical import Scenario, flame_box

COMBUSTIVEL = "gás de refinaria típico (FISPQ ex-RLAM)"   #@param ["gás de refinaria típico (FISPQ ex-RLAM)", "gás de tocha médio de refinaria (Emam 2015)", "rico em H2 (despressurização, hipotético)", "propano", "metano", "personalizado"]
COMPOSICAO = "H2: 33, CH4: 29, C2H6: 15, C2H4: 10.5, C3H8: 1.5, C3H6: 1, nC4H10: 2.5, nC5H12: 0.5, N2: 3.5, CO2: 1.5, CO: 1.5, H2S: 0.5"  #@param {type:"string"}
VAZAO = 12.6           #@param {type:"number"}
CORRIGIR_XRAD = True   #@param {type:"boolean"}

# COMPOSICAO (% molar) só vale com "personalizado". Espécies: H2, CH4, C2H6, C2H4, C3H8, C3H6, nC4H10,
# iC4H10, C4H8, nC5H12, CO, CO2, N2, O2, H2O, H2S. Não há composição medida pública da REPLAN: o
# padrão usa os pontos médios da FISPQ do "Gás Residual de Refinaria" da ex-RLAM (Acelen).
# VAZAO [kg/s]: 12,6 kg/s desse gás ≈ 570 MW, chama de 30–50 m (API 521); alívios de emergência
# podem ser bem maiores (ex.: 100 kg/s = 360 t/h).
if COMBUSTIVEL in FLARE_GAS_PRESETS:
    fuel = flare_gas(COMBUSTIVEL)
elif COMBUSTIVEL == "personalizado":
    fuel = mixture("gás personalizado", parse_composition(COMPOSICAO))
else:
    fuel = FUELS[COMBUSTIVEL]
NOME_COMB = {"propano": "propano", "metano": "metano"}.get(COMBUSTIVEL, "gás de refinaria")
sc = Scenario(fuel, mdot=VAZAO, T_j=311.0, mach=0.5, H=ALTURA, u_w=U_LES, T_inf=298.15, RH=0.5,
              p_atm=P_LOCAL, xrad_comp=CORRIGIR_XRAD)
tip, ch, ch0 = sc.tip, sc.cham, sc.cham0
print(fuel.describe())
print(f"PCI = {lhv_volumetric(fuel)/1e6:.1f} MJ/Nm³ · Z_st = {sc.st['Z_st']:.4f} · "
      f"correção de F_s pela composição = {sc.fs_factor:.3f}")
print(f"Tip: u_j = {tip['u_j']:.1f} m/s (Mach 0,5) · d = {tip['d_j']:.3f} m · Q = {sc.Q/1e6:.0f} MW")
print(f"API 521 (Beychok):      L = {sc.L_api:5.1f} m")
print(f"Chamberlain, sem vento: L = {ch0.L_b:5.1f} m")
print(f"Chamberlain, {sc.u_w:.1f} m/s: L = {ch.L_b:5.1f} m · α = {ch.alpha:4.1f}° · b = {ch.b:.1f} m · "
      f"W1 = {ch.W1:.2f} m · W2 = {ch.W2:.1f} m · F_s = {ch.F_s:.3f} · SEP = {ch.SEP/1e3:.0f} kW/m²")

## 3 · Termoquímica

In [ ]:
#@title 3 · Termoquímica: equilíbrio em Z e tabela β-PDF
st = sc.st
sr = state_relation(fuel, sc.T_j, sc.T_inf, chi_loss=ch.F_s, p=sc.p_atm)   # perda radiativa = fração radiante
tab = beta_pdf_table(sr, st["Z_st"])
refs = sc.references(sr.T_ad_st)
print(f"{sr.source} · Z_st = {st['Z_st']:.4f} · T_ad = {sr.T_ad_st:.0f} K")
print(f"Delichatsios: Fr_f = {refs['delichatsios']['Fr_f']:.2f}, L = {refs['delichatsios']['L']:.1f} m · "
      f"Heskestad: L = {refs['L_heskestad']:.1f} m · Molina: X_rad = {refs['molina']['X_rad']:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(sr.Z, sr.T, lw=2); ax[0].axvline(st["Z_st"], ls="--", c="k", lw=1)
zlim = max(0.4, min(1.0, 2.5 * st["Z_st"]))
ax[0].set_xlim(0, zlim); ax[0].set_xlabel("Z"); ax[0].set_ylabel("T [K]"); ax[0].set_title("Relação de estado")
for j in (0, 4, 8, 16):
    ax[1].plot(tab.Zt, tab.T[:, j], label=f"s = {tab.s[j]:.2f}")
ax[1].set_xlim(0, zlim); ax[1].set_xlabel("Z̃"); ax[1].set_ylabel("T̃ [K]"); ax[1].legend()
ax[1].set_title("Média na β-PDF (variância de submalha)")
plt.tight_layout(); plt.show()

## 4 · LES com gráficos em tempo real
O painel abaixo é atualizado durante a solução; cada quadro (a cada 0,1 s simulado) é gravado para o vídeo.

In [ ]:
#@title 4 · LES 3D (60 s) com painel ao vivo, gravando os quadros do vídeo
from flarekit.les import FlareLES, LESConfig
from flarekit.render import RecordedLES, Recorder, run_and_record

PRESET = "gpu" if torch.cuda.is_available() else "cpu"   #@param ["gpu", "gpu_fino", "gpu_ultra", "cpu", "teste"] {allow-input: true}
T_END = 60.0        #@param {type:"number"}
T_AVG = 15.0        #@param {type:"number"}
FRAME_DT = 0.1      #@param {type:"number"}
LIVE_EVERY = 5      #@param {type:"integer"}
USAR_QUADROS_SALVOS = False   #@param {type:"boolean"}
ARQUIVO_QUADROS = "flare_quadros.npz"   #@param {type:"string"}

# USAR_QUADROS_SALVOS: pula a LES e usa um flare_quadros.npz de uma rodada anterior (mesmo cenário) para
# refazer as figuras e o vídeo — no Colab, o arquivo é pedido por upload se não estiver na pasta.
title = f"Flare · {NOME_COMB} · {sc.Q/1e6:.0f} MW · LES 3D · {LOCAL}"
if USAR_QUADROS_SALVOS:
    import os
    if not os.path.exists(ARQUIVO_QUADROS):
        from google.colab import files
        ARQUIVO_QUADROS = next(iter(files.upload()))
    rec = Recorder.load(ARQUIVO_QUADROS)
    les = RecordedLES(rec)
    print(f"Quadros carregados: {len(rec.frames)} · {les.summary()}")
else:
    box = flame_box(ch, sc.H)          # malha fina ajustada à chama prevista por Chamberlain
    cfg = LESConfig.for_flame(PRESET, box, H=sc.H, u_ref=U_LES, z0=clima.z0, mdot=sc.mdot, u_jet=tip["u_j"],
                              X_rad=ch.F_s, T_inf=sc.T_inf, RH=sc.RH,
                              receiver_box=wind.receiver_box(sc, U_LES))   # receptores cobrem a zona de 1,58 kW/m²
    les = FlareLES(cfg, tab, sc.Q)
    print(les.summary())
dash_sub = (f"{les.summary()} · {wind_label} · perfil log z0 = {clima.z0:.2f} m · "
            f"Smagorinsky + Z̃/β-PDF (equilíbrio) + fuligem · média a partir de {T_AVG:.0f} s")
if not USAR_QUADROS_SALVOS:
    rec = run_and_record(les, sc, title, T_END, T_AVG, FRAME_DT, LIVE_EVERY, dash_title=title, dash_sub=dash_sub,
                         wind_label=wind_label)
    rec.save("flare_quadros.npz")     # guarde este arquivo: refaz figuras e vídeo sem rodar a LES de novo
    print(f"Concluído: {les.step_n} passos, {les.wall/60:.1f} min de solver, {les.wall/les.step_n*1e3:.0f} ms/passo")

## 5 · Validação e segurança

In [ ]:
#@title 5 · Validação e segurança no vento da LES
from flarekit.report import summary_figure, print_validation
from IPython.display import Image, display
fig, rows, zones = summary_figure(les, sc, refs, "flare_resumo.png", title)
print_validation(rows)
print()
for zz in zones:
    ge = "≥" if zz["truncado"] else ""   # zona chega à borda da grade de receptores
    print(f"{zz['nivel_kW_m2']:5.2f} kW/m² · área {ge}{zz['area_m2']:7.0f} m² · alcance {ge}{zz['raio_max_m']:5.0f} m"
          f" · {zz['descricao']}")
display(Image("flare_resumo.png"))

## 6 · Varredura das direções do vento

In [ ]:
#@title 6 · Varredura das direções: probabilidade de excedência e envoltória
from flarekit import render
RAIO_MAPA = 150.0       #@param {type:"number"}
# o mapa cresce sozinho se a zona de 1,58 kW/m² no vento máximo observado passar do raio escolhido
_rb = wind.receiver_box(sc, float(clima.U_H.max()))
RAIO_MAPA = max(RAIO_MAPA, 1.2 * max(abs(_rb[0]), _rb[1], _rb[2]))
RADIACAO_SOLAR = 0.0    #@param {type:"number"}
# RADIACAO_SOLAR [kW/m²]: some a solar (~0,8–1,0) se o critério adotado for de radiação TOTAL;
# 0 compara os níveis do API 521 só com a radiação do flare (confira a edição/critério usado)
grid = wind.SiteGrid.square(RAIO_MAPA, 121)
mapas = wind.exceedance_maps(sc, clima, grid, q_solar=RADIACAO_SOLAR)   # Chamberlain por classe × setor
R_dw = RAIO_MAPA * 1.45
les_down = wind.les_footprint_downwind(les.rec_x, les.rec_y, rec.final["q_mean"], sc, U_LES, R_dw,
                                       2 * int(R_dw / 2.0) + 1)
linhas = render.risk_summary_figure(clima, mapas, sc, "flare_risco.png",
                                    f"Zonas de radiação ponderadas pela rosa dos ventos · {LOCAL}")
print("nível [kW/m²]  envoltória [m]  P ≥ 1% [m]  P ≥ 10% [m]")
for lev, e, p1, p10 in linhas:
    print(f"   {lev:5.2f}        {e:6.0f}        {p1:6.0f}      {p10:6.0f}")
display(Image("flare_risco.png"))

## 7 · Vídeo para o LinkedIn

In [ ]:
#@title 7 · Vídeo final em HD: abertura, contexto para gestão, chama em HD, painel, varredura e resumos
from IPython.display import Video
from flarekit import hd
AUTOR = ""            #@param {type:"string"}
AVISO = "Cenário ilustrativo: local e clima de vento reais; composição e vazão de literatura aberta, não dados operacionais"  #@param {type:"string"}
RESOLUCAO = "1080p"   #@param ["1080p", "1440p", "4k"]
DETALHE_VISUAL = 0.35 #@param {type:"slider", min:0, max:0.8, step:0.05}
SEGUNDOS_POR_CARTAO = 7.0  #@param {type:"number"}
FPS = 30              #@param {type:"integer"}
PAINEL_ACELERADO = 2  #@param {type:"integer"}

# DETALHE_VISUAL: realce de detalhes abaixo da malha só na imagem da chama (0 = só a LES); não altera
# nenhum número da simulação. Os textos de contexto estão logo abaixo — edite à vontade.
q_solo = float(mapas["envelope"].max())
erro = {r["grandeza"]: r["erro_%"] for r in rows}
CARTOES = [
    {"kicker": "Por que importa",
     "headline": "Onde o calor do flare pode atingir pessoas e equipamentos?",
     "bullets": ["Em emergências, o flare queima o gás liberado e irradia calor.",
                 "A resposta orienta zonas de segurança, acessos e o arranjo da planta.",
                 "Cada “e se?” (vazão, gás, vento) costuma pedir um novo estudo."]},
    {"kicker": "O que foi feito",
     "headline": "Fluxo automatizado em código aberto: do vento real ao mapa de risco",
     "bullets": [f"Vento real do local ({LOCAL}) e simulação 3D da chama (CFD).",
                 "Calor que chega ao solo e zonas de segurança pela API 521.",
                 "Mapas de probabilidade nas 16 direções de vento, além do pior caso."]},
    {"kicker": "Para time e liderança",
     "headline": "Dados e um visual claro para decidir, sem substituir o estudo detalhado",
     "bullets": ["Vazão maior? Muda-se um parâmetro e o cenário roda de novo.",
                 "Reprodutível e auditável: outra pessoa do time refaz e confere.",
                 "Desenvolve o time em física, dados e automação ao mesmo tempo."]},
]
RODAPE = "Cenário ilustrativo: local e vento reais; composição do gás e vazão de fontes abertas, não dados operacionais da Petrobras. Estudo pessoal."  #@param {type:"string"}


def _vs(e, mais, menos):   # "+16%" → "16% mais longa"; |e| < 3% → "praticamente igual"
    return "praticamente igual" if abs(e) < 3 else f"{abs(e):.0f}% {mais if e > 0 else menos}"


e_L = erro.get("Comprimento da chama L [m]", 0.0)
e_a = erro.get("Inclinação da chama [°]", 0.0)
e_q = erro.get("Fluxo máximo no solo [kW/m²]", 0.0)
abaixo = q_solo < 1.58
alc = {lev: e for lev, e, _, _ in linhas}
FECHAMENTO = {
    "kicker": "Resultado do caso ilustrativo",
    "headline": (f"Neste cenário, a altura de {ALTURA:.0f} m do flare cumpre seu papel" if abaixo else
                 f"Neste cenário, o calor no solo passa de 1,58 kW/m² perto do flare"),
    "bullets": [f"Pico de ~{q_solo:.1f} kW/m² no solo, considerando todas as direções de vento.".replace(
                    f"{q_solo:.1f}", f"{q_solo:.1f}".replace(".", ",")),
                ("Abaixo de 1,58 kW/m², o menor nível da API 521 (exposição contínua)." if abaixo else
                 f"1,58 kW/m² (exposição contínua, API 521) até ~{alc.get(1.58, 0):.0f} m da base, na pior direção."),
                f"Frente ao modelo Shell/Chamberlain: radiação máxima {_vs(e_q, 'maior', 'menor')}; "
                f"chama {_vs(e_L, 'mais longa', 'mais curta')}."],
    "footnote": RODAPE,
}
resultado = (f"No caso ilustrativo, o pico no solo (~{q_solo:.1f} kW/m²) fica abaixo do menor nível da API 521 "
             f"(1,58 kW/m²) com qualquer vento." if abaixo else
             f"No caso ilustrativo, o nível de 1,58 kW/m² da API 521 chega a ~{alc.get(1.58, 0):.0f} m da base "
             f"na pior direção de vento.")
LEGENDA = "\n".join([
    "Onde o calor de um flare de refinaria pode atingir pessoas e equipamentos?",
    "E com que frequência, dado o vento do local?", "",
    "Essa pergunta virou um estudo pessoal: um fluxo em Python, de código aberto, no Google Colab (GPU na nuvem), "
    "do vento real até este vídeo.", "",
    "No caminho: simulação 3D da chama (CFD), radiação no solo, zonas de segurança (API 521) e probabilidade em "
    "16 direções de vento.", "",
    f"Frente ao modelo Shell/Chamberlain: radiação máxima {_vs(e_q, 'maior', 'menor')}, chama "
    f"{_vs(e_L, 'mais longa', 'mais curta')}, inclinação {_vs(e_a, 'maior', 'menor')}.", "",
    resultado.replace(f"{q_solo:.1f}", f"{q_solo:.1f}".replace(".", ",")), "",
    "Para a gestão, o valor está em:",
    "• triar cenários “e se” antes dos estudos detalhados;",
    "• ver o risco com o vento real, não só o pior caso;",
    "• comunicar segurança visualmente a quem decide;",
    "• ter análise reprodutível e auditável, não presa a uma pessoa;",
    "• desenvolver o time em física, dados e automação.", "",
    "Complementa, não substitui, estudos detalhados em ferramentas validadas (como o Simcenter STAR-CCM+) e as normas.",
    "", RODAPE.replace("Estudo pessoal.", "").strip(), "",
    "Na sua equipe, como o risco chega hoje a quem decide?", "",
    "#CFD #SegurançaDeProcessos #GestãoDeEngenharia #EngenhariaQuímica"])
size = hd.RES[RESOLUCAO]
sub_hero = f"LES 3D em {'GPU' if les.dev.type == 'cuda' else 'CPU'} · {wind_label} · {LOCAL}"
fonte = serie.source.split(",")[0].split(" (")[0] if not serie.synthetic else "rosa sintética de exemplo"
capa = hd.flame_hd_png(rec, "capa_chama.png", resolution=RESOLUCAO, detail=DETALHE_VISUAL)   # também serve de capa do post
from PIL import Image as PILImage
from IPython.display import Image
fundo = np.asarray(PILImage.open(capa).convert("RGB"))
segs = [
    hd.render_title_card("v0_titulo.mp4", f"Flare · {NOME_COMB} · {sc.Q/1e6:.0f} MW",
                         f"LES 3D · vento: {fonte} · {LOCAL}", AUTOR, AVISO, size=size, fps=FPS, backdrop=fundo),
    hd.render_text_cards("v0_contexto.mp4", CARTOES, size=size, fps=FPS, seconds=SEGUNDOS_POR_CARTAO,
                         backdrop=fundo),
    hd.render_flame_hd(rec, "v1_chama.mp4", f"Flare · {NOME_COMB} · {sc.Q/1e6:.0f} MW", sub_hero,
                       resolution=RESOLUCAO, fps=FPS, detail=DETALHE_VISUAL),
    render.ZoomDashboard(rec, sc, ch, title, dash_sub, t_end=T_END).render("v2_painel.mp4", fps=FPS,
                                                                          every=PAINEL_ACELERADO),
    render.render_wind_sweep(clima, mapas, les_down, "v3_vento.mp4", U_LES,
                             f"Varredura das direções do vento · {LOCAL}", fps=FPS),
    render.render_still("flare_resumo.png", "v4_resumo.mp4", 5, FPS),
    render.render_still("flare_risco.png", "v5_risco.mp4", 6, FPS),
    hd.render_text_cards("v6_fechamento.mp4", [FECHAMENTO], size=size, fps=FPS,
                         seconds=SEGUNDOS_POR_CARTAO + 1, backdrop=fundo),
]
render.assemble(segs, "flare_linkedin.mp4", fps=FPS, size=size)
import os
if os.path.getsize("flare_linkedin.mp4") < 60e6:   # vídeos grandes (4K) não são embutidos no notebook
    display(Video("flare_linkedin.mp4", embed=True, width=960))
else:
    display(Image(capa, width=960))
print("Texto sugerido para o post:\n")
print(LEGENDA)
try:
    from google.colab import files
    files.download("flare_linkedin.mp4")
    files.download("capa_chama.png")
    files.download("flare_quadros.npz")
except Exception:
    print("Arquivos salvos: flare_linkedin.mp4, capa_chama.png, flare_quadros.npz")

## 8 · Estudo de malha (opcional)

In [ ]:
#@title 8 · (Opcional) Estudo de malha: comprimento da chama vs. Δ
RUN_MESH_STUDY = False  #@param {type:"boolean"}
if RUN_MESH_STUDY:
    res = []
    for d in (1.0, 0.7, 0.5, 0.35):
        c = LESConfig.for_flame("gpu", box, d_fine=d, H=sc.H, u_ref=U_LES, z0=clima.z0, mdot=sc.mdot,
                                u_jet=tip["u_j"], X_rad=ch.F_s)
        m = FlareLES(c, tab, sc.Q)
        while m.time < 20.0:
            m.step()
            if m.step_n % 20 == 0:
                m.diagnostics(m.time > 8.0)
        res.append((d, m.mean_flame_length(), m.mean_tilt()))
        print(f"Δ = {d:.2f} m → L = {res[-1][1]:.1f} m, inclinação = {res[-1][2]:.0f}°")
    d, L, a = map(np.array, zip(*res))
    plt.plot(d, L, "o-", label="LES"); plt.axhline(ch.L_b, ls="--", label="Chamberlain")
    plt.gca().invert_xaxis(); plt.xlabel("Δ fino [m]"); plt.ylabel("L [m]"); plt.legend(); plt.show()

## Notas de modelagem e limitações

- **Vento:** o Global Wind Atlas dá o clima *generalizado* (terreno plano, rugosidade padrão) por setor de 30°, como Weibull A/k em várias alturas; o notebook usa a classe de rugosidade mais próxima de `Z0_LOCAL` e amostra horas equivalentes. NASA POWER (MERRA-2, ~0,5°) e Open-Meteo (ERA5, 0,25°) dão séries horárias a 10/50 m e 10/100 m; o perfil log é ajustado às duas alturas. Estabilidade atmosférica não é considerada (perfil neutro). O endpoint do GWA não é documentado oficialmente, e o Open-Meteo exige chave para uso comercial.
- **Direções:** para um flare isolado em terreno aberto a física não depende da direção, então a pegada é girada para cada setor (exato nesse caso). Com estruturas no entorno, cada direção pediria uma LES própria. A probabilidade de excedência é condicional ao flare estar queimando na vazão simulada.
- **Química rápida + equilíbrio** em fração de mistura; a perda radiativa entra como fração radiante constante (F_s de Chamberlain), como no FDS. Eficiência de combustão (CE/DRE) exigiria EDC/FPV.
- **Radiação no solo:** emissão da zona luminosa (fuligem, relação de estado no lado rico), escalada para X_rad·Q, com transmissividade de Wayne em cada caminho; sem reabsorção dentro da chama.
- **Tip:** o jato real (diâmetro e velocidade impressos na etapa 2) é sub-malha; a fonte injeta a vazão e o fluxo de quantidade de movimento reais num bloco de 2×2 células finas.
- **Numérica e malha:** com Superbee (o limitador do FDS), Smagorinsky C_s = 0,1 e Sc_t = 0,7, a chama converge para Chamberlain com o refino. Estudo de malha do caso de calibração (propano, 12,6 kg/s, tocha de 30 m, vento de 8,9 m/s; Chamberlain: L = 27,4 m, α = 51°): Δ = 0,7 m → 21,9 m; Δ = 0,5 m → 27,9 m e α = 56°. Para chamas maiores, a malha cresce com a caixa da chama (mesma resolução relativa).

**Próximo passo (PINNeAPPle):** com LES em algumas classes de velocidade, treinar um surrogate (vento, vazão → mapa de radiação) com incerteza para zonas de exclusão em tempo real.